In [20]:

from google.colab import drive
from pathlib import Path
import shutil
import sys

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/LexiLabs")
LOCAL_ROOT = Path("/content/curatelaw_m4")
DATA_DIR = LOCAL_ROOT / "datasets"
ARTIFACT_DIR = DRIVE_ROOT / "finetuning_artifacts"

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FILES = [
    "raw_150_messages.jsonl",
    "curated_150_messages.jsonl",
]

for filename in FILES:
    source = DRIVE_ROOT / filename

    if not source.is_file():
        raise FileNotFoundError(
            f"Missing Google Drive file: {source}"
        )

    destination = DATA_DIR / filename
    shutil.copy2(source, destination)

    print(
        f"Copied {filename}: "
        f"{destination.stat().st_size:,} bytes"
    )

manifest_source = DRIVE_ROOT / "cohort_manifest.json"

if manifest_source.exists():
    shutil.copy2(
        manifest_source,
        DATA_DIR / "cohort_manifest.json"
    )
    print("Cohort manifest copied.")
else:
    print("Cohort manifest not uploaded.")

import torch

print("\nPython:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU unavailable. Select a T4 GPU runtime."
    )

print("GPU:", torch.cuda.get_device_name(0))

memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"GPU memory: {memory_gb:.2f} GB")

if memory_gb < 14:
    raise RuntimeError(
        "Insufficient GPU memory for the planned setup."
    )

print("\nEnvironment preflight passed.")


ValueError: Mountpoint must not already contain files

## Validate both datasets

In [ ]:

import hashlib
import json
from collections import Counter

EXPECTED_COUNT = 150

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def load_and_validate(path):
    records = []
    exact_pairs = []
    prompts = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                raise ValueError(
                    f"Blank line at {path.name}:{line_number}"
                )

            record = json.loads(line)

            if not isinstance(record, dict):
                raise ValueError(
                    f"Invalid record at line {line_number}"
                )

            messages = record.get("messages")

            if not isinstance(messages, list):
                raise ValueError(
                    f"Missing messages at line {line_number}"
                )

            if len(messages) != 2:
                raise ValueError(
                    f"Expected two messages at line {line_number}"
                )

            roles = [
                message.get("role")
                for message in messages
                if isinstance(message, dict)
            ]

            if roles != ["user", "assistant"]:
                raise ValueError(
                    f"Invalid roles at line {line_number}: {roles}"
                )

            for message in messages:
                content = message.get("content")

                if (
                    not isinstance(content, str)
                    or not content.strip()
                ):
                    raise ValueError(
                        f"Missing message content at line {line_number}"
                    )

            prompt = " ".join(
                messages[0]["content"].split()
            ).casefold()

            response = " ".join(
                messages[1]["content"].split()
            ).casefold()

            prompts.append(prompt)
            exact_pairs.append((prompt, response))
            records.append(record)

    if len(records) != EXPECTED_COUNT:
        raise ValueError(
            f"{path.name}: expected 150, found {len(records)}"
        )

    duplicate_pairs = (
        len(exact_pairs) - len(set(exact_pairs))
    )

    duplicate_prompts = (
        len(prompts) - len(set(prompts))
    )

    if duplicate_pairs:
        raise ValueError(
            f"{path.name}: {duplicate_pairs} duplicate pairs"
        )

    if duplicate_prompts:
        raise ValueError(
            f"{path.name}: {duplicate_prompts} duplicate prompts"
        )

    return records, set(exact_pairs), set(prompts)


raw_path = DATA_DIR / "raw_150_messages.jsonl"
curated_path = DATA_DIR / "curated_150_messages.jsonl"

raw_records, raw_pairs, raw_prompts = load_and_validate(
    raw_path
)

curated_records, curated_pairs, curated_prompts = (
    load_and_validate(curated_path)
)

overlap = raw_pairs & curated_pairs

print("=" * 60)
print("CURATELAW M4 DATASET VALIDATION")
print("=" * 60)

print("Raw examples:", len(raw_records))
print("Curated examples:", len(curated_records))
print("Overlapping normalized pairs:", len(overlap))
print("Overlapping prompts:", len(raw_prompts & curated_prompts))

print("\nSHA-256:")
print("Raw:", sha256(raw_path))
print("Curated:", sha256(curated_path))

manifest_path = DATA_DIR / "cohort_manifest.json"

if manifest_path.is_file():
    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    if manifest.get("examples_per_cohort") != 150:
        raise ValueError(
            "Manifest example count does not match."
        )

    if len(manifest.get("raw_ids", [])) != 150:
        raise ValueError("Invalid raw manifest IDs.")

    if len(manifest.get("curated_ids", [])) != 150:
        raise ValueError("Invalid curated manifest IDs.")

    print("\nManifest task quotas:")
    print(manifest.get("task_quotas", {}))

    print(
        "Manifest scenario-ID overlap:",
        len(manifest.get("overlapping_ids", []))
    )

print("\nDataset validation passed.")


CURATELAW M4 DATASET VALIDATION
Raw examples: 150
Curated examples: 150
Overlapping normalized pairs: 71
Overlapping prompts: 71

SHA-256:
Raw: 408a5ad9e023051f62da8b7fa1a7e6c00b57c6ebe2f9028e88f5028e0dda40f2
Curated: e5881453eae42849eab4e2092587f78778be97b9f203e8347de96be6b1a5a2b8

Manifest task quotas:
{'full_agreement': 12, 'clause_drafting': 45, 'contract_customization': 45, 'contradiction_correction': 32, 'missing_clause_completion': 16}
Manifest scenario-ID overlap: 71

Dataset validation passed.


## AlignTune

In [ ]:

%pip install -q "aligntune==0.1.35"


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:

%pip uninstall -y torchvision


Found existing installation: torchvision 0.28.0
Uninstalling torchvision-0.28.0:
  Successfully uninstalled torchvision-0.28.0


In [ ]:
from aligntune.core.backend_factory import create_sft_trainer
print("AlignTune import: SUCCESS")

W1113 06:34:23.859000 5873 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1113 06:34:24.044000 5873 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


[aligntune] 2026-11-13 06:34:52 WARNING aligntune.core.rl.registries:54 - Could not initialize reward 'rubric' from rewards registry: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.


AlignTune import: SUCCESS


In [ ]:

import sys
import torch
import transformers
import importlib.metadata

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

try:
    print(
        "AlignTune:",
        importlib.metadata.version("aligntune")
    )

    from aligntune.core.backend_factory import (
        create_sft_trainer
    )

    print("AlignTune backend import: SUCCESS")

except Exception as exc:
    print("AlignTune backend import: FAILED")
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc))
    raise


Python: 3.13.16
PyTorch: 2.14.1+cu130
Transformers: 5.22.1
CUDA available: True
GPU: Tesla T4
AlignTune: 0.1.35


W1113 06:22:38.485000 1329 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1113 06:22:38.487000 1329 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
W1113 06:22:38.488000 1329 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1113 06:22:38.489000 1329 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
W1113 06:22:38.490000 1329 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1113 06:22:38.490000 1329 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
W1113 06:22:38.491000 1329 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1113 06:22:38.492000 1329 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' 

[aligntune] 2026-11-13 06:23:05 WARNING aligntune._imports:279 - Unified RLHF system not available: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.


[aligntune] 2026-11-13 06:23:05 WARNING aligntune._imports:394 - Evaluation system not available: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.


[aligntune] 2026-11-13 06:23:05 WARNING aligntune._imports:431 - Rewards system not available: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.


[aligntune] 2026-11-13 06:23:06 WARNING aligntune._imports:479 - Backend factory not available: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.


AlignTune backend import: FAILED
Error type: ModuleNotFoundError
Error: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.


ModuleNotFoundError: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

## Verify AlignTune and Hugging Face access



In [ ]:
# from google.colab import userdata
# from huggingface_hub import login, model_info
# from transformers import AutoConfig, AutoTokenizer

# MODEL_ID = "CohereLabs/tiny-aya-global"

# # Retrieve token securely from Colab Secrets
# hf_token = userdata.get("HF_LEXI_TOKEN")

# if not hf_token:
#     raise ValueError("HF_LEXI_TOKEN is missing from Colab Secrets.")

# login(token=hf_token, add_to_git_credential=False)

# # Retrieve and pin the exact model revision
# info = model_info(MODEL_ID, token=hf_token)

# config = AutoConfig.from_pretrained(
#     MODEL_ID,
#     revision=info.sha,
#     token=hf_token
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_ID,
#     revision=info.sha,
#     token=hf_token
# )

# print("Model:", MODEL_ID)
# print("Revision:", info.sha)
# print("Architecture:", config.model_type)
# print("Chat template available:", bool(tokenizer.chat_template))
# print("EOS token:", tokenizer.eos_token)
# print("PAD token:", tokenizer.pad_token)

# assert config.model_type == "cohere2"
# assert tokenizer.chat_template is not None

# print("\nModel and tokenizer preflight: PASSED")

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

hf_token = userdata.get("HF_LEXI_TOKEN")

if not hf_token:
    raise ValueError("HF_LEXI_TOKEN not found.")

api = HfApi(token=hf_token)
account = api.whoami()

print("Authenticated Hugging Face account:", account["name"])
print("Model:", "CohereLabs/tiny-aya-global")
print("Token authentication: SUCCESS")


Authenticated Hugging Face account: Jibank
Model: CohereLabs/tiny-aya-global
Token authentication: SUCCESS


In [ ]:

from google.colab import userdata
from huggingface_hub import hf_hub_download
from transformers import AutoConfig, AutoTokenizer

MODEL_ID = "CohereLabs/tiny-aya-global"
hf_token = userdata.get("HF_LEXI_TOKEN")

# Check actual file access before loading model weights
config_file = hf_hub_download(
    repo_id=MODEL_ID,
    filename="config.json",
    token=hf_token
)

config = AutoConfig.from_pretrained(
    MODEL_ID,
    token=hf_token
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=hf_token
)

assert config.model_type == "cohere2"
assert tokenizer.chat_template is not None

print("Model access: SUCCESS")
print("Architecture:", config.model_type)
print("Tokenizer: SUCCESS")
print("Ready for token-length validation.")


config.json:   0%|          | 0.00/1.96k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/8.17k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 21.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/567 [00:00<?, ?B/s]

Model access: SUCCESS
Architecture: cohere2
Tokenizer: SUCCESS
Ready for token-length validation.


## Measure token lengths and freeze shared settings
Our earlier dataset validation checked structure, but not whether entire agreements fit into the configured context length.

The following cell measures the actual training sequence lengths using Tiny Aya's tokenizer and stores one shared configuration for both experiments.

In [ ]:

import json
import math
import statistics
from pathlib import Path

DATA_DIR = Path("/content/curatelaw_m4/datasets")
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/LexiLabs/finetuning_artifacts"
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MAX_SEQ_LENGTH = 3072


def load_jsonl(filename):
    path = DATA_DIR / filename

    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]


raw_records = load_jsonl("raw_150_messages.jsonl")
curated_records = load_jsonl("curated_150_messages.jsonl")

assert len(raw_records) == 150
assert len(curated_records) == 150


def get_token_ids(messages, add_generation_prompt=False):
    result = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
        padding=False,
        truncation=False,
        return_dict=True
    )

    ids = result["input_ids"]

    # Handle a possible single-example batch dimension.
    if ids and isinstance(ids[0], list):
        ids = ids[0]

    if not isinstance(ids, list):
        raise TypeError(
            f"Unexpected input_ids type: {type(ids)}"
        )

    return ids


def analyze_dataset(name, records):
    total_lengths = []
    prompt_lengths = []
    response_lengths = []

    for index, record in enumerate(records, 1):
        messages = record["messages"]

        full_ids = get_token_ids(
            messages,
            add_generation_prompt=False
        )

        prompt_ids = get_token_ids(
            messages[:1],
            add_generation_prompt=True
        )

        response_ids = tokenizer.encode(
            messages[1]["content"],
            add_special_tokens=False
        )

        if len(full_ids) < len(response_ids):
            raise ValueError(
                f"{name} example {index}: "
                "full sequence shorter than response."
            )

        total_lengths.append(len(full_ids))
        prompt_lengths.append(len(prompt_ids))
        response_lengths.append(len(response_ids))

    sorted_lengths = sorted(total_lengths)

    def percentile(p):
        index = math.ceil(p * len(sorted_lengths)) - 1
        return sorted_lengths[max(0, index)]

    stats = {
        "count": len(records),
        "min_tokens": min(total_lengths),
        "median_tokens": statistics.median(total_lengths),
        "p90_tokens": percentile(0.90),
        "p95_tokens": percentile(0.95),
        "max_tokens": max(total_lengths),
        "mean_tokens": round(
            statistics.mean(total_lengths), 2
        ),
        "mean_prompt_tokens": round(
            statistics.mean(prompt_lengths), 2
        ),
        "mean_response_tokens": round(
            statistics.mean(response_lengths), 2
        ),
        "total_training_tokens": sum(total_lengths),
        "over_2048": sum(
            length > 2048 for length in total_lengths
        ),
        "over_3072": sum(
            length > 3072 for length in total_lengths
        ),
        "over_4096": sum(
            length > 4096 for length in total_lengths
        )
    }

    print(f"\n{'=' * 55}")
    print(name)
    print("=" * 55)

    for key, value in stats.items():
        print(f"{key}: {value}")

    return stats


# Sanity check using the first real example.
sample = get_token_ids(raw_records[0]["messages"])

print("Sanity-check token count:", len(sample))
print("First 10 token IDs:", sample[:10])

if len(sample) <= 2:
    raise RuntimeError(
        "Tokenizer still appears to be counting "
        "fields instead of tokens."
    )


raw_stats = analyze_dataset(
    "MODEL A — RAW DATASET",
    raw_records
)

curated_stats = analyze_dataset(
    "MODEL B — CURATED DATASET",
    curated_records
)

report = {
    "model": "CohereLabs/tiny-aya-global",
    "transformers_version": __import__(
        "transformers"
    ).__version__,
    "max_sequence_length_candidate": MAX_SEQ_LENGTH,
    "raw": raw_stats,
    "curated": curated_stats
}

report_path = ARTIFACT_DIR / "token_length_report.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\nCorrected report saved:", report_path)

longest = max(
    raw_stats["max_tokens"],
    curated_stats["max_tokens"]
)

print("\n" + "=" * 55)

if longest <= 2048:
    print("RECOMMENDATION: 2048-token candidate.")
elif longest <= 3072:
    print("RECOMMENDATION: 3072-token candidate.")
elif longest <= 4096:
    print("RECOMMENDATION: Evaluate 4096 on T4.")
else:
    print(
        "WARNING: Examples exceed 4096 tokens. "
        "Resolve context and memory constraints."
    )

print("=" * 55)


Sanity-check token count: 558
First 10 token IDs: [2, 5, 9, 17, 5669, 6604, 61477, 213, 4981, 569]

MODEL A — RAW DATASET
count: 150
min_tokens: 456
median_tokens: 489.5
p90_tokens: 726
p95_tokens: 1850
max_tokens: 2134
mean_tokens: 623.84
mean_prompt_tokens: 444.37
mean_response_tokens: 177.47
total_training_tokens: 93576
over_2048: 2
over_3072: 0
over_4096: 0

MODEL B — CURATED DATASET
count: 150
min_tokens: 456
median_tokens: 487.0
p90_tokens: 701
p95_tokens: 1762
max_tokens: 2082
mean_tokens: 615.73
mean_prompt_tokens: 444.44
mean_response_tokens: 169.29
total_training_tokens: 92359
over_2048: 1
over_3072: 0
over_4096: 0

Corrected report saved: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/token_length_report.json

RECOMMENDATION: 3072-token candidate.


## Save the shared configuration to Google Drive

saves one immutable configuration for Model A and Model B and records the exact dataset checksums and base-model revision.

In [ ]:

import json
import hashlib
import importlib.metadata as md
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

DATA_DIR = Path("/content/curatelaw_m4/datasets")
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/LexiLabs/finetuning_artifacts"
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "CohereLabs/tiny-aya-global"
HF_TOKEN = userdata.get("HF_LEXI_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_LEXI_TOKEN is unavailable.")

model_info = HfApi(token=HF_TOKEN).model_info(MODEL_ID)


def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


raw_file = DATA_DIR / "raw_150_messages.jsonl"
curated_file = DATA_DIR / "curated_150_messages.jsonl"

assert raw_file.exists()
assert curated_file.exists()

# Use verified token statistics from the previous cell.
assert raw_stats["count"] == 150
assert curated_stats["count"] == 150
assert raw_stats["max_tokens"] <= 3072
assert curated_stats["max_tokens"] <= 3072

# Confirm that the datasets have not changed since validation.
assert raw_stats["total_training_tokens"] == 93576
assert curated_stats["total_training_tokens"] == 92359

shared = {
    "experiment": "CurateLaw_M4",
    "model_id": MODEL_ID,
    "model_revision": model_info.sha,
    "backend": "trl",
    "training_method": "qlora_sft",
    "max_seq_length": 3072,
    "quantization": {
        "load_in_4bit": True,
        "bnb_4bit_quant_type": "nf4",
        "bnb_4bit_compute_dtype": "float16",
        "bnb_4bit_use_double_quant": True
    },
    "lora_r": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "learning_rate": 0.0001,
    "num_train_epochs": 2,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "optimizer": "paged_adamw_8bit",
    "weight_decay": 0.01,
    "warmup_ratio": 0.03,
    "fp16": True,
    "bf16": False,
    "gradient_checkpointing": True,
    "packing": False,
    "completion_only_loss": True,
    "seed": 42,
    "data_seed": 42,
    "examples_per_cohort": 150,
    "raw_sha256": file_hash(raw_file),
    "curated_sha256": file_hash(curated_file),
    "raw_total_tokens": 93576,
    "curated_total_tokens": 92359,
    "aligntune_version": md.version("aligntune"),
    "transformers_version": md.version("transformers"),
    "status": "PENDING_TRAINER_SMOKE_TEST"
}

config_path = ARTIFACT_DIR / "shared_training_config.json"

serialized = json.dumps(
    shared, indent=2, sort_keys=True
) + "\n"

if config_path.exists():
    existing = json.loads(config_path.read_text())

    if existing != shared:
        raise RuntimeError(
            "A different shared config already exists. "
            "Inspect it before changing experiment settings."
        )

    print("Existing configuration verified.")
else:
    config_path.write_text(serialized)
    print("Shared configuration created.")

print("\nModel:", shared["model_id"])
print("Revision:", shared["model_revision"])
print("Sequence length:", shared["max_seq_length"])
print("Dataset A SHA-256:", shared["raw_sha256"])
print("Dataset B SHA-256:", shared["curated_sha256"])
print("Config:", config_path)
print("\nConfiguration integrity: PASSED")


## QLoRA compatibility smoke test

In [ ]:

import json
import gc
import torch

from pathlib import Path
from google.colab import userdata
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

CONFIG_PATH = Path(
    "/content/drive/MyDrive/LexiLabs/"
    "finetuning_artifacts/shared_training_config.json"
)

shared = json.loads(CONFIG_PATH.read_text())

MODEL_ID = shared["model_id"]
MODEL_REVISION = shared["model_revision"]
HF_TOKEN = userdata.get("HF_LEXI_TOKEN")

assert shared["backend"] == "trl"
assert shared["training_method"] == "qlora_sft"
assert shared["max_seq_length"] == 3072
assert shared["lora_r"] == 8
assert shared["lora_alpha"] == 16

print("=" * 60)
print("CURATELAW M4 — 4-BIT MODEL SMOKE TEST")
print("=" * 60)

# Free previously allocated GPU memory.
gc.collect()
torch.cuda.empty_cache()

# 4-bit NF4, FP16 computation for Tesla T4.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("\nLoading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=HF_TOKEN,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading 4-bit base model...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=HF_TOKEN,
    quantization_config=quantization_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="sdpa",
)

if not getattr(model, "is_loaded_in_4bit", False):
    raise RuntimeError(
        "Base model is not loaded in 4-bit mode."
    )

print("\nBase model loaded successfully.")
print("Architecture:", model.config.model_type)
print("Quantized:", model.is_loaded_in_4bit)

# Discover projection names used by the model.
linear4bit_names = set()

for name, module in model.named_modules():
    if module.__class__.__name__ == "Linear4bit":
        linear4bit_names.add(name.split(".")[-1])

print("\n4-bit linear module names:")
print(sorted(linear4bit_names))

# Choose only attention projection layers.
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]

missing_modules = [
    name for name in target_modules
    if name not in linear4bit_names
]

if missing_modules:
    raise RuntimeError(
        "Required LoRA projections unavailable: "
        + str(missing_modules)
    )

print("\nPreparing model for QLoRA...")

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },
)

lora_config = LoraConfig(
    r=shared["lora_r"],
    lora_alpha=shared["lora_alpha"],
    lora_dropout=shared["lora_dropout"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_pct = (
    100 * trainable_params / total_params
)

if trainable_params == 0:
    raise RuntimeError(
        "No LoRA parameters are trainable."
    )

if trainable_pct >= 5:
    raise RuntimeError(
        "Too many trainable parameters. "
        "Check whether base weights were frozen."
    )

print("\nTrainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print("Trainable percentage:", f"{trainable_pct:.4f}%")

model.print_trainable_parameters()

allocated_gb = (
    torch.cuda.memory_allocated()
    / (1024 ** 3)
)

reserved_gb = (
    torch.cuda.memory_reserved()
    / (1024 ** 3)
)

print("\nGPU memory allocated:", round(allocated_gb, 2), "GB")
print("GPU memory reserved:", round(reserved_gb, 2), "GB")

print("\n" + "=" * 60)
print("4-BIT BASE MODEL + LORA INITIALIZATION: PASSED")
print("=" * 60)


ModuleNotFoundError: Could not import module 'BloomPreTrainedModel'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

## Inspect the installed AlignTune trainer interface

In [ ]:

import inspect
import importlib.metadata as md

from aligntune.core.backend_factory import (
    create_sft_trainer
)

print("=" * 60)
print("ALIGNTUNE SFT BACKEND INSPECTION")
print("=" * 60)

print("AlignTune:", md.version("aligntune"))
print("TRL:", md.version("trl"))
print("PEFT:", md.version("peft"))
print("Transformers:", md.version("transformers"))

print("\nFactory signature:")
print(inspect.signature(create_sft_trainer))

print("\nFactory source module:")
print(inspect.getmodule(create_sft_trainer).__file__)

source = inspect.getsource(create_sft_trainer)
lines = source.splitlines()

keywords = (
    "dataset_name",
    "dataset_path",
    "backend",
    "quantization",
    "peft",
    "lora",
    "completion_only_loss",
    "gradient_accumulation",
    "seed",
    "trainer",
)

print("\nRelevant factory source lines:")

for number, line in enumerate(lines, 1):
    if any(word in line.lower() for word in keywords):
        print(f"{number:04d}: {line[:220]}")

print("\nInspection completed.")


ModuleNotFoundError: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

## Prepare the smoke-test dataset

In [ ]:

import json
import gc
import hashlib
import torch
from pathlib import Path

DATA_DIR = Path("/content/curatelaw_m4/datasets")
WORK_DIR = Path("/content/curatelaw_m4")
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/LexiLabs/finetuning_artifacts"
)

shared = json.loads(
    (ARTIFACT_DIR / "shared_training_config.json").read_text()
)

raw_path = DATA_DIR / "raw_150_messages.jsonl"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256(raw_path) == shared["raw_sha256"]

with raw_path.open("r", encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(records) == 150

def get_ids(messages):
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        truncation=False,
        return_dict=True
    )
    ids = encoded["input_ids"]
    return ids[0] if ids and isinstance(ids[0], list) else ids

lengths = [
    len(get_ids(record["messages"]))
    for record in records
]

shortest_index = min(
    range(len(records)),
    key=lambda i: lengths[i]
)

longest_index = max(
    range(len(records)),
    key=lambda i: lengths[i]
)

selected = [
    records[shortest_index],
    records[longest_index]
]

smoke_records = []

for record in selected:
    messages = record["messages"]

    assert [m["role"] for m in messages] == [
        "user", "assistant"
    ]

    smoke_records.append({
        "prompt": [
            {
                "role": "user",
                "content": messages[0]["content"]
            }
        ],
        "completion": [
            {
                "role": "assistant",
                "content": messages[1]["content"]
            }
        ]
    })

smoke_dir = WORK_DIR / "smoke_dataset"
smoke_dir.mkdir(parents=True, exist_ok=True)

smoke_path = smoke_dir / "train.jsonl"

with smoke_path.open("w", encoding="utf-8") as f:
    for record in smoke_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Smoke-test records:", len(smoke_records))
print("Shortest example:", lengths[shortest_index], "tokens")
print("Longest example:", lengths[longest_index], "tokens")
print("Smoke dataset:", smoke_path)

# The previous PEFT smoke test left the model on GPU.
# Release it before creating a new AlignTune model instance.
if "model" in globals():
    del model

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU memory allocated after cleanup:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

print("Smoke dataset preparation: PASSED")


Smoke-test records: 2
Shortest example: 456 tokens
Longest example: 2134 tokens
Smoke dataset: /content/curatelaw_m4/smoke_dataset/train.jsonl
GPU memory allocated after cleanup: 0.0 GB
Smoke dataset preparation: PASSED


## Initialize AlignTune's actual trainer

In [ ]:

from aligntune.core.backend_factory import create_sft_trainer

SMOKE_OUTPUT = WORK_DIR / "smoke_output"

trainer = create_sft_trainer(
    model_name=shared["model_id"],
    dataset_name=str(smoke_path),
    backend="trl",
    output_dir=str(SMOKE_OUTPUT),

    # Shared training parameters
    num_epochs=shared["num_train_epochs"],
    batch_size=shared["per_device_train_batch_size"],
    learning_rate=shared["learning_rate"],
    max_seq_length=shared["max_seq_length"],

    # Only one optimizer step for this smoke test
    max_steps=1,

    gradient_accumulation_steps=shared[
        "gradient_accumulation_steps"
    ],

    # LoRA
    peft_enabled=True,
    lora_r=shared["lora_r"],
    lora_alpha=shared["lora_alpha"],
    lora_dropout=shared["lora_dropout"],
    lora_target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],

    # Quantization
    quantization=shared["quantization"],

    # Training objective
    task_type="chat_completion",
    completion_only_loss=True,
    assistant_only_loss=False,

    # Memory and precision
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    # Optimization
    optimizer=shared["optimizer"],
    weight_decay=shared["weight_decay"],
    warmup_ratio=shared["warmup_ratio"],

    # Reproducibility
    seed=shared["seed"],
    data_seed=shared["data_seed"],

    # Dataset
    split="train",
    packing=False,
    auto_detect_fields=True,

    # No external experiment tracker
    report_to="none",
)

print("\nAlignTune trainer created.")
print("Trainer type:", type(trainer))
print("Trainer module:", type(trainer).__module__)

print("\nAvailable training methods:")
for name in [
    "train",
    "save_model",
    "evaluate",
]:
    print(name, callable(getattr(trainer, name, None)))

print("\nTrainer initialization: PASSED")
print("No optimizer steps executed yet.")


ModuleNotFoundError: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

## Inspect the effective trainer configuration

In [ ]:

import inspect
import torch

print("=" * 60)
print("ALIGNTUNE EFFECTIVE TRAINER INSPECTION")
print("=" * 60)

print("\nWrapper:", type(trainer).__name__)

print("\nTrain method signature:")
print(inspect.signature(trainer.train))

# Show actual configuration, not only what was
# passed into the factory.
config_obj = getattr(trainer, "config", None)

if config_obj is not None:
    print("\nEffective AlignTune configuration:")

    sections = [
        "model",
        "training",
        "peft",
        "quantization",
        "dataset",
        "data",
    ]

    for section in sections:
        value = getattr(config_obj, section, None)

        if value is not None:
            print(f"\n[{section}]")

            if hasattr(value, "model_dump"):
                data = value.model_dump()
            elif hasattr(value, "__dict__"):
                data = vars(value)
            else:
                data = str(value)

            print(str(data)[:5000])
else:
    print("\nNo direct config attribute.")

# Find any already-initialized underlying trainer.
print("\nPossible underlying trainer attributes:")

for name in [
    "trainer",
    "_trainer",
    "sft_trainer",
    "_sft_trainer",
]:
    obj = getattr(trainer, name, None)

    if obj is not None:
        print(name, "=>", type(obj).__name__)

        args = getattr(obj, "args", None)

        if args is not None:
            print("max_steps:", getattr(args, "max_steps", None))
            print(
                "gradient_accumulation_steps:",
                getattr(
                    args,
                    "gradient_accumulation_steps",
                    None
                )
            )
            print(
                "max_length:",
                getattr(args, "max_length", None)
            )
            print(
                "completion_only_loss:",
                getattr(
                    args,
                    "completion_only_loss",
                    None
                )
            )

print("\nGPU memory:")
print(
    "Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)
print(
    "Reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GB"
)

print("\nInspection complete. No training executed.")


ALIGNTUNE EFFECTIVE TRAINER INSPECTION


NameError: name 'trainer' is not defined

First, the model configuration also contains lora_rank: 16, but the structured peft.rank is correctly set to 8. We must verify which field the backend actually uses; our earlier independent PEFT test successfully used rank 8.

Second, auto_detect_fields appears as False despite us passing True. That is not necessarily a failure because our dataset has explicit prompt and completion columns, but we must verify that AlignTune preserves them.

In [ ]:

import inspect
from pathlib import Path

print("=" * 60)
print("CURATELAW — FINAL PRE-TRAINING CHECKS")
print("=" * 60)

cfg = trainer.config

# AlignTune stores training settings under config.train.
train_cfg = getattr(cfg, "train", None)

if train_cfg is None:
    raise RuntimeError(
        "AlignTune config.train is missing."
    )

def read_setting(obj, name):
    if isinstance(obj, dict):
        return obj.get(name)
    return getattr(obj, name, None)

expected = {
    "max_steps": 1,
    "per_device_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 0.0001,
    "completion_only_loss": True,
    "fp16": True,
    "packing": False,
    "seed": 42,
}

failures = []

print("\nEffective training settings:")

for name, required in expected.items():
    actual = read_setting(train_cfg, name)
    passed = actual == required

    print(
        f"{name}: {actual!r} "
        f"| Expected: {required!r} "
        f"| {'PASS' if passed else 'FAIL'}"
    )

    if not passed:
        failures.append(name)

peft_cfg = cfg.model.peft

print("\nLoRA configuration:")
print("Enabled:", peft_cfg.enabled)
print("Rank:", peft_cfg.rank)
print("Alpha:", peft_cfg.alpha)
print("Dropout:", peft_cfg.dropout)

if (
    not peft_cfg.enabled
    or peft_cfg.rank != 8
    or peft_cfg.alpha != 16
    or peft_cfg.dropout != 0.05
):
    failures.append("peft_config")

print("\nDataset configuration:")
print("Path:", cfg.dataset.name)
print("Task type:", cfg.dataset.task_type)
print("Auto-detect fields:", cfg.dataset.auto_detect_fields)

if Path(cfg.dataset.name).resolve() != smoke_path.resolve():
    failures.append("dataset_path")

# Check the actual implementation path for lazy initialization.
print("\nTrainer implementation:")
source = inspect.getsource(type(trainer).train)

for number, line in enumerate(source.splitlines(), 1):
    if any(
        keyword in line.lower()
        for keyword in [
            "trainer",
            "dataset",
            "prepare",
            "train(",
            "config",
        ]
    ):
        print(f"{number:03d}: {line[:180]}")

print("\n" + "=" * 60)

if failures:
    print("FAILED SETTINGS:", failures)
    print("DO NOT START TRAINING.")
else:
    print("CONFIGURATION CHECKS PASSED")
    print("Next: inspect loss masking and run one optimizer step.")

print("=" * 60)


CURATELAW — FINAL PRE-TRAINING CHECKS


NameError: name 'trainer' is not defined

One-step AlignTune smoke test

In [ ]:

import gc
import json
import math
import torch
from pathlib import Path

print("=" * 60)
print("CURATELAW M4 — ALIGNTUNE ONE-STEP SMOKE TEST")
print("=" * 60)

# Safety checks: never accidentally run the full experiment.
assert trainer.config.train.max_steps == 1
assert trainer.config.train.completion_only_loss is True
assert trainer.config.train.gradient_accumulation_steps == 4
assert trainer.config.model.peft.enabled is True
assert trainer.config.model.peft.rank == 8

print("One-step limit verified.")
print("Completion-only loss requested.")
print("Starting AlignTune training...")

torch.cuda.reset_peak_memory_stats()

# Executes one optimizer step through AlignTune.
result = trainer.train()

print("\nTraining returned:")
print(str(result)[:3000])

# Verify the underlying TRL trainer was created.
trl_trainer = getattr(trainer, "trainer", None)

if trl_trainer is None:
    raise RuntimeError(
        "AlignTune did not expose its underlying TRL trainer."
    )

print("\nUnderlying trainer:", type(trl_trainer).__name__)

actual_steps = trl_trainer.state.global_step

print("Completed optimizer steps:", actual_steps)

if actual_steps != 1:
    raise RuntimeError(
        f"Expected exactly 1 optimizer step, got {actual_steps}"
    )

# Verify a finite training loss.
losses = [
    item["loss"]
    for item in trl_trainer.state.log_history
    if "loss" in item
]

if not losses:
    raise RuntimeError(
        "No training loss was logged."
    )

if not all(math.isfinite(float(x)) for x in losses):
    raise RuntimeError(
        "Non-finite training loss detected."
    )

print("Logged losses:", losses)

# Check the labels produced by the actual TRL data collator.
train_dataset = trl_trainer.train_dataset

print("\nProcessed dataset columns:")
print(train_dataset.column_names)

sample = train_dataset[0]

batch = trl_trainer.data_collator([sample])

labels = batch["labels"][0].detach().cpu().tolist()
input_ids = batch["input_ids"][0].detach().cpu().tolist()

ignored_positions = sum(x == -100 for x in labels)
trained_positions = sum(x != -100 for x in labels)

print("\nLoss-masking inspection:")
print("Sequence tokens:", len(input_ids))
print("Ignored tokens:", ignored_positions)
print("Supervised tokens:", trained_positions)
print("First 12 labels:", labels[:12])

if ignored_positions == 0:
    raise RuntimeError(
        "No masked positions found. "
        "Completion-only training may not be working."
    )

if trained_positions == 0:
    raise RuntimeError(
        "All tokens are masked. No learning signal exists."
    )

# Check that the trained model remains a PEFT adapter.
trained_model = trl_trainer.model

if not hasattr(trained_model, "peft_config"):
    raise RuntimeError(
        "Trainer model does not expose PEFT configuration."
    )

print("\nPEFT adapter verified.")

# AlignTune's train() implementation saves its model.
output_dir = Path(trainer.config.logging.output_dir)

print("Output directory:", output_dir)

adapter_config = list(
    output_dir.rglob("adapter_config.json")
)

adapter_weights = list(
    output_dir.rglob("adapter_model.safetensors")
)

print("Adapter configuration files:", len(adapter_config))
print("Adapter weight files:", len(adapter_weights))

if not adapter_config or not adapter_weights:
    raise RuntimeError(
        "Adapter files not found after training."
    )

peak_vram = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
)

print("\nPeak GPU allocation:", round(peak_vram, 2), "GB")

print("\n" + "=" * 60)
print("ONE-STEP TRAINING SMOKE TEST: PASSED")
print("=" * 60)


In [ ]:

import math
import json
import torch
from pathlib import Path

print("=" * 60)
print("CURATELAW — POST-TRAINING SMOKE VERIFICATION")
print("=" * 60)

# Reuse existing objects. Do not train again.
trl_trainer = trainer.trainer

# 1. Verify optimizer step
steps = trl_trainer.state.global_step
assert steps == 1, f"Expected 1 optimizer step, got {steps}"

print("\nOptimizer steps:", steps)

# 2. Get loss from the completed AlignTune result
final_loss = result.get("final_loss")

if final_loss is None:
    final_loss = result.get("metrics", {}).get(
        "train_loss"
    )

if final_loss is None or not math.isfinite(
    float(final_loss)
):
    raise RuntimeError("Missing or non-finite training loss.")

print("Final training loss:", final_loss)

# 3. Verify trainer's PEFT model
trained_model = trl_trainer.model

assert hasattr(
    trained_model, "peft_config"
), "Trained model is not a PEFT model."

print("PEFT adapter:", "PRESENT")

# 4. Verify adapter files
output_dir = Path(
    trainer.config.logging.output_dir
)

adapter_configs = list(
    output_dir.rglob("adapter_config.json")
)

adapter_weights = (
    list(output_dir.rglob("adapter_model.safetensors"))
    + list(output_dir.rglob("adapter_model.bin"))
)

print("\nOutput directory:", output_dir)
print("Adapter configuration files:", len(adapter_configs))
print("Adapter weight files:", len(adapter_weights))

if not adapter_configs or not adapter_weights:
    raise RuntimeError(
        "Trained adapter files not found."
    )

# 5. Inspect the processed training data
dataset = trl_trainer.train_dataset

print("\nProcessed dataset columns:")
print(dataset.column_names)

sample = dataset[0]

if "input_ids" not in sample:
    raise RuntimeError(
        "Processed sample has no input_ids."
    )

# Use the trainer's actual data collator.
batch = trl_trainer.data_collator([sample])

labels = batch["labels"][0].detach().cpu().tolist()
input_ids = batch["input_ids"][0].detach().cpu().tolist()

attention_mask = batch.get("attention_mask")

if attention_mask is not None:
    attention = (
        attention_mask[0].detach().cpu().tolist()
    )
else:
    attention = [1] * len(input_ids)

ignored = sum(
    label == -100 and mask == 1
    for label, mask in zip(labels, attention)
)

supervised = sum(
    label != -100 and mask == 1
    for label, mask in zip(labels, attention)
)

print("\nLoss-masking statistics:")
print("Non-padding tokens:", sum(attention))
print("Ignored non-padding tokens:", ignored)
print("Supervised tokens:", supervised)

if ignored <= 0:
    raise RuntimeError(
        "No prompt tokens appear to be masked."
    )

if supervised <= 0:
    raise RuntimeError(
        "No supervised training tokens."
    )

# 6. Inspect supervised token text
supervised_ids = [
    label
    for label, mask in zip(labels, attention)
    if label != -100 and mask == 1
]

supervised_text = tokenizer.decode(
    supervised_ids,
    skip_special_tokens=True
)

print("\nFirst 500 characters of supervised text:")
print("-" * 50)
print(supervised_text[:500])
print("-" * 50)

# 7. Compare supervised text with original response
with smoke_path.open("r", encoding="utf-8") as f:
    smoke_examples = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

expected_response = (
    smoke_examples[0]["completion"][0]["content"]
)

print("\nOriginal response preview:")
print(expected_response[:300])

# 8. Print the positions of supervised tokens
positions = [
    i
    for i, (label, mask) in enumerate(
        zip(labels, attention)
    )
    if label != -100 and mask == 1
]

print("\nFirst supervised position:", positions[0])
print("Last supervised position:", positions[-1])

# 9. GPU statistics
peak_vram = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
)

print("\nPeak GPU allocation:", round(peak_vram, 2), "GB")

print("\n" + "=" * 60)
print("POST-TRAINING BASIC CHECKS PASSED")
print("Review supervised text for correct loss masking.")
print("=" * 60)


## Check the exact masking boundary and AlignTune dataset handling

In [ ]:

import inspect
import json

print("=" * 60)
print("CURATELAW — LOSS MASKING DIAGNOSIS")
print("=" * 60)

# Inspect how AlignTune passes datasets to TRL.
source = inspect.getsource(type(trainer).train)
lines = source.splitlines()

print("\nAlignTune dataset-selection implementation:")
for i in range(58, min(87, len(lines))):
    print(f"{i+1:03d}: {lines[i]}")

# Inspect the already-processed dataset.
processed = trl_trainer.train_dataset

print("\nProcessed dataset columns:", processed.column_names)

with smoke_path.open("r", encoding="utf-8") as f:
    original = json.loads(next(f))

messages = (
    original["prompt"] + original["completion"]
)

def ids_for(messages, generation_prompt=False):
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=generation_prompt,
        truncation=False,
        return_dict=True
    )

    ids = encoded["input_ids"]

    if ids and isinstance(ids[0], list):
        ids = ids[0]

    return ids

full_ids = ids_for(messages)
prefix_ids = ids_for(
    original["prompt"],
    generation_prompt=True
)

stored_ids = processed[0]["input_ids"]
stored_labels = processed[0]["labels"]

# Compare tokenized data to the original example.
print("\nFull sequence length:", len(full_ids))
print("Prompt prefix length:", len(prefix_ids))
print("Stored sequence length:", len(stored_ids))
print("Stored labels length:", len(stored_labels))

prefix_matches = (
    full_ids[:len(prefix_ids)] == prefix_ids
)

stored_matches = stored_ids == full_ids

print("Prompt prefix matches full sequence:", prefix_matches)
print("Stored tokens match original:", stored_matches)

if not prefix_matches:
    print(
        "WARNING: Prompt-prefix tokens differ. "
        "Do not construct labels until alignment is resolved."
    )
else:
    # Demonstrate correct labels without modifying
    # the existing trainer.
    corrected_labels = (
        [-100] * len(prefix_ids)
        + full_ids[len(prefix_ids):]
    )

    assert len(corrected_labels) == len(full_ids)

    print("\nExpected masked tokens:", len(prefix_ids))
    print(
        "Expected supervised tokens:",
        len(full_ids) - len(prefix_ids)
    )

    response_ids = [
        token
        for token in corrected_labels
        if token != -100
    ]

    response_text = tokenizer.decode(
        response_ids,
        skip_special_tokens=True
    )

    print("\nExpected supervised-text preview:")
    print(response_text[:400])

print("\nDiagnostic completed. No training performed.")


In [ ]:

import gc
import json
import math
import torch

from pathlib import Path
from datasets import Dataset
from aligntune.core.backend_factory import create_sft_trainer

print("=" * 60)
print("CURATELAW — CORRECTED ALIGNTUNE SMOKE TEST")
print("=" * 60)

WORK_DIR = Path("/content/curatelaw_m4")
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/LexiLabs/finetuning_artifacts"
)

shared = json.loads(
    (ARTIFACT_DIR / "shared_training_config.json").read_text()
)

smoke_path = WORK_DIR / "smoke_dataset/train.jsonl"

with smoke_path.open("r", encoding="utf-8") as f:
    smoke_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(smoke_records) == 2

# These records must contain prompt/completion only.
clean_records = []

for record in smoke_records:
    assert set(record) == {"prompt", "completion"}

    clean_records.append({
        "prompt": record["prompt"],
        "completion": record["completion"]
    })

clean_dataset = Dataset.from_list(clean_records)

assert clean_dataset.column_names == [
    "prompt", "completion"
]

print("Dataset columns:", clean_dataset.column_names)
print("Examples:", len(clean_dataset))

# Release the previous smoke-test trainer and model.
for name in ["trainer", "trl_trainer", "trained_model", "model"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# Initialize from the original model, not the previous
# smoke-test adapter.
corrected_trainer = create_sft_trainer(
    model_name=shared["model_id"],
    dataset_name=str(smoke_path),
    backend="trl",
    output_dir=str(WORK_DIR / "smoke_corrected"),

    num_epochs=shared["num_train_epochs"],
    batch_size=shared["per_device_train_batch_size"],
    learning_rate=shared["learning_rate"],
    max_seq_length=shared["max_seq_length"],

    max_steps=1,
    gradient_accumulation_steps=shared[
        "gradient_accumulation_steps"
    ],

    peft_enabled=True,
    lora_r=shared["lora_r"],
    lora_alpha=shared["lora_alpha"],
    lora_dropout=shared["lora_dropout"],
    lora_target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],

    quantization=shared["quantization"],

    task_type="chat_completion",
    completion_only_loss=True,
    assistant_only_loss=False,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optimizer=shared["optimizer"],
    weight_decay=shared["weight_decay"],
    warmup_ratio=shared["warmup_ratio"],

    seed=shared["seed"],
    data_seed=shared["data_seed"],

    packing=False,
    report_to="none",
)

# Critical fix:
# AlignTune normally converts the data to a messages
# column, which causes full-conversation training.
# Supply prompt/completion directly to the training stage.
corrected_trainer.dataset = clean_dataset
corrected_trainer.train_dataset = clean_dataset

assert corrected_trainer.config.train.max_steps == 1
assert corrected_trainer.config.train.completion_only_loss

print("\nClean prompt-completion dataset injected.")
print("Starting one-step smoke test...")

torch.cuda.reset_peak_memory_stats()

corrected_result = corrected_trainer.train()

print("\nTraining result:")
print(str(corrected_result)[:2000])

underlying = corrected_trainer.trainer

assert underlying.state.global_step == 1

final_loss = corrected_result.get(
    "final_loss",
    corrected_result.get("metrics", {}).get("train_loss")
)

assert final_loss is not None
assert math.isfinite(float(final_loss))

# Inspect actual labels after TRL preprocessing.
processed = underlying.train_dataset

print("\nProcessed columns:", processed.column_names)

sample = processed[0]

batch = underlying.data_collator([sample])

labels = batch["labels"][0].cpu().tolist()
input_ids = batch["input_ids"][0].cpu().tolist()

attention = batch.get("attention_mask")

if attention is not None:
    attention = attention[0].cpu().tolist()
else:
    attention = [1] * len(input_ids)

ignored = sum(
    label == -100 and mask == 1
    for label, mask in zip(labels, attention)
)

supervised = sum(
    label != -100 and mask == 1
    for label, mask in zip(labels, attention)
)

print("\nLoss masking:")
print("Ignored prompt tokens:", ignored)
print("Supervised completion tokens:", supervised)

if ignored == 0 or supervised == 0:
    raise RuntimeError(
        "Completion-only masking is still incorrect."
    )

# Independently verify the prefix boundary.
prefix = tokenizer.apply_chat_template(
    smoke_records[0]["prompt"],
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    truncation=False,
)

prefix_ids = prefix["input_ids"]

if prefix_ids and isinstance(prefix_ids[0], list):
    prefix_ids = prefix_ids[0]

if input_ids[:len(prefix_ids)] != prefix_ids:
    raise RuntimeError(
        "Processed prompt tokens differ from source."
    )

if any(label != -100 for label in labels[:len(prefix_ids)]):
    raise RuntimeError(
        "Some user-prompt tokens are supervised."
    )

supervised_ids = [
    label
    for label, mask in zip(labels, attention)
    if label != -100 and mask == 1
]

supervised_text = tokenizer.decode(
    supervised_ids,
    skip_special_tokens=True
)

print("\nSupervised text:")
print(supervised_text[:400])

# Verify adapter output.
output_dir = Path(corrected_trainer.config.logging.output_dir)

adapter_configs = list(
    output_dir.rglob("adapter_config.json")
)
adapter_weights = (
    list(output_dir.rglob("adapter_model.safetensors"))
    + list(output_dir.rglob("adapter_model.bin"))
)

assert adapter_configs and adapter_weights

print("\nFinal loss:", final_loss)
print("Optimizer steps:", underlying.state.global_step)
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "GB"
)
print("Adapter saved:", output_dir)

print("\n" + "=" * 60)
print("CORRECTED ALIGNTUNE SMOKE TEST: PASSED")
print("=" * 60)


In [ ]:

import json
import torch

print("=" * 60)
print("CURATELAW — POST-OOM LOSS MASK VERIFICATION")
print("=" * 60)

inner = getattr(corrected_trainer, "trainer", None)

if inner is None:
    raise RuntimeError(
        "Underlying TRL trainer was not initialized."
    )

dataset = inner.train_dataset

print("Dataset columns:", dataset.column_names)
print("Dataset size:", len(dataset))
print("Configured max steps:", inner.args.max_steps)
print(
    "Completion-only loss:",
    getattr(inner.args, "completion_only_loss", None)
)

assert len(dataset) == 2

for index, original in enumerate(smoke_records):
    sample = dataset[index]

    input_ids = sample["input_ids"]
    labels = sample["labels"]

    assert len(input_ids) == len(labels)

    # Reconstruct the expected user prompt boundary.
    encoded = tokenizer.apply_chat_template(
        original["prompt"],
        tokenize=True,
        add_generation_prompt=True,
        truncation=False,
        return_dict=True
    )

    prefix_ids = encoded["input_ids"]

    if prefix_ids and isinstance(prefix_ids[0], list):
        prefix_ids = prefix_ids[0]

    prompt_length = len(prefix_ids)

    prefix_matches = (
        input_ids[:prompt_length] == prefix_ids
    )

    prompt_masked = all(
        value == -100
        for value in labels[:prompt_length]
    )

    completion_supervised = all(
        value != -100
        for value in labels[prompt_length:]
    )

    supervised_count = sum(
        value != -100 for value in labels
    )

    print(f"\nExample {index + 1}")
    print("Total tokens:", len(input_ids))
    print("Prompt tokens:", prompt_length)
    print("Supervised tokens:", supervised_count)
    print("Prefix matches:", prefix_matches)
    print("Prompt masked:", prompt_masked)
    print("Completion supervised:", completion_supervised)

    if not (
        prefix_matches
        and prompt_masked
        and completion_supervised
        and supervised_count > 0
    ):
        raise RuntimeError(
            f"Incorrect loss masking in example {index + 1}"
        )

print("\nGPU allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2),
      "GB")

print("GPU reserved:",
      round(torch.cuda.memory_reserved() / 1024**3, 2),
      "GB")

print("\nLOSS MASKING VERIFICATION: PASSED")
print("No additional training performed.")


In [ ]:

import gc
import json
import math
import hashlib
import torch

from pathlib import Path
from datasets import Dataset
from google.colab import userdata
from huggingface_hub import HfApi
from transformers import AutoTokenizer
from aligntune.core.backend_factory import create_sft_trainer

print("=" * 60)
print("CURATELAW — CLEAN T4 LONG-SEQUENCE SMOKE TEST")
print("=" * 60)

ROOT = Path("/content/curatelaw_m4")
DATA_DIR = ROOT / "datasets"
ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/LexiLabs/finetuning_artifacts"
)

shared = json.loads(
    (ARTIFACT_DIR / "shared_training_config.json").read_text()
)

raw_path = DATA_DIR / "raw_150_messages.jsonl"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert sha256(raw_path) == shared["raw_sha256"]

token = userdata.get("HF_LEXI_TOKEN")

# Verify the checkpoint has not changed.
remote_revision = HfApi(token=token).model_info(
    shared["model_id"]
).sha

assert remote_revision == shared["model_revision"], (
    "Model revision has changed. "
    "Pin the original revision before training."
)

tokenizer = AutoTokenizer.from_pretrained(
    shared["model_id"],
    revision=shared["model_revision"],
    token=token
)

records = [
    json.loads(line)
    for line in raw_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

def token_length(record):
    encoded = tokenizer.apply_chat_template(
        record["messages"],
        tokenize=True,
        truncation=False,
        return_dict=True
    )

    ids = encoded["input_ids"]
    return len(ids[0]) if ids and isinstance(ids[0], list) else len(ids)

longest = max(records, key=token_length)

length = token_length(longest)

print("Selected sequence length:", length)
assert length == 2134
assert length <= shared["max_seq_length"]

# Preserve the prompt/completion representation.
messages = longest["messages"]

clean_record = {
    "prompt": [messages[0]],
    "completion": [messages[1]]
}

clean_dataset = Dataset.from_list([clean_record])

smoke_dir = ROOT / "long_smoke"
smoke_dir.mkdir(parents=True, exist_ok=True)

input_path = smoke_dir / "train.jsonl"

input_path.write_text(
    json.dumps(clean_record, ensure_ascii=False) + "\n",
    encoding="utf-8"
)

gc.collect()
torch.cuda.empty_cache()

print(
    "GPU allocated before training:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

assert torch.cuda.memory_allocated() < 1 * 1024**3, (
    "GPU is not clean. Restart the runtime."
)

long_trainer = create_sft_trainer(
    model_name=shared["model_id"],
    dataset_name=str(input_path),
    backend="trl",
    output_dir=str(smoke_dir / "output"),

    num_epochs=2,
    batch_size=1,
    learning_rate=shared["learning_rate"],
    max_seq_length=shared["max_seq_length"],
    max_steps=1,

    gradient_accumulation_steps=4,

    peft_enabled=True,
    lora_r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    lora_target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],

    quantization=shared["quantization"],

    task_type="chat_completion",
    completion_only_loss=True,
    assistant_only_loss=False,

    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optimizer=shared["optimizer"],
    weight_decay=shared["weight_decay"],
    warmup_ratio=shared["warmup_ratio"],

    seed=42,
    data_seed=42,
    packing=False,
    report_to="none",
)

# Bypass AlignTune's conversation-flattening stage.
long_trainer.dataset = clean_dataset
long_trainer.train_dataset = clean_dataset

assert long_trainer.config.train.max_steps == 1
assert long_trainer.config.train.completion_only_loss
assert long_trainer.config.model.peft.rank == 8

torch.cuda.reset_peak_memory_stats()

print("\nStarting one-step long-sequence training...")

try:
    long_result = long_trainer.train()

    actual_steps = long_trainer.trainer.state.global_step

    final_loss = long_result.get("final_loss")

    assert actual_steps == 1
    assert final_loss is not None
    assert math.isfinite(float(final_loss))

    processed = long_trainer.trainer.train_dataset[0]

    labels = processed["labels"]
    ids = processed["input_ids"]

    prefix_encoded = tokenizer.apply_chat_template(
        clean_record["prompt"],
        tokenize=True,
        add_generation_prompt=True,
        truncation=False,
        return_dict=True
    )

    prefix = prefix_encoded["input_ids"]

    if prefix and isinstance(prefix[0], list):
        prefix = prefix[0]

    assert ids[:len(prefix)] == prefix
    assert all(x == -100 for x in labels[:len(prefix)])
    assert all(x != -100 for x in labels[len(prefix):])

    output_dir = Path(
        long_trainer.config.logging.output_dir
    )

    assert list(output_dir.rglob("adapter_config.json"))
    assert (
        list(output_dir.rglob("adapter_model.safetensors"))
        or list(output_dir.rglob("adapter_model.bin"))
    )

    print("\nTraining: SUCCESS")
    print("Optimizer steps:", actual_steps)
    print("Final loss:", final_loss)
    print("Ignored prompt tokens:", len(prefix))
    print(
        "Supervised completion tokens:",
        len(labels) - len(prefix)
    )

except torch.OutOfMemoryError:
    print("\nTraining: FAILED — CUDA OUT OF MEMORY")
    print(
        "The long sequence still exceeds T4 memory "
        "under the current loss implementation."
    )
    raise

finally:
    print(
        "\nPeak GPU allocation:",
        round(
            torch.cuda.max_memory_allocated() / 1024**3,
            2
        ),
        "GB"
    )

print("\nLONG-SEQUENCE SMOKE TEST: PASSED")


## Model A — full fine-tuning on 150 raw examples.

In [ ]:
# Step 1 — Prepare Model A
import os
import gc
import csv
import json
import math
import shutil
import hashlib
import importlib.metadata as md
from pathlib import Path

import torch
from datasets import Dataset
from google.colab import userdata
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from aligntune.core.backend_factory import create_sft_trainer

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/LexiLabs")
ARTIFACT_DIR = DRIVE_ROOT / "finetuning_artifacts"
LOCAL_ROOT = Path("/content/curatelaw_m4")
DATA_DIR = LOCAL_ROOT / "datasets"

# RUN_NAME = "model_a_raw"
# RUN_DIR = ARTIFACT_DIR / RUN_NAME
# OUTPUT_DIR = RUN_DIR / "checkpoints"

RUN_NAME = "model_a_raw"

# New attempt, preserving any earlier files.
attempt = 2

while (
    ARTIFACT_DIR / f"{RUN_NAME}_attempt_{attempt:02d}"
).exists():
    attempt += 1

RUN_DIR = (
    ARTIFACT_DIR
    / f"{RUN_NAME}_attempt_{attempt:02d}"
)

OUTPUT_DIR = RUN_DIR / "checkpoints"

print("New training attempt:", RUN_DIR)

DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Read frozen shared configuration
# --------------------------------------------------

shared_path = ARTIFACT_DIR / "shared_training_config.json"

shared = json.loads(
    shared_path.read_text(encoding="utf-8")
)

assert shared["model_id"] == "CohereLabs/tiny-aya-global"
assert shared["max_seq_length"] == 3072
assert shared["training_method"] == "qlora_sft"

assert shared["lora_r"] == 8
assert shared["lora_alpha"] == 16
assert shared["lora_dropout"] == 0.05

assert shared["learning_rate"] == 0.0001
assert shared["num_train_epochs"] == 2
assert shared["gradient_accumulation_steps"] == 4

assert torch.cuda.is_available()

print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# 3. Copy and verify original dataset
# --------------------------------------------------

def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


raw_source = DRIVE_ROOT / "raw_150_messages.jsonl"
raw_path = DATA_DIR / "raw_150_messages.jsonl"

if not raw_source.is_file():
    raise FileNotFoundError(raw_source)

shutil.copy2(raw_source, raw_path)

actual_hash = sha256(raw_path)

assert actual_hash == shared["raw_sha256"], (
    "Raw dataset differs from frozen configuration."
)

records = [
    json.loads(line)
    for line in raw_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(records) == 150

print("Raw examples:", len(records))
print("Raw SHA-256:", actual_hash)

# --------------------------------------------------
# 4. Load exact base-model snapshot
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_LEXI_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_LEXI_TOKEN missing.")

print("\nPreparing frozen model snapshot...")

snapshot_path = snapshot_download(
    repo_id=shared["model_id"],
    revision=shared["model_revision"],
    token=HF_TOKEN,
    allow_patterns=[
        "*.safetensors",
        "*.json",
        "*.model",
        "*.tiktoken",
        "tokenizer*",
        "*.py",
        "*.txt",
    ],
)

BASE_MODEL_PATH = str(snapshot_path)

print("Frozen checkpoint:", shared["model_revision"])
print("Snapshot location:", BASE_MODEL_PATH)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_PATH
)

assert tokenizer.chat_template is not None

# --------------------------------------------------
# 5. Convert to prompt/completion format
# --------------------------------------------------

clean_records = []

for index, record in enumerate(records, 1):
    messages = record["messages"]

    if (
        len(messages) != 2
        or messages[0]["role"] != "user"
        or messages[1]["role"] != "assistant"
    ):
        raise ValueError(
            f"Invalid conversation at example {index}"
        )

    clean_records.append({
        "prompt": [messages[0]],
        "completion": [messages[1]],
    })

clean_dataset = Dataset.from_list(clean_records)

assert clean_dataset.column_names == [
    "prompt", "completion"
]

# --------------------------------------------------
# 6. Independently verify all token boundaries
# --------------------------------------------------

def token_ids(messages, generation_prompt=False):
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=generation_prompt,
        truncation=False,
        return_dict=True,
    )

    ids = encoded["input_ids"]

    if ids and isinstance(ids[0], list):
        ids = ids[0]

    return ids


total_lengths = []
supervised_lengths = []

for index, record in enumerate(clean_records, 1):
    messages = record["prompt"] + record["completion"]

    full_ids = token_ids(messages)
    prompt_ids = token_ids(
        record["prompt"],
        generation_prompt=True
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise RuntimeError(
            f"Prompt alignment failed at example {index}"
        )

    completion_length = len(full_ids) - len(prompt_ids)

    if completion_length <= 0:
        raise RuntimeError(
            f"Empty completion at example {index}"
        )

    if len(full_ids) > shared["max_seq_length"]:
        raise RuntimeError(
            f"Example {index} exceeds sequence limit."
        )

    total_lengths.append(len(full_ids))
    supervised_lengths.append(completion_length)

print("\nTraining data validation:")
print("Examples:", len(clean_dataset))
print("Maximum sequence:", max(total_lengths))
print("Total sequence tokens:", sum(total_lengths))
print("Total supervised tokens:", sum(supervised_lengths))

assert sum(total_lengths) == 93576

# --------------------------------------------------
# 7. Save run metadata before training
# --------------------------------------------------

if RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise RuntimeError(
        "Model A output directory already contains files. "
        "Do not overwrite an existing experiment."
    )

RUN_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "run_name": RUN_NAME,
    "model_id": shared["model_id"],
    "model_revision": shared["model_revision"],
    "dataset": raw_source.name,
    "dataset_sha256": actual_hash,
    "examples": len(clean_dataset),
    "max_sequence_length": max(total_lengths),
    "total_sequence_tokens": sum(total_lengths),
    "total_supervised_tokens": sum(supervised_lengths),
    "shared_config_sha256": sha256(shared_path),
    "aligntune_version": md.version("aligntune"),
    "transformers_version": md.version("transformers"),
    "trl_version": md.version("trl"),
    "status": "PREPARED",
}

(RUN_DIR / "run_metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

shutil.copy2(
    shared_path,
    RUN_DIR / "shared_training_config.json"
)

print("\nModel A preparation: PASSED")
print("Output directory:", RUN_DIR)
print("No training performed yet.")


ModuleNotFoundError: Could not import module 'pipeline'. Are this object's requirements defined correctly? Set the logging verbosity to DEBUG for the original import error.

In [ ]:
# Step 2 — Initialize the full Model A trainer
import torch
from aligntune.core.backend_factory import create_sft_trainer

print("=" * 60)
print("CURATELAW M4 — MODEL A TRAINER")
print("=" * 60)

gc.collect()
torch.cuda.empty_cache()

assert torch.cuda.memory_allocated() < 1 * 1024**3, (
    "GPU memory is not clean. Restart the runtime."
)

model_a_trainer = create_sft_trainer(
    model_name=BASE_MODEL_PATH,
    dataset_name=str(raw_path),
    backend="trl",
    output_dir=str(OUTPUT_DIR),

    num_epochs=shared["num_train_epochs"],
    batch_size=shared["per_device_train_batch_size"],
    learning_rate=shared["learning_rate"],
    max_seq_length=shared["max_seq_length"],

    # Full training: follow epoch count.
    max_steps=-1,

    gradient_accumulation_steps=shared[
        "gradient_accumulation_steps"
    ],

    peft_enabled=True,
    lora_r=shared["lora_r"],
    lora_alpha=shared["lora_alpha"],
    lora_dropout=shared["lora_dropout"],
    lora_target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],

    quantization=shared["quantization"],

    task_type="chat_completion",
    completion_only_loss=True,
    assistant_only_loss=False,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optimizer=shared["optimizer"],
    weight_decay=shared["weight_decay"],
    warmup_ratio=shared["warmup_ratio"],

    seed=shared["seed"],
    data_seed=shared["data_seed"],

    packing=False,

    # Persist intermediate training progress.
    save_strategy="steps",
    save_steps=10,
    logging_steps=1,

    report_to="none",
)

# CRITICAL:
# Bypass AlignTune's conversation normalization,
# which previously removed completion-only masking.
model_a_trainer.dataset = clean_dataset
model_a_trainer.train_dataset = clean_dataset

cfg = model_a_trainer.config

assert cfg.model.peft.enabled
assert cfg.model.peft.rank == 8
assert cfg.model.peft.alpha == 16

assert cfg.train.max_steps == -1
assert cfg.train.gradient_accumulation_steps == 4
assert cfg.train.completion_only_loss is True
assert cfg.train.packing is False
assert cfg.train.seed == 42

assert cfg.model.max_seq_length == 3072

print("Training examples:", len(clean_dataset))
print("Epochs:", cfg.train.epochs)
print("Max steps:", cfg.train.max_steps)
print("Learning rate:", cfg.train.learning_rate)
print("Completion-only loss:", cfg.train.completion_only_loss)
print("Output directory:", cfg.logging.output_dir)

assert cfg.train.epochs == 2

print("\nTrainer configuration: PASSED")
print("Ready to train Model A.")


In [ ]:
# Step 3 — Train Model A and save artifacts
import json
import csv
import math
import torch

print("=" * 60)
print("CURATELAW M4 — MODEL A RAW FINE-TUNING")
print("=" * 60)

assert len(model_a_trainer.train_dataset) == 150
assert model_a_trainer.config.train.max_steps == -1
assert model_a_trainer.config.train.epochs == 2
assert model_a_trainer.config.train.completion_only_loss

metadata_path = RUN_DIR / "run_metadata.json"

metadata = json.loads(metadata_path.read_text())
metadata["status"] = "TRAINING_STARTED"

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

torch.cuda.reset_peak_memory_stats()

# Full 150-example Model A training.
model_a_result = model_a_trainer.train()

inner = model_a_trainer.trainer

steps = inner.state.global_step

final_loss = model_a_result.get(
    "final_loss",
    model_a_result.get("metrics", {}).get("train_loss")
)

if final_loss is None or not math.isfinite(
    float(final_loss)
):
    raise RuntimeError("Training did not produce a finite loss.")

# Confirm all 150 examples survived preprocessing.
processed_dataset = inner.train_dataset

assert len(processed_dataset) == 150

# Verify completion masking across all examples.
for index, example in enumerate(processed_dataset):
    labels = example["labels"]

    if not (
        any(label == -100 for label in labels)
        and any(label != -100 for label in labels)
    ):
        raise RuntimeError(
            f"Invalid loss labels for example {index}"
        )

# Verify the final LoRA adapter.
adapter_configs = list(
    OUTPUT_DIR.rglob("adapter_config.json")
)

adapter_weights = (
    list(OUTPUT_DIR.rglob("adapter_model.safetensors"))
    + list(OUTPUT_DIR.rglob("adapter_model.bin"))
)

if not adapter_configs or not adapter_weights:
    raise RuntimeError("Final adapter was not saved.")

# Save the trainer's log history.
history = inner.state.log_history

(RUN_DIR / "training_log_history.json").write_text(
    json.dumps(history, indent=2, default=str),
    encoding="utf-8"
)

# Prepare step-level loss history for plotting.
loss_rows = []

for entry in history:
    if "loss" in entry:
        loss_rows.append({
            "step": entry.get("step"),
            "epoch": entry.get("epoch"),
            "loss": entry["loss"],
            "learning_rate": entry.get(
                "learning_rate"
            ),
        })

loss_csv = RUN_DIR / "loss_history.csv"

with loss_csv.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "step",
            "epoch",
            "loss",
            "learning_rate",
        ]
    )

    writer.writeheader()
    writer.writerows(loss_rows)

# Save complete result metadata.
peak_memory_gb = (
    torch.cuda.max_memory_allocated()
    / (1024 ** 3)
)

metadata["status"] = "TRAINING_COMPLETED"
metadata["optimizer_steps"] = steps
metadata["final_loss"] = float(final_loss)
metadata["peak_gpu_memory_gb"] = round(
    peak_memory_gb, 3
)
metadata["adapter_output_dir"] = str(OUTPUT_DIR)
metadata["logged_loss_points"] = len(loss_rows)

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

(RUN_DIR / "training_result.json").write_text(
    json.dumps(model_a_result, indent=2, default=str),
    encoding="utf-8"
)

print("\n" + "=" * 60)
print("MODEL A TRAINING COMPLETED")
print("=" * 60)

print("Optimizer steps:", steps)
print("Final training loss:", final_loss)
print("Peak GPU memory:", round(peak_memory_gb, 2), "GB")
print("Loss points saved:", len(loss_rows))

print("\nSaved artifacts:")
print("Adapter:", OUTPUT_DIR)
print("Metadata:", metadata_path)
print("Loss history:", loss_csv)
print("Training logs:", RUN_DIR / "training_log_history.json")

print("\nModel A: COMPLETED")


In [ ]:

from google.colab import drive
from pathlib import Path
import json

drive.mount("/content/drive")

RUN_DIR = Path(
    "/content/drive/MyDrive/LexiLabs/"
    "finetuning_artifacts/model_a_raw"
)

print("=" * 60)
print("MODEL A — RECOVERY DIAGNOSTIC")
print("=" * 60)

print("Output directory exists:", RUN_DIR.exists())

if RUN_DIR.exists():
    metadata_path = RUN_DIR / "run_metadata.json"

    if metadata_path.exists():
        metadata = json.loads(
            metadata_path.read_text(encoding="utf-8")
        )
        print("Recorded status:", metadata.get("status"))
        print("Optimizer steps:", metadata.get("optimizer_steps"))
    else:
        print("No run metadata found.")

    print("\nSaved files:")
    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            print(path.relative_to(RUN_DIR))

    checkpoints = sorted(
        path
        for path in RUN_DIR.rglob("checkpoint-*")
        if path.is_dir()
    )

    print("\nCheckpoint directories:", len(checkpoints))

    for checkpoint in checkpoints:
        state_file = checkpoint / "trainer_state.json"
        if state_file.exists():
            state = json.loads(state_file.read_text())
            print(
                checkpoint.name,
                "step:",
                state.get("global_step")
            )

    adapters = list(RUN_DIR.rglob("adapter_config.json"))
    print("\nAdapter configs found:", len(adapters))

else:
    print("No previous run directory.")

print("\nRecovery inspection complete.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
MODEL A — RECOVERY DIAGNOSTIC
Output directory exists: True
Recorded status: TRAINING_STARTED
Optimizer steps: None

Saved files:
checkpoints/README.md
checkpoints/checkpoint-10/README.md
checkpoints/checkpoint-10/adapter_config.json
checkpoints/checkpoint-10/adapter_model.safetensors
checkpoints/checkpoint-10/chat_template.jinja
checkpoints/checkpoint-10/optimizer.pt
checkpoints/checkpoint-10/rng_state.pth
checkpoints/checkpoint-10/scheduler.pt
checkpoints/checkpoint-10/tokenizer.json
checkpoints/checkpoint-10/tokenizer_config.json
checkpoints/checkpoint-10/trainer_state.json
checkpoints/checkpoint-10/training_args.bin
checkpoints/checkpoint-20/README.md
checkpoints/checkpoint-20/adapter_config.json
checkpoints/checkpoint-20/adapter_model.safetensors
checkpoints/checkpoint-20/chat_template.jinja
checkpoints/checkpoint-20/optimizer.pt
checkpoints/checkpoint-2

In [ ]:

import json
import gc
import hashlib
import shutil
import math
import torch

from pathlib import Path
from unittest.mock import patch
from datasets import Dataset
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from trl import SFTTrainer
from aligntune.core.backend_factory import create_sft_trainer

drive.mount("/content/drive")

print("=" * 60)
print("CURATELAW — RESUME MODEL A FROM STEP 30")
print("=" * 60)

# --------------------------------------------------
# 1. Restore paths and configuration
# --------------------------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive/LexiLabs")
ARTIFACT_DIR = DRIVE_ROOT / "finetuning_artifacts"

RUN_DIR = ARTIFACT_DIR / "model_a_raw"
OUTPUT_DIR = RUN_DIR / "checkpoints"
CHECKPOINT = OUTPUT_DIR / "checkpoint-30"

LOCAL_DIR = Path("/content/curatelaw_m4/datasets")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

shared_path = ARTIFACT_DIR / "shared_training_config.json"
shared = json.loads(shared_path.read_text())

assert shared["model_id"] == "CohereLabs/tiny-aya-global"
assert shared["max_seq_length"] == 3072

# --------------------------------------------------
# 2. Verify saved checkpoint
# --------------------------------------------------

required = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "optimizer.pt",
    "scheduler.pt",
    "trainer_state.json",
    "rng_state.pth",
]

missing = [
    filename
    for filename in required
    if not (CHECKPOINT / filename).is_file()
]

if missing:
    raise RuntimeError(
        f"Incomplete checkpoint: {missing}"
    )

state = json.loads(
    (CHECKPOINT / "trainer_state.json").read_text()
)

saved_step = state["global_step"]

assert saved_step == 30

print("Checkpoint:", CHECKPOINT)
print("Saved optimizer step:", saved_step)

# --------------------------------------------------
# 3. Verify dataset integrity
# --------------------------------------------------

def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            digest.update(block)

    return digest.hexdigest()


source = DRIVE_ROOT / "raw_150_messages.jsonl"
raw_path = LOCAL_DIR / "raw_150_messages.jsonl"

shutil.copy2(source, raw_path)

assert sha256(raw_path) == shared["raw_sha256"]

records = [
    json.loads(line)
    for line in raw_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(records) == 150

clean_records = []

for record in records:
    messages = record["messages"]

    assert len(messages) == 2
    assert messages[0]["role"] == "user"
    assert messages[1]["role"] == "assistant"

    clean_records.append({
        "prompt": [messages[0]],
        "completion": [messages[1]]
    })

clean_dataset = Dataset.from_list(clean_records)

print("Dataset integrity: VERIFIED")
print("Examples:", len(clean_dataset))

# --------------------------------------------------
# 4. Restore original frozen base model
# --------------------------------------------------

token = userdata.get("HF_LEXI_TOKEN")

if not token:
    raise RuntimeError("HF_LEXI_TOKEN missing.")

base_path = snapshot_download(
    repo_id=shared["model_id"],
    revision=shared["model_revision"],
    token=token,
)

print("Base model revision:", shared["model_revision"])

tokenizer = AutoTokenizer.from_pretrained(base_path)

assert tokenizer.chat_template is not None

# --------------------------------------------------
# 5. Ensure the GPU is clean
# --------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

allocated_gb = (
    torch.cuda.memory_allocated() / 1024**3
)

print("GPU allocated before loading:", round(allocated_gb, 2), "GB")

if allocated_gb >= 1:
    raise RuntimeError(
        "GPU is not clean. Restart the Colab session."
    )

# --------------------------------------------------
# 6. Recreate original AlignTune configuration
# --------------------------------------------------

resume_trainer = create_sft_trainer(
    model_name=str(base_path),
    dataset_name=str(raw_path),

    backend="trl",
    output_dir=str(OUTPUT_DIR),

    num_epochs=shared["num_train_epochs"],
    batch_size=shared["per_device_train_batch_size"],
    learning_rate=shared["learning_rate"],
    max_seq_length=shared["max_seq_length"],

    max_steps=-1,

    gradient_accumulation_steps=shared[
        "gradient_accumulation_steps"
    ],

    peft_enabled=True,
    lora_r=shared["lora_r"],
    lora_alpha=shared["lora_alpha"],
    lora_dropout=shared["lora_dropout"],

    lora_target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],

    quantization=shared["quantization"],

    task_type="chat_completion",
    completion_only_loss=True,
    assistant_only_loss=False,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optimizer=shared["optimizer"],
    weight_decay=shared["weight_decay"],
    warmup_ratio=shared["warmup_ratio"],

    seed=shared["seed"],
    data_seed=shared["data_seed"],

    packing=False,

    save_strategy="steps",
    save_steps=10,
    logging_steps=1,

    report_to="none",
)

# Critical: preserve assistant-only loss masking.
resume_trainer.dataset = clean_dataset
resume_trainer.train_dataset = clean_dataset

assert resume_trainer.config.train.max_steps == -1
assert resume_trainer.config.train.epochs == 2
assert resume_trainer.config.train.completion_only_loss
assert resume_trainer.config.model.peft.rank == 8

print("\nAlignTune configuration restored.")

# --------------------------------------------------
# 7. Resume underlying TRL training
# --------------------------------------------------

# AlignTune calls self.trainer.train() internally.
# Pass resume_from_checkpoint only to that call.
original_train = SFTTrainer.train

def train_from_saved_checkpoint(self, *args, **kwargs):
    if kwargs.get("resume_from_checkpoint") is not None:
        raise RuntimeError(
            "Conflicting checkpoint resume argument."
        )

    kwargs["resume_from_checkpoint"] = str(CHECKPOINT)

    print("\nRestoring optimizer, scheduler,")
    print("LoRA adapter and RNG from:", CHECKPOINT)

    return original_train(self, *args, **kwargs)

print("\nResuming Model A...")

torch.cuda.reset_peak_memory_stats()

# Patch only during this call; automatically restore afterward.
with patch.object(SFTTrainer, "train", train_from_saved_checkpoint):
    resume_result = resume_trainer.train()

# --------------------------------------------------
# 8. Verify completed run
# --------------------------------------------------

inner = resume_trainer.trainer

final_step = inner.state.global_step
final_loss = resume_result.get("final_loss")

if final_loss is None:
    final_loss = resume_result.get(
        "metrics", {}
    ).get("train_loss")

if final_step <= saved_step:
    raise RuntimeError(
        "Training did not progress beyond checkpoint 30."
    )

if final_loss is None or not math.isfinite(float(final_loss)):
    raise RuntimeError("Invalid final training loss.")

# Confirm final adapter is present in output directory.
adapter_config = OUTPUT_DIR / "adapter_config.json"
adapter_weights = OUTPUT_DIR / "adapter_model.safetensors"

if not adapter_config.exists() or not adapter_weights.exists():
    raise RuntimeError("Final adapter files not found.")

# --------------------------------------------------
# 9. Save completion metadata
# --------------------------------------------------

metadata_path = RUN_DIR / "run_metadata.json"
metadata = json.loads(metadata_path.read_text())

metadata.update({
    "status": "TRAINING_COMPLETED",
    "resumed_from": str(CHECKPOINT),
    "resumed_from_step": saved_step,
    "optimizer_steps": final_step,
    "final_loss": float(final_loss),
    "peak_gpu_memory_gb": round(
        torch.cuda.max_memory_allocated() / 1024**3, 3
    ),
})

metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8"
)

(RUN_DIR / "training_result.json").write_text(
    json.dumps(resume_result, indent=2, default=str),
    encoding="utf-8"
)

(RUN_DIR / "training_log_history.json").write_text(
    json.dumps(inner.state.log_history, indent=2, default=str),
    encoding="utf-8"
)

print("\n" + "=" * 60)
print("MODEL A TRAINING COMPLETED")
print("=" * 60)

print("Resumed from step:", saved_step)
print("Final optimizer step:", final_step)
print("Final loss:", final_loss)
print("Adapter directory:", OUTPUT_DIR)
print("Metadata:", metadata_path)
print("Training logs saved to Drive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CURATELAW — RESUME MODEL A FROM STEP 30
Checkpoint: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_a_raw/checkpoints/checkpoint-30
Saved optimizer step: 30
Dataset integrity: VERIFIED
Examples: 150


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Base model revision: af89d219b53ed9b13b8a4645f9c8028973510324
GPU allocated before loading: 0.0 GB

AlignTune configuration restored.

Resuming Model A...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

... (3 outputs left out)


Restoring optimizer, scheduler,
LoRA adapter and RNG from: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_a_raw/checkpoints/checkpoint-30


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6}.


Step,Training Loss
31,1.105605
32,1.155283
33,1.021783
34,1.258419
35,0.158518
36,0.683652
37,0.825630
38,1.012749
39,0.958993
40,0.902145



MODEL A TRAINING COMPLETED
Resumed from step: 30
Final optimizer step: 76
Final loss: 0.9735657864290735
Adapter directory: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_a_raw/checkpoints
Metadata: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_a_raw/run_metadata.json
Training logs saved to Drive.


## Model B

In [ ]:

from pathlib import Path
import json
import hashlib
import math

ROOT = Path("/content/drive/MyDrive/LexiLabs")
ARTIFACTS = ROOT / "finetuning_artifacts"

A_DIR = ARTIFACTS / "model_a_raw"
A_OUT = A_DIR / "checkpoints"
SHARED_PATH = ARTIFACTS / "shared_training_config.json"

shared = json.loads(SHARED_PATH.read_text())
metadata = json.loads(
    (A_DIR / "run_metadata.json").read_text()
)

print("=" * 60)
print("MODEL A — FINAL ARTIFACT VERIFICATION")
print("=" * 60)

assert metadata["status"] == "TRAINING_COMPLETED"
assert metadata["optimizer_steps"] == 76
assert math.isfinite(float(metadata["final_loss"]))

assert metadata["model_revision"] == shared["model_revision"]
assert metadata["dataset_sha256"] == shared["raw_sha256"]

assert (A_OUT / "adapter_config.json").is_file()
assert (
    (A_OUT / "adapter_model.safetensors").is_file()
    or (A_OUT / "adapter_model.bin").is_file()
)

state_files = sorted(
    A_OUT.glob("checkpoint-*/trainer_state.json"),
    key=lambda p: int(p.parent.name.split("-")[-1])
)

assert state_files, "No checkpoint state files found."

history_path = A_DIR / "training_log_history.json"
history = json.loads(history_path.read_text())

logged_steps = {
    int(row["step"])
    for row in history
    if row.get("step") is not None
    and "loss" in row
}

print("Model:", metadata["model_id"])
print("Training status:", metadata["status"])
print("Optimizer steps:", metadata["optimizer_steps"])
print("Final loss:", metadata["final_loss"])
print("Saved checkpoints:", [
    p.parent.name for p in state_files
])
print("Logged loss steps:", len(logged_steps))
print("First logged step:", min(logged_steps) if logged_steps else None)
print("Last logged step:", max(logged_steps) if logged_steps else None)

if not set(range(1, 77)).issubset(logged_steps):
    print(
        "NOTICE: Some step-level loss entries are absent "
        "from the saved log. Checkpoint history may "
        "contain additional entries."
    )

print("\nMODEL A FINAL ADAPTER: VERIFIED")
print("No training performed.")


MODEL A — FINAL ARTIFACT VERIFICATION
Model: CohereLabs/tiny-aya-global
Training status: TRAINING_COMPLETED
Optimizer steps: 76
Final loss: 0.9735657864290735
Saved checkpoints: ['checkpoint-10', 'checkpoint-20', 'checkpoint-30', 'checkpoint-40', 'checkpoint-50', 'checkpoint-60', 'checkpoint-70', 'checkpoint-76']
Logged loss steps: 76
First logged step: 1
Last logged step: 76

MODEL A FINAL ADAPTER: VERIFIED
No training performed.


Inspect Model B's existing files

In [ ]:

from pathlib import Path
import json

RUN_DIR = Path(
    "/content/drive/MyDrive/LexiLabs/"
    "finetuning_artifacts/model_b_curated"
)

OUTPUT_DIR = RUN_DIR / "checkpoints"

print("=" * 60)
print("MODEL B — EXISTING RUN DIAGNOSTIC")
print("=" * 60)

print("Run directory exists:", RUN_DIR.exists())

if not RUN_DIR.exists():
    print("No previous Model B experiment found.")

else:
    metadata_path = RUN_DIR / "run_metadata.json"

    if metadata_path.exists():
        metadata = json.loads(
            metadata_path.read_text(encoding="utf-8")
        )

        print("\nRun status:", metadata.get("status"))
        print("Dataset:", metadata.get("dataset"))
        print("Dataset SHA:", metadata.get("dataset_sha256"))
        print("Optimizer steps:", metadata.get("optimizer_steps"))
    else:
        print("\nNo run metadata found.")

    print("\nExisting files:")

    for path in sorted(RUN_DIR.rglob("*")):
        if path.is_file():
            print(path.relative_to(RUN_DIR))

    checkpoints = sorted(
        [
            p for p in OUTPUT_DIR.glob("checkpoint-*")
            if p.is_dir()
        ],
        key=lambda p: int(p.name.split("-")[-1])
    ) if OUTPUT_DIR.exists() else []

    print("\nSaved checkpoints:", len(checkpoints))

    for checkpoint in checkpoints:
        state_file = checkpoint / "trainer_state.json"

        if state_file.exists():
            state = json.loads(
                state_file.read_text(encoding="utf-8")
            )

            print(
                checkpoint.name,
                "global_step:",
                state.get("global_step")
            )
        else:
            print(
                checkpoint.name,
                "trainer_state.json missing"
            )

    final_adapter = (
        (OUTPUT_DIR / "adapter_config.json").exists()
        and (
            (OUTPUT_DIR / "adapter_model.safetensors").exists()
            or (OUTPUT_DIR / "adapter_model.bin").exists()
        )
    )

    print("\nFinal adapter present:", final_adapter)

print("\nDiagnostic completed.")
print("No training performed.")


MODEL B — EXISTING RUN DIAGNOSTIC
Run directory exists: True

Run status: PREPARED
Dataset: curated_150_messages.jsonl
Dataset SHA: e5881453eae42849eab4e2092587f78778be97b9f203e8347de96be6b1a5a2b8
Optimizer steps: None

Existing files:
run_metadata.json
shared_training_config.json

Saved checkpoints: 0

Final adapter present: False

Diagnostic completed.
No training performed.


 Prepare Model B from the curated dataset

In [ ]:

import gc
import json
import hashlib
import shutil
import importlib.metadata as md
from pathlib import Path

import torch
from datasets import Dataset
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer
from aligntune.core.backend_factory import create_sft_trainer

print("=" * 60)
print("CURATELAW M4 — MODEL B INITIALIZATION")
print("=" * 60)

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/LexiLabs")
ARTIFACT_DIR = DRIVE_ROOT / "finetuning_artifacts"

LOCAL_ROOT = Path("/content/curatelaw_m4")
DATA_DIR = LOCAL_ROOT / "datasets"

RUN_NAME = "model_b_curated"
RUN_DIR = ARTIFACT_DIR / RUN_NAME
OUTPUT_DIR = RUN_DIR / "checkpoints"

DATA_DIR.mkdir(parents=True, exist_ok=True)

SHARED_PATH = ARTIFACT_DIR / "shared_training_config.json"

shared = json.loads(
    SHARED_PATH.read_text(encoding="utf-8")
)

# --------------------------------------------------
# 2. Verify existing Model B preparation
# --------------------------------------------------

metadata_path = RUN_DIR / "run_metadata.json"

if not metadata_path.is_file():
    raise RuntimeError(
        "Model B run_metadata.json is missing."
    )

metadata = json.loads(
    metadata_path.read_text(encoding="utf-8")
)

assert metadata["status"] == "PREPARED", (
    "Model B has a different status. "
    "Do not overwrite a started experiment."
)

assert metadata["dataset"] == "curated_150_messages.jsonl"
assert metadata["dataset_sha256"] == shared["curated_sha256"]
assert metadata["model_revision"] == shared["model_revision"]

if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise RuntimeError(
        "Checkpoint directory is not empty. "
        "Inspect the existing artifacts before training."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Existing prepared run: VERIFIED")

# --------------------------------------------------
# 3. Verify frozen configuration
# --------------------------------------------------

assert shared["model_id"] == "CohereLabs/tiny-aya-global"
assert shared["max_seq_length"] == 3072
assert shared["training_method"] == "qlora_sft"
assert shared["num_train_epochs"] == 2

assert shared["lora_r"] == 8
assert shared["lora_alpha"] == 16
assert shared["lora_dropout"] == 0.05

assert shared["learning_rate"] == 0.0001
assert shared["gradient_accumulation_steps"] == 4

assert md.version("aligntune") == shared["aligntune_version"]
assert md.version("transformers") == shared["transformers_version"]

assert torch.cuda.is_available()
assert "T4" in torch.cuda.get_device_name(0)

print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# 4. Verify dataset integrity
# --------------------------------------------------

def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


source = DRIVE_ROOT / "curated_150_messages.jsonl"
destination = DATA_DIR / source.name

shutil.copy2(source, destination)

actual_hash = sha256(destination)

assert actual_hash == shared["curated_sha256"]

records = [
    json.loads(line)
    for line in destination.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(records) == 150

print("Curated dataset integrity: VERIFIED")
print("Examples:", len(records))
print("SHA-256:", actual_hash)

# --------------------------------------------------
# 5. Load original frozen checkpoint
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_LEXI_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_LEXI_TOKEN missing.")

BASE_PATH = snapshot_download(
    repo_id=shared["model_id"],
    revision=shared["model_revision"],
    token=HF_TOKEN
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_PATH
)

assert tokenizer.chat_template is not None

print("Base model:", shared["model_id"])
print("Frozen revision:", shared["model_revision"])

# --------------------------------------------------
# 6. Prepare clean prompt/completion dataset
# --------------------------------------------------

def get_ids(messages, add_generation_prompt=False):
    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=add_generation_prompt,
        truncation=False,
        return_dict=True
    )

    ids = encoded["input_ids"]

    if ids and isinstance(ids[0], list):
        ids = ids[0]

    return ids


clean_records = []

total_tokens = 0
supervised_tokens = 0
maximum_length = 0

for index, record in enumerate(records, 1):
    messages = record["messages"]

    assert len(messages) == 2
    assert [m["role"] for m in messages] == [
        "user", "assistant"
    ]

    full_ids = get_ids(messages)

    prompt_ids = get_ids(
        messages[:1],
        add_generation_prompt=True
    )

    if full_ids[:len(prompt_ids)] != prompt_ids:
        raise RuntimeError(
            f"Prompt alignment failed at example {index}"
        )

    if len(full_ids) > 3072:
        raise RuntimeError(
            f"Example {index} exceeds sequence limit."
        )

    if len(full_ids) <= len(prompt_ids):
        raise RuntimeError(
            f"Example {index} has empty completion."
        )

    total_tokens += len(full_ids)
    supervised_tokens += len(full_ids) - len(prompt_ids)

    maximum_length = max(
        maximum_length,
        len(full_ids)
    )

    clean_records.append({
        "prompt": [messages[0]],
        "completion": [messages[1]]
    })

assert total_tokens == shared["curated_total_tokens"]

clean_dataset = Dataset.from_list(clean_records)

assert clean_dataset.column_names == [
    "prompt", "completion"
]

print("\nDataset prepared:")
print("Examples:", len(clean_dataset))
print("Total sequence tokens:", total_tokens)
print("Supervised completion tokens:", supervised_tokens)
print("Longest sequence:", maximum_length)

# --------------------------------------------------
# 7. Verify GPU memory
# --------------------------------------------------

gc.collect()
torch.cuda.empty_cache()

allocated_gb = (
    torch.cuda.memory_allocated() / 1024**3
)

print("\nGPU memory allocated:", round(allocated_gb, 2), "GB")

if allocated_gb >= 1:
    raise RuntimeError(
        "GPU memory is not clean. "
        "Restart Colab and rerun this cell."
    )

# --------------------------------------------------
# 8. Initialize AlignTune trainer
# --------------------------------------------------

model_b_trainer = create_sft_trainer(
    model_name=str(BASE_PATH),
    dataset_name=str(destination),
    backend="trl",
    output_dir=str(OUTPUT_DIR),

    num_epochs=shared["num_train_epochs"],
    batch_size=shared["per_device_train_batch_size"],
    learning_rate=shared["learning_rate"],
    max_seq_length=shared["max_seq_length"],

    max_steps=-1,

    gradient_accumulation_steps=shared[
        "gradient_accumulation_steps"
    ],

    peft_enabled=True,
    lora_r=shared["lora_r"],
    lora_alpha=shared["lora_alpha"],
    lora_dropout=shared["lora_dropout"],

    lora_target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ],

    quantization=shared["quantization"],

    task_type="chat_completion",

    completion_only_loss=True,
    assistant_only_loss=False,

    fp16=True,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False
    },

    optimizer=shared["optimizer"],
    weight_decay=shared["weight_decay"],
    warmup_ratio=shared["warmup_ratio"],

    seed=shared["seed"],
    data_seed=shared["data_seed"],

    packing=False,

    save_strategy="steps",
    save_steps=10,
    logging_steps=1,

    report_to="none"
)

# --------------------------------------------------
# 9. Preserve correct assistant-only loss masking
# --------------------------------------------------

# AlignTune otherwise normalizes records into "messages",
# which previously caused prompt tokens to be supervised.
model_b_trainer.dataset = clean_dataset
model_b_trainer.train_dataset = clean_dataset

# --------------------------------------------------
# 10. Verify effective training settings
# --------------------------------------------------

cfg = model_b_trainer.config

assert cfg.train.max_steps == -1
assert cfg.train.epochs == 2
assert cfg.train.per_device_batch_size == 1
assert cfg.train.gradient_accumulation_steps == 4
assert cfg.train.learning_rate == 0.0001

assert cfg.train.completion_only_loss is True
assert cfg.train.packing is False
assert cfg.train.seed == 42

assert cfg.model.max_seq_length == 3072
assert cfg.model.peft.enabled is True
assert cfg.model.peft.rank == 8
assert cfg.model.peft.alpha == 16
assert cfg.model.peft.dropout == 0.05

assert cfg.train.save_strategy == "steps"
assert cfg.train.save_steps == 10

print("\n" + "=" * 60)
print("MODEL B INITIALIZATION: PASSED")
print("=" * 60)

print("Base model:", shared["model_id"])
print("Base revision:", shared["model_revision"])
print("Examples:", len(clean_dataset))
print("Epochs:", cfg.train.epochs)
print("Learning rate:", cfg.train.learning_rate)
print("Gradient accumulation:", cfg.train.gradient_accumulation_steps)
print("Completion-only loss:", cfg.train.completion_only_loss)
print("Checkpoint every:", cfg.train.save_steps, "steps")
print("Output directory:", cfg.logging.output_dir)

print("\nReady for full Model B training.")
print("No optimizer steps executed yet.")


CURATELAW M4 — MODEL B INITIALIZATION
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Existing prepared run: VERIFIED
GPU: Tesla T4
Curated dataset integrity: VERIFIED
Examples: 150
SHA-256: e5881453eae42849eab4e2092587f78778be97b9f203e8347de96be6b1a5a2b8


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Base model: CohereLabs/tiny-aya-global
Frozen revision: af89d219b53ed9b13b8a4645f9c8028973510324

Dataset prepared:
Examples: 150
Total sequence tokens: 92359
Supervised completion tokens: 25693
Longest sequence: 2082

GPU memory allocated: 0.0 GB


AttributeError: 'TrainingConfig' object has no attribute 'save_strategy'

In [ ]:
print("=" * 60)
print("MODEL B — FINAL CONFIGURATION CHECK")
print("=" * 60)

# The trainer was created successfully before the
# previous cell failed on an unsupported attribute.

assert "model_b_trainer" in globals()

cfg = model_b_trainer.config

assert cfg.train.max_steps == -1
assert cfg.train.epochs == 2
assert cfg.train.per_device_batch_size == 1
assert cfg.train.gradient_accumulation_steps == 4
assert cfg.train.learning_rate == 0.0001
assert cfg.train.completion_only_loss is True
assert cfg.train.packing is False
assert cfg.train.seed == 42

assert cfg.model.max_seq_length == 3072
assert cfg.model.peft.enabled is True
assert cfg.model.peft.rank == 8
assert cfg.model.peft.alpha == 16
assert cfg.model.peft.dropout == 0.05

assert len(model_b_trainer.train_dataset) == 150

assert model_b_trainer.train_dataset.column_names == [
    "prompt",
    "completion"
]

# Check available checkpoint configuration fields
# without assuming their locations.
print("\nCheckpoint-related configuration:")

for section_name in ["train", "logging"]:
    section = getattr(cfg, section_name, None)

    if section is None:
        continue

    for name in dir(section):
        if name.startswith("_"):
            continue

        if "save" in name or "checkpoint" in name:
            try:
                print(
                    f"{section_name}.{name} =",
                    getattr(section, name)
                )
            except Exception:
                pass

print("\nModel B hyperparameters: VERIFIED")
print("Dataset: 150 curated examples")
print("Completion-only loss: ENABLED")
print("Ready for training.")

MODEL B — FINAL CONFIGURATION CHECK

Checkpoint-related configuration:
train.gradient_checkpointing = True
train.gradient_checkpointing_kwargs = {'use_reentrant': False}
train.save_interval = 10
logging.save_strategy = steps

Model B hyperparameters: VERIFIED
Dataset: 150 curated examples
Completion-only loss: ENABLED
Ready for training.


In [ ]:
print("Dataset:", len(model_b_trainer.train_dataset))
print("Columns:", model_b_trainer.train_dataset.column_names)
print("Training status:", metadata["status"])
print("Save interval:", model_b_trainer.config.train.save_interval)
print("Save strategy:", model_b_trainer.config.logging.save_strategy)
print("Output:", OUTPUT_DIR)

assert len(model_b_trainer.train_dataset) == 150
assert model_b_trainer.train_dataset.column_names == [
    "prompt", "completion"
]
assert metadata["status"] == "PREPARED"
assert model_b_trainer.config.train.save_interval == 10
assert model_b_trainer.config.logging.save_strategy == "steps"

print("\nALL CHECKS PASSED — START MODEL B TRAINING")

Dataset: 150
Columns: ['prompt', 'completion']
Training status: PREPARED
Save interval: 10
Save strategy: steps
Output: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_b_curated/checkpoints

ALL CHECKS PASSED — START MODEL B TRAINING


Train Model B and save artifacts

In [ ]:

import json
import math
import csv
import torch
from pathlib import Path

print("=" * 60)
print("CURATELAW — MODEL B CURATED FINE-TUNING")
print("=" * 60)

assert len(model_b_trainer.train_dataset) == 150
assert model_b_trainer.config.train.max_steps == -1
assert model_b_trainer.config.train.epochs == 2
assert model_b_trainer.config.train.completion_only_loss

metadata_path = RUN_DIR / "run_metadata.json"
metadata = json.loads(metadata_path.read_text())

assert metadata["status"] == "PREPARED"

metadata["status"] = "TRAINING_STARTED"
metadata_path.write_text(
    json.dumps(metadata, indent=2)
)

torch.cuda.reset_peak_memory_stats()

print("Starting from original Tiny Aya Global checkpoint.")
print("Dataset:", metadata["dataset"])
print("Output:", OUTPUT_DIR)
print("Training Model B...\n")

# Full 150-example, two-epoch training.
model_b_result = model_b_trainer.train()

inner = model_b_trainer.trainer
steps = inner.state.global_step

if steps != 76:
    raise RuntimeError(
        f"Expected 76 optimizer steps, got {steps}"
    )

final_loss = model_b_result.get("final_loss")

if final_loss is None:
    final_loss = model_b_result.get(
        "metrics", {}
    ).get("train_loss")

if final_loss is None or not math.isfinite(
    float(final_loss)
):
    raise RuntimeError("Missing or non-finite final loss.")

# Check every processed example, not just a sample.
processed = inner.train_dataset
assert len(processed) == 150

masked_tokens = 0
supervised_tokens = 0

for index, example in enumerate(processed):
    ids = example["input_ids"]
    labels = example["labels"]

    original = clean_records[index]

    prefix = get_ids(
        original["prompt"],
        add_generation_prompt=True
    )

    if ids[:len(prefix)] != prefix:
        raise RuntimeError(
            f"Input misalignment at example {index}"
        )

    if not all(x == -100 for x in labels[:len(prefix)]):
        raise RuntimeError(
            f"Unmasked prompt at example {index}"
        )

    if not all(x != -100 for x in labels[len(prefix):]):
        raise RuntimeError(
            f"Missing completion labels at example {index}"
        )

    masked_tokens += len(prefix)
    supervised_tokens += len(ids) - len(prefix)

assert supervised_tokens == metadata["total_supervised_tokens"]

# Verify final adapter, separate from intermediate checkpoints.
assert (OUTPUT_DIR / "adapter_config.json").is_file()
assert (
    (OUTPUT_DIR / "adapter_model.safetensors").is_file()
    or (OUTPUT_DIR / "adapter_model.bin").is_file()
)

# Export complete training history.
history = inner.state.log_history

(RUN_DIR / "training_log_history.json").write_text(
    json.dumps(history, indent=2, default=str)
)

loss_rows = [
    {
        "step": entry.get("step"),
        "epoch": entry.get("epoch"),
        "loss": entry["loss"],
        "learning_rate": entry.get("learning_rate")
    }
    for entry in history
    if "loss" in entry
]

with (RUN_DIR / "loss_history.csv").open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "step", "epoch", "loss", "learning_rate"
        ]
    )
    writer.writeheader()
    writer.writerows(loss_rows)

peak_gb = torch.cuda.max_memory_allocated() / 1024**3

metadata.update({
    "status": "TRAINING_COMPLETED",
    "optimizer_steps": steps,
    "final_loss": float(final_loss),
    "peak_gpu_memory_gb": round(peak_gb, 3),
    "masked_prompt_tokens": masked_tokens,
    "supervised_completion_tokens": supervised_tokens,
    "adapter_output_dir": str(OUTPUT_DIR),
    "logged_loss_points": len(loss_rows)
})

metadata_path.write_text(
    json.dumps(metadata, indent=2)
)

(RUN_DIR / "training_result.json").write_text(
    json.dumps(model_b_result, indent=2, default=str)
)

print("\n" + "=" * 60)
print("MODEL B — TRAINING COMPLETED")
print("=" * 60)

print("Optimizer steps:", steps)
print("Final loss:", final_loss)
print("Peak GPU memory:", round(peak_gb, 2), "GB")
print("Masked prompt tokens:", masked_tokens)
print("Supervised completion tokens:", supervised_tokens)
print("Logged loss points:", len(loss_rows))
print("Adapter:", OUTPUT_DIR)
print("Metadata:", metadata_path)
print("Training history saved to Google Drive.")


CURATELAW — MODEL B CURATED FINE-TUNING
Starting from original Tiny Aya Global checkpoint.
Dataset: curated_150_messages.jsonl
Output: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_b_curated/checkpoints
Training Model B...



[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 3,833,856 || all params: 3,353,061,376 || trainable%: 0.1143


Tokenizing train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/150 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 6}.


Step,Training Loss
1,1.927779
2,1.641465
3,0.962697
4,1.677434
5,1.029238
6,1.606631
7,1.808179
8,2.074099
9,1.049604
10,0.919225



MODEL B — TRAINING COMPLETED
Optimizer steps: 76
Final loss: 1.1422367844926684
Peak GPU memory: 11.1 GB
Masked prompt tokens: 66666
Supervised completion tokens: 25693
Logged loss points: 76
Adapter: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_b_curated/checkpoints
Metadata: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_b_curated/run_metadata.json
Training history saved to Google Drive.


# AuditKIT evaluation

In [ ]:

from google.colab import drive
from pathlib import Path
import json
import hashlib

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/LexiLabs")
ART = ROOT / "finetuning_artifacts"

SHARED = ART / "shared_training_config.json"
shared = json.loads(SHARED.read_text(encoding="utf-8"))

runs = {
    "raw": ART / "model_a_raw",
    "curated": ART / "model_b_curated",
}

print("=" * 65)
print("CURATELAW M5 — MODEL AND DATASET READINESS")
print("=" * 65)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for name, run_dir in runs.items():
    metadata_file = run_dir / "run_metadata.json"
    adapter_dir = run_dir / "checkpoints"

    assert metadata_file.is_file(), metadata_file

    metadata = json.loads(
        metadata_file.read_text(encoding="utf-8")
    )

    adapter_config = adapter_dir / "adapter_config.json"
    adapter_weights = adapter_dir / "adapter_model.safetensors"

    if not adapter_weights.is_file():
        adapter_weights = adapter_dir / "adapter_model.bin"

    assert metadata["status"] == "TRAINING_COMPLETED"
    assert metadata["optimizer_steps"] == 76
    assert metadata["model_id"] == shared["model_id"]
    assert metadata["model_revision"] == shared["model_revision"]

    expected_hash = shared[
        "raw_sha256" if name == "raw" else "curated_sha256"
    ]

    assert metadata["dataset_sha256"] == expected_hash
    assert adapter_config.is_file()
    assert adapter_weights.is_file()

    config = json.loads(
        adapter_config.read_text(encoding="utf-8")
    )

    assert config["r"] == shared["lora_r"]
    assert config["lora_alpha"] == shared["lora_alpha"]

    print(f"\n{name.upper()} MODEL")
    print("  Status: COMPLETE")
    print("  Optimizer steps:", metadata["optimizer_steps"])
    print("  Training loss:", metadata["final_loss"])
    print("  LoRA rank:", config["r"])
    print("  Adapter size:",
          round(adapter_weights.stat().st_size / 1024**2, 2),
          "MB")
    print("  Adapter:", adapter_weights)

print("\n" + "=" * 65)
print("FROZEN HELD-OUT DATASET DISCOVERY")
print("=" * 65)

# Find candidate held-out files without assuming their
# exact names or moving/modifying any files.

keywords = (
    "heldout",
    "held_out",
    "eval",
    "benchmark",
    "test_40",
    "scenarios",
    "audit",
)

candidates = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    # Exclude model weights, checkpoints, logs.
    if "finetuning_artifacts" in path.parts:
        continue

    if path.suffix.lower() not in {
        ".jsonl", ".json", ".csv"
    }:
        continue

    if any(word in path.name.lower() for word in keywords):
        candidates.append(path)

if not candidates:
    print("No candidates found under:", ROOT)
else:
    for path in sorted(candidates)[:40]:
        print("\nFile:", path.relative_to(ROOT))

        if path.suffix == ".jsonl":
            try:
                with path.open(encoding="utf-8") as f:
                    rows = [
                        json.loads(line)
                        for line in f
                        if line.strip()
                    ]

                print("  JSONL records:", len(rows))
                if rows and isinstance(rows[0], dict):
                    print("  Fields:", list(rows[0].keys()))
                print("  SHA-256:", sha256(path))

            except Exception as exc:
                print("  Could not parse:", str(exc)[:180])

        elif path.suffix == ".json":
            try:
                data = json.loads(
                    path.read_text(encoding="utf-8")
                )

                print("  JSON type:", type(data).__name__)

                if isinstance(data, list):
                    print("  Records:", len(data))
                elif isinstance(data, dict):
                    print("  Top-level fields:", list(data.keys())[:15])

                print("  SHA-256:", sha256(path))

            except Exception as exc:
                print("  Could not parse:", str(exc)[:180])

        else:
            print("  SHA-256:", sha256(path))

print("\n" + "=" * 65)
print("MODEL VERIFICATION AND DATASET DISCOVERY COMPLETE")
print("=" * 65)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CURATELAW M5 — MODEL AND DATASET READINESS

RAW MODEL
  Status: COMPLETE
  Optimizer steps: 76
  Training loss: 0.9735657864290735
  LoRA rank: 8
  Adapter size: 14.66 MB
  Adapter: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_a_raw/checkpoints/adapter_model.safetensors

CURATED MODEL
  Status: COMPLETE
  Optimizer steps: 76
  Training loss: 1.1422367844926684
  LoRA rank: 8
  Adapter size: 7.35 MB
  Adapter: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_b_curated/checkpoints/adapter_model.safetensors

FROZEN HELD-OUT DATASET DISCOVERY

File: evaluation_artifacts/benchmark_provenance.json
  JSON type: dict
  Top-level fields: ['project', 'source_repository', 'source_branch', 'repository_commit', 'source_path', 'benchmark_sha256', 'count', 'scenario_id_overlap_raw', 'scenario_id_overlap_curated', 'exact_instruction_overlap_raw', 'exa

Adapter integrity and benchmark discovery

In [ ]:

from pathlib import Path
from collections import Counter
import json
import hashlib

from safetensors import safe_open

print("=" * 65)
print("CURATELAW M5.2 — ADAPTER INTEGRITY + EVAL DISCOVERY")
print("=" * 65)

ROOT = Path("/content/drive/MyDrive/LexiLabs")
MYDRIVE = Path("/content/drive/MyDrive")
ART = ROOT / "finetuning_artifacts"

adapters = {
    "RAW": ART / "model_a_raw/checkpoints/adapter_model.safetensors",
    "CURATED": ART / "model_b_curated/checkpoints/adapter_model.safetensors",
}

def inspect_adapter(path):
    assert path.is_file(), path

    with safe_open(str(path), framework="pt", device="cpu") as f:
        keys = sorted(f.keys())
        shapes = {}
        dtypes = Counter()
        parameter_count = 0

        for key in keys:
            # Read tensor metadata only, not GPU tensors.
            info = f.get_slice(key)
            shape = tuple(info.get_shape())
            dtype = str(info.get_shape())  # replaced below

            # Safetensors slices expose get_shape(), get_dtype().
            dtype = str(info.get_dtype())

            shapes[key] = shape
            dtypes[dtype] += 1

            count = 1
            for dimension in shape:
                count *= dimension
            parameter_count += count

    return {
        "keys": keys,
        "shapes": shapes,
        "dtypes": dict(dtypes),
        "parameter_count": parameter_count,
        "bytes": path.stat().st_size,
    }

reports = {}

for name, path in adapters.items():
    report = inspect_adapter(path)
    reports[name] = report

    print(f"\n{name} ADAPTER")
    print("File:", path)
    print("Size (MiB):", round(report["bytes"] / 1024**2, 3))
    print("Tensor count:", len(report["keys"]))
    print("Tensor dtypes:", report["dtypes"])
    print("Saved parameters:", report["parameter_count"])

raw = reports["RAW"]
curated = reports["CURATED"]

same_keys = raw["keys"] == curated["keys"]
same_shapes = raw["shapes"] == curated["shapes"]

print("\nADAPTER COMPARISON")
print("Same tensor names:", same_keys)
print("Same tensor shapes:", same_shapes)
print("Same saved parameter count:",
      raw["parameter_count"] == curated["parameter_count"])

if not (same_keys and same_shapes):
    print(
        "WARNING: Adapter structures differ. "
        "Investigate before evaluating."
    )
else:
    print("Adapter structures: COMPATIBLE")

print("\n" + "=" * 65)
print("SEARCHING MYDRIVE FOR EVALUATION DATA")
print("=" * 65)

# Search beyond LexiLabs in case the held-out benchmark
# was saved in another Drive project folder.
keywords = (
    "heldout", "held_out", "holdout",
    "eval", "benchmark", "test",
    "scenario", "audit", "curatelaw"
)

extensions = {
    ".json", ".jsonl", ".csv", ".parquet", ".xlsx"
}

matches = []

for path in MYDRIVE.rglob("*"):
    if not path.is_file():
        continue

    # Ignore generated training checkpoints and weights.
    if "checkpoints" in path.parts:
        continue

    if path.suffix.lower() not in extensions:
        continue

    relative = str(path.relative_to(MYDRIVE)).lower()

    if any(keyword in relative for keyword in keywords):
        matches.append(path)

print("Candidate files:", len(matches))

for path in sorted(matches)[:80]:
    print("\n", path.relative_to(MYDRIVE))

    try:
        if path.suffix.lower() == ".jsonl":
            count = 0
            first = None

            with path.open(encoding="utf-8") as f:
                for line in f:
                    if not line.strip():
                        continue
                    obj = json.loads(line)
                    count += 1
                    if first is None:
                        first = obj

            print("  Rows:", count)

            if isinstance(first, dict):
                print("  Fields:", list(first.keys())[:20])

        elif path.suffix.lower() == ".json":
            data = json.loads(
                path.read_text(encoding="utf-8")
            )

            if isinstance(data, list):
                print("  List entries:", len(data))
                if data and isinstance(data[0], dict):
                    print("  Fields:", list(data[0].keys())[:20])

            elif isinstance(data, dict):
                print("  Top-level fields:", list(data.keys())[:20])

    except Exception as error:
        print("  Preview unavailable:", str(error)[:120])

if len(matches) > 80:
    print("\nOnly the first 80 candidates were displayed.")

print("\nM5.2 DIAGNOSTIC COMPLETE")
print("No training or model inference performed.")


CURATELAW M5.2 — ADAPTER INTEGRITY + EVAL DISCOVERY

RAW ADAPTER
File: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_a_raw/checkpoints/adapter_model.safetensors
Size (MiB): 14.661
Tensor count: 288
Tensor dtypes: {'F32': 288}
Saved parameters: 3833856

CURATED ADAPTER
File: /content/drive/MyDrive/LexiLabs/finetuning_artifacts/model_b_curated/checkpoints/adapter_model.safetensors
Size (MiB): 7.349
Tensor count: 288
Tensor dtypes: {'BF16': 288}
Saved parameters: 3833856

ADAPTER COMPARISON
Same tensor names: True
Same tensor shapes: True
Same saved parameter count: True
Adapter structures: COMPATIBLE

SEARCHING MYDRIVE FOR EVALUATION DATA
Candidate files: 77

 LexiLabs/evaluation_artifacts/benchmark_provenance.json
  Top-level fields: ['project', 'source_repository', 'source_branch', 'repository_commit', 'source_path', 'benchmark_sha256', 'count', 'scenario_id_overlap_raw', 'scenario_id_overlap_curated', 'exact_instruction_overlap_raw', 'exact_instruction_overlap_curated', '

Recover the original evaluation dataset

In [ ]:

from pathlib import Path
import subprocess
import json

print("=" * 65)
print("CURATELAW M5.3 — RECOVER ORIGINAL HELD-OUT DATA")
print("=" * 65)

REPO_URL = "https://github.com/Jiban243/CurateLaw.git"
BRANCH = "jiban/m1-m3-progress"
REPO_DIR = Path("/content/CurateLaw_M1_M3")

# Clone only if the repository is not already present.
if not (REPO_DIR / ".git").exists():
    result = subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR)
        ],
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print("Git clone failed:")
        print(result.stderr[-1500:])
        print(
            "\nIf the repository is private, obtain "
            "the M1-M3 evaluation dataset from Jiban "
            "rather than creating a replacement."
        )
    else:
        print("Repository cloned successfully.")
else:
    print("Existing local repository found.")

if (REPO_DIR / ".git").exists():
    commit = subprocess.run(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
        capture_output=True,
        text=True
    )

    print("Repository commit:", commit.stdout.strip())

    print("\nCandidate evaluation files:")

    candidates = []
    keywords = (
        "heldout", "held_out", "holdout",
        "eval", "benchmark", "scenario",
        "test", "audit"
    )

    for path in REPO_DIR.rglob("*"):
        if not path.is_file():
            continue

        if ".git" in path.parts:
            continue

        if path.suffix.lower() not in {
            ".jsonl", ".json", ".csv", ".parquet"
        }:
            continue

        if any(
            keyword in str(path.relative_to(REPO_DIR)).lower()
            for keyword in keywords
        ):
            candidates.append(path)

    print("Candidates:", len(candidates))

    for path in sorted(candidates):
        print("\nFile:", path.relative_to(REPO_DIR))

        if path.suffix == ".jsonl":
            try:
                with path.open(encoding="utf-8") as f:
                    rows = [
                        json.loads(line)
                        for line in f
                        if line.strip()
                    ]

                print("Records:", len(rows))

                if rows and isinstance(rows[0], dict):
                    print("Fields:", list(rows[0].keys()))

            except Exception as error:
                print("Parse error:", str(error)[:150])

        elif path.suffix == ".json":
            try:
                data = json.loads(
                    path.read_text(encoding="utf-8")
                )

                print("Type:", type(data).__name__)

                if isinstance(data, list):
                    print("Records:", len(data))
                    if data and isinstance(data[0], dict):
                        print("Fields:", list(data[0].keys()))
                elif isinstance(data, dict):
                    print("Top-level fields:", list(data.keys())[:20])

            except Exception as error:
                print("Parse error:", str(error)[:150])

    print("\nAll repository data files:")

    for path in sorted(REPO_DIR.rglob("*")):
        if (
            path.is_file()
            and ".git" not in path.parts
            and path.suffix.lower() in {
                ".jsonl", ".json", ".csv", ".parquet"
            }
        ):
            print(path.relative_to(REPO_DIR))

print("\nDISCOVERY COMPLETE")
print("No training or inference performed.")


CURATELAW M5.3 — RECOVER ORIGINAL HELD-OUT DATA
Repository cloned successfully.
Repository commit: 4b503b96f87ec9c017c2e66c52c7e72f2d965ad8

Candidate evaluation files:
Candidates: 10

File: data/m2/evaluation/heldout_eval_040.jsonl
Records: 40
Fields: ['scenario_id', 'task_type', 'instruction', 'input', 'expected_facts', 'origin', 'split', 'training_excluded', 'human_verified']

File: data/m2/evaluation/heldout_eval_manifest.json
Type: dict
Top-level fields: ['dataset', 'status', 'count', 'task_distribution', 'sha256', 'train_overlap_check', 'training_excluded', 'legal_verification_completed', 'gold_reference_answers_available']

File: data/m2/frozen/heldout_eval_040.jsonl
Records: 40
Fields: ['scenario_id', 'task_type', 'instruction', 'input', 'expected_facts', 'origin', 'split', 'training_excluded', 'human_verified']

File: data/m2/quality_reports/M2_BATCH_QUALITY_AUDIT.csv

File: data/m2/quality_reports/M2_SMOKE_AUDIT_SUMMARY.json
Type: dict
Top-level fields: ['total_generated', 's

Freeze and validate the 40 evaluation scenarios

In [ ]:

import json
import hashlib
import re
import shutil
from pathlib import Path
from collections import Counter

print("=" * 68)
print("CURATELAW M5.4 — FROZEN EVALUATION VALIDATION")
print("=" * 68)

REPO = Path("/content/CurateLaw_M1_M3")
DRIVE = Path("/content/drive/MyDrive/LexiLabs")

EVAL_SOURCE = REPO / "data/m2/frozen/heldout_eval_040.jsonl"
EVAL_COPY = REPO / "data/m2/evaluation/heldout_eval_040.jsonl"
EVAL_MANIFEST = (
    REPO / "data/m2/evaluation/heldout_eval_manifest.json"
)

RAW_TRAIN = (
    REPO / "data/m3/finetune_ready/raw_150_messages.jsonl"
)
CURATED_TRAIN = (
    REPO / "data/m3/finetune_ready/curated_150_messages.jsonl"
)

EVAL_DIR = DRIVE / "evaluation_artifacts"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_jsonl(path):
    assert path.is_file(), f"Missing file: {path}"
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

def normalize(value):
    return re.sub(
        r"\s+", " ", str(value).strip()
    ).casefold()

# --------------------------------------------------
# 1. Confirm frozen and evaluation copies match
# --------------------------------------------------

assert EVAL_SOURCE.is_file()
assert EVAL_COPY.is_file()
assert EVAL_MANIFEST.is_file()

source_sha = sha256(EVAL_SOURCE)
copy_sha = sha256(EVAL_COPY)

assert source_sha == copy_sha, (
    "Frozen and evaluation copies differ."
)

manifest = json.loads(
    EVAL_MANIFEST.read_text(encoding="utf-8")
)

print("\nBenchmark SHA-256:", source_sha)
print("Frozen/evaluation copies match: YES")
print("\nManifest contents:")
print(json.dumps(manifest, indent=2, default=str)[:5000])

# Check any published SHA-256 in manifest.
manifest_sha = manifest.get("sha256")

if isinstance(manifest_sha, str):
    assert manifest_sha.lower() == source_sha.lower(), (
        "Manifest hash differs from frozen file."
    )
    print("\nManifest SHA-256: VERIFIED")
else:
    print("\nManifest SHA-256: not a plain string; inspect above.")

# --------------------------------------------------
# 2. Verify schema and number of scenarios
# --------------------------------------------------

eval_rows = read_jsonl(EVAL_SOURCE)

assert len(eval_rows) == 40

required_fields = {
    "scenario_id",
    "task_type",
    "instruction",
    "input",
    "expected_facts",
    "split",
    "training_excluded",
    "human_verified",
}

for index, row in enumerate(eval_rows, 1):
    missing = required_fields - set(row)
    assert not missing, (
        f"Scenario {index} missing fields: {missing}"
    )

    assert str(row["scenario_id"]).strip()
    assert str(row["instruction"]).strip()

scenario_ids = [
    str(row["scenario_id"])
    for row in eval_rows
]

assert len(set(scenario_ids)) == 40, (
    "Duplicate scenario IDs."
)

print("\nScenario count:", len(eval_rows))
print("Unique scenario IDs:", len(set(scenario_ids)))

print("\nTask distribution:")
for task, count in sorted(
    Counter(row["task_type"] for row in eval_rows).items()
):
    print(f"  {task}: {count}")

for field in [
    "split",
    "training_excluded",
    "human_verified",
]:
    print(
        f"\n{field} values:",
        dict(Counter(
            str(row.get(field)) for row in eval_rows
        ))
    )

# These are declarations in the benchmark; they do
# not replace independent overlap or legal review.
assert all(
    row["training_excluded"] is True
    for row in eval_rows
), "Some held-out scenarios are not marked excluded."

# --------------------------------------------------
# 3. Independently check training overlap
# --------------------------------------------------

raw_rows = read_jsonl(RAW_TRAIN)
curated_rows = read_jsonl(CURATED_TRAIN)

assert len(raw_rows) == 150
assert len(curated_rows) == 150

heldout_ids = set(scenario_ids)

def collect_training_ids(rows):
    ids = set()
    for row in rows:
        for container in [
            row,
            row.get("metadata", {})
        ]:
            if isinstance(container, dict):
                value = container.get("scenario_id")
                if value is not None:
                    ids.add(str(value))
    return ids

raw_ids = collect_training_ids(raw_rows)
curated_ids = collect_training_ids(curated_rows)

raw_id_overlap = heldout_ids & raw_ids
curated_id_overlap = heldout_ids & curated_ids

print("\nScenario-ID overlap:")
print("  Raw:", len(raw_id_overlap))
print("  Curated:", len(curated_id_overlap))
print("  IDs available in raw:", len(raw_ids))
print("  IDs available in curated:", len(curated_ids))

assert not raw_id_overlap
assert not curated_id_overlap

# Exact normalized string checks supplement scenario
# IDs. They do not prove absence of paraphrase leakage.
heldout_instructions = {
    normalize(row["instruction"])
    for row in eval_rows
}

def training_contents(rows):
    texts = set()

    for row in rows:
        for message in row.get("messages", []):
            if isinstance(message, dict):
                text = normalize(message.get("content", ""))
                if text:
                    texts.add(text)
    return texts

raw_texts = training_contents(raw_rows)
curated_texts = training_contents(curated_rows)

raw_exact = heldout_instructions & raw_texts
curated_exact = heldout_instructions & curated_texts

print("\nExact normalized instruction overlap:")
print("  Raw:", len(raw_exact))
print("  Curated:", len(curated_exact))

assert not raw_exact
assert not curated_exact

# --------------------------------------------------
# 4. Inspect evaluation requirements
# --------------------------------------------------

print("\nExpected-facts structures:")

fact_types = Counter(
    type(row["expected_facts"]).__name__
    for row in eval_rows
)
print(dict(fact_types))

print("\nRepresentative scenarios:")

seen_tasks = set()

for row in eval_rows:
    task = row["task_type"]

    if task in seen_tasks:
        continue

    seen_tasks.add(task)

    print("\n" + "-" * 60)
    print("Scenario ID:", row["scenario_id"])
    print("Task:", task)
    print("Instruction preview:")
    print(str(row["instruction"])[:350])

    print("Input preview:")
    print(str(row["input"])[:250])

    print("Expected facts preview:")
    print(
        json.dumps(
            row["expected_facts"],
            ensure_ascii=False,
            default=str
        )[:750]
    )

# --------------------------------------------------
# 5. Inspect training prompt format
# --------------------------------------------------

print("\n" + "=" * 68)
print("TRAINING PROMPT FORMAT")
print("=" * 68)

print("\nRaw cohort example — USER message:")
print(raw_rows[0]["messages"][0]["content"][:1200])

print("\nCurated cohort example — USER message:")
print(curated_rows[0]["messages"][0]["content"][:1200])

# --------------------------------------------------
# 6. Preserve frozen benchmark in Drive
# --------------------------------------------------

TARGET = EVAL_DIR / "heldout_eval_040.jsonl"

if TARGET.exists():
    assert sha256(TARGET) == source_sha, (
        "Existing Drive evaluation file differs. "
        "Refusing to overwrite it."
    )
else:
    shutil.copy2(EVAL_SOURCE, TARGET)

provenance = {
    "project": "CurateLaw-India",
    "source_repository": "Jiban243/CurateLaw",
    "source_branch": "jiban/m1-m3-progress",
    "repository_commit": (
        "4b503b96f87ec9c017c2e66c52c7e72f2d965ad8"
    ),
    "source_path": str(
        EVAL_SOURCE.relative_to(REPO)
    ),
    "benchmark_sha256": source_sha,
    "count": len(eval_rows),
    "scenario_id_overlap_raw": len(raw_id_overlap),
    "scenario_id_overlap_curated": len(curated_id_overlap),
    "exact_instruction_overlap_raw": len(raw_exact),
    "exact_instruction_overlap_curated": len(curated_exact),
    "verification_scope": (
        "Schema, hashes, scenario IDs, and exact "
        "normalized instruction equality. "
        "Does not establish semantic non-overlap "
        "or legal correctness."
    )
}

PROVENANCE_PATH = EVAL_DIR / "benchmark_provenance.json"

if PROVENANCE_PATH.exists():
    previous = json.loads(
        PROVENANCE_PATH.read_text(encoding="utf-8")
    )
    assert previous["benchmark_sha256"] == source_sha
else:
    PROVENANCE_PATH.write_text(
        json.dumps(provenance, indent=2),
        encoding="utf-8"
    )

print("\n" + "=" * 68)
print("FROZEN BENCHMARK VALIDATION COMPLETE")
print("=" * 68)

print("Scenarios:", len(eval_rows))
print("SHA-256:", source_sha)
print("Frozen benchmark:", TARGET)
print("Provenance:", PROVENANCE_PATH)
print("No training or inference performed.")


CURATELAW M5.4 — FROZEN EVALUATION VALIDATION

Benchmark SHA-256: 944ba82ed4937a41f32c064aaaa89f1b34aeb2d44a89e0860aac51aa3d46bff9
Frozen/evaluation copies match: YES

Manifest contents:
{
  "dataset": "heldout_eval_040",
  "status": "provisional_pending_human_review",
  "count": 40,
  "task_distribution": {
    "full_agreement": 8,
    "clause_drafting": 8,
    "contract_customization": 8,
    "contradiction_correction": 8,
    "missing_clause_completion": 8
  },
  "sha256": "944ba82ed4937a41f32c064aaaa89f1b34aeb2d44a89e0860aac51aa3d46bff9",
  "train_overlap_check": "exact_instruction_only",
  "training_excluded": true,
  "legal_verification_completed": false,
  "gold_reference_answers_available": false
}

Manifest SHA-256: VERIFIED

Scenario count: 40
Unique scenario IDs: 40

Task distribution:
  clause_drafting: 8
  contract_customization: 8
  contradiction_correction: 8
  full_agreement: 8
  missing_clause_completion: 8

split values: {'heldout_evaluation': 40}

training_excluded v

Release Model B from GPU memory

In [ ]:

import gc
import torch

print("=" * 60)
print("CURATELAW — GPU MEMORY CLEANUP")
print("=" * 60)

if torch.cuda.is_available():
    print(
        "GPU allocated before:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

# Delete models and trainers from previous stages.
# These variables are no longer needed for inference.
variables_to_delete = [
    "model_b_trainer",
    "model_a_trainer",
    "resume_trainer",
    "corrected_trainer",
    "long_trainer",
    "trainer",
    "trl_trainer",
    "inner",
    "trained_model",
    "model",
    "base_model",
    "inference_model",
]

for name in variables_to_delete:
    if name in globals():
        del globals()[name]
        print("Deleted:", name)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3

    print("\nGPU allocated after:", round(allocated, 2), "GB")
    print("GPU reserved after:", round(reserved, 2), "GB")

    if allocated < 1:
        print("\nGPU CLEANUP: PASSED")
    else:
        print(
            "\nGPU still has active allocations. "
            "Restart the Colab session."
        )


CURATELAW — GPU MEMORY CLEANUP
GPU allocated before: 0.0 GB

GPU allocated after: 0.0 GB
GPU reserved after: 0.0 GB

GPU CLEANUP: PASSED


Pilot inference for all three models

In [ ]:
%pip install -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 30.8 MB/s  0:00:01


In [ ]:
import torch
import bitsandbytes as bnb
import transformers
import importlib.metadata as md

from transformers.utils import is_bitsandbytes_available

print("=" * 60)
print("CURATELAW — INFERENCE DEPENDENCY CHECK")
print("=" * 60)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", md.version("bitsandbytes"))
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print(
    "Transformers detects bitsandbytes:",
    is_bitsandbytes_available()
)

assert torch.cuda.is_available()
assert is_bitsandbytes_available(), (
    "Transformers still cannot detect bitsandbytes."
)

print("\nDEPENDENCY CHECK: PASSED")

CURATELAW — INFERENCE DEPENDENCY CHECK
PyTorch: 2.14.1+cu130
Transformers: 5.22.1
bitsandbytes: 0.50.2
CUDA available: True
GPU: Tesla T4
Transformers detects bitsandbytes: True

DEPENDENCY CHECK: PASSED


In [ ]:

import gc
import json
import hashlib
import math
from pathlib import Path

import torch
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel

drive.mount("/content/drive")

print("=" * 66)
print("CURATELAW M5.5 — THREE-MODEL INFERENCE PILOT")
print("=" * 66)

ROOT = Path("/content/drive/MyDrive/LexiLabs")
ART = ROOT / "finetuning_artifacts"
EVAL_DIR = ROOT / "evaluation_artifacts"

shared = json.loads(
    (ART / "shared_training_config.json").read_text()
)

EVAL_PATH = EVAL_DIR / "heldout_eval_040.jsonl"
PROVENANCE_PATH = EVAL_DIR / "benchmark_provenance.json"
PILOT_PATH = EVAL_DIR / "inference_pilot_v1.jsonl"
CONFIG_PATH = EVAL_DIR / "inference_config_v1.json"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

benchmark_hash = sha256(EVAL_PATH)

provenance = json.loads(
    PROVENANCE_PATH.read_text(encoding="utf-8")
)

assert benchmark_hash == provenance["benchmark_sha256"]
assert benchmark_hash == (
    "944ba82ed4937a41f32c064aaaa89f1b34"
    "aeb2d44a89e0860aac51aa3d46bff9"
)

with EVAL_PATH.open(encoding="utf-8") as f:
    scenarios = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(scenarios) == 40

by_id = {
    str(row["scenario_id"]): row
    for row in scenarios
}

assert len(by_id) == 40

PILOT_IDS = ["EVAL001", "EVAL017"]

for scenario_id in PILOT_IDS:
    assert scenario_id in by_id

ADAPTERS = {
    "base": None,
    "raw": (
        ART / "model_a_raw/checkpoints"
    ),
    "curated": (
        ART / "model_b_curated/checkpoints"
    ),
}

for model_name, adapter_dir in ADAPTERS.items():
    if adapter_dir is None:
        continue

    assert (adapter_dir / "adapter_config.json").is_file()
    assert (
        (adapter_dir / "adapter_model.safetensors").is_file()
        or (adapter_dir / "adapter_model.bin").is_file()
    )

assert torch.cuda.is_available()

print("GPU:", torch.cuda.get_device_name(0))

# --------------------------------------------------
# 1. Freeze the evaluation prompt and decoding policy
# --------------------------------------------------

def build_prompt(scenario):
    instruction = str(
        scenario["instruction"]
    ).strip()

    supplied_input = scenario.get("input", "")

    if supplied_input is None:
        supplied_input = ""

    if not isinstance(supplied_input, str):
        supplied_input = json.dumps(
            supplied_input,
            ensure_ascii=False,
            sort_keys=True
        )

    supplied_input = supplied_input.strip()

    # Exactly the same prompt construction will be
    # used for base, raw, and curated models.
    if supplied_input:
        return (
            instruction
            + "\n\nProvided text:\n"
            + supplied_input
        )

    return instruction

def new_token_limit(scenario):
    # Same limit for each model on the same task.
    if scenario["task_type"] == "full_agreement":
        return 1024
    return 384

inference_config = {
    "experiment": "CurateLaw_M5",
    "version": "pilot_v1",
    "benchmark_sha256": benchmark_hash,
    "base_model": shared["model_id"],
    "base_revision": shared["model_revision"],
    "model_names": list(ADAPTERS),
    "pilot_scenario_ids": PILOT_IDS,
    "prompt_policy": (
        "instruction, followed by two newlines and "
        "'Provided text:' plus input when nonempty"
    ),
    "chat_template": "base_model_tokenizer",
    "generation": {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens_full_agreement": 1024,
        "max_new_tokens_other": 384,
        "use_cache": True,
    },
    "evaluation_status": "exploratory_not_legally_verified",
}

if CONFIG_PATH.exists():
    existing_config = json.loads(
        CONFIG_PATH.read_text(encoding="utf-8")
    )
    assert existing_config == inference_config, (
        "Existing inference config differs. "
        "Do not silently change evaluation settings."
    )
else:
    CONFIG_PATH.write_text(
        json.dumps(inference_config, indent=2),
        encoding="utf-8"
    )

# --------------------------------------------------
# 2. Resolve frozen base checkpoint and tokenizer
# --------------------------------------------------

token = userdata.get("HF_LEXI_TOKEN")
assert token, "HF_LEXI_TOKEN missing"

BASE_PATH = snapshot_download(
    repo_id=shared["model_id"],
    revision=shared["model_revision"],
    token=token
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_PATH
)

assert tokenizer.chat_template is not None

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --------------------------------------------------
# 3. Resume-safe pilot output
# --------------------------------------------------

completed = set()

if PILOT_PATH.exists():
    with PILOT_PATH.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)

            assert row["benchmark_sha256"] == benchmark_hash

            completed.add((
                row["model"],
                row["scenario_id"]
            ))

print("Previously completed pilot responses:", len(completed))

# --------------------------------------------------
# 4. Generate from each model independently
# --------------------------------------------------


import torch

print("=" * 60)
print("CURATELAW M5.5 — TOKENIZER OUTPUT FIX")
print("=" * 60)

def generate_one(model, scenario):
    prompt_text = build_prompt(scenario)

    messages = [
        {
            "role": "user",
            "content": prompt_text
        }
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Handle BatchEncoding / dict returned by the
    # installed Transformers tokenizer.
    if isinstance(encoded, torch.Tensor):
        input_ids = encoded
        attention_mask = torch.ones_like(input_ids)

    elif isinstance(encoded, dict) or hasattr(
        encoded, "input_ids"
    ):
        input_ids = encoded["input_ids"]

        attention_mask = encoded.get(
            "attention_mask"
        )

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

    else:
        raise TypeError(
            f"Unsupported tokenizer output: {type(encoded)}"
        )

    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)

    if attention_mask.ndim == 1:
        attention_mask = attention_mask.unsqueeze(0)

    input_ids = input_ids.to("cuda")
    attention_mask = attention_mask.to("cuda")

    limit = new_token_limit(scenario)

    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,
            num_beams=1,
            max_new_tokens=limit,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[
        0,
        input_ids.shape[-1]:
    ]

    response = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return {
        "prompt": prompt_text,
        "response": response,
        "prompt_tokens": int(input_ids.shape[-1]),
        "generated_tokens": int(generated_ids.numel()),
        "max_new_tokens": limit,
        "hit_token_limit": (
            int(generated_ids.numel()) >= limit
        ),
    }

print("Tokenizer output handling: FIXED")
print("Generation function updated.")
print("No model reload or training performed.")


for model_name, adapter_dir in ADAPTERS.items():

    pending_ids = [
        sid
        for sid in PILOT_IDS
        if (model_name, sid) not in completed
    ]

    if not pending_ids:
        print(f"\n{model_name}: already completed")
        continue

    gc.collect()
    torch.cuda.empty_cache()

    allocated_gb = (
        torch.cuda.memory_allocated() / 1024**3
    )

    if allocated_gb >= 1:
        raise RuntimeError(
            "GPU still holds a previous model. "
            "Restart the session before inference."
        )

    print(f"\nLoading model: {model_name}")

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_PATH,
        quantization_config=quantization,
        torch_dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True
    )

    if adapter_dir is None:
        inference_model = base_model
    else:
        inference_model = PeftModel.from_pretrained(
            base_model,
            str(adapter_dir),
            is_trainable=False,
            autocast_adapter_dtype=True
        )

    inference_model.eval()

    for scenario_id in pending_ids:
        scenario = by_id[scenario_id]

        print(
            f"\nGenerating {model_name} / {scenario_id} "
            f"({scenario['task_type']})"
        )

        generated = generate_one(
            inference_model,
            scenario
        )

        result = {
            "model": model_name,
            "scenario_id": scenario_id,
            "task_type": scenario["task_type"],
            "benchmark_sha256": benchmark_hash,
            **generated
        }

        # Write immediately so a Colab disconnection
        # does not erase already completed responses.
        with PILOT_PATH.open(
            "a", encoding="utf-8"
        ) as f:
            f.write(
                json.dumps(result, ensure_ascii=False)
                + "\n"
            )

        completed.add((model_name, scenario_id))

        print("Generated tokens:", result["generated_tokens"])
        print("Hit limit:", result["hit_token_limit"])
        print("Response preview:")
        print(result["response"][:500])

        if not result["response"]:
            print("WARNING: Empty generated response.")

    del inference_model
    del base_model

    gc.collect()
    torch.cuda.empty_cache()

# --------------------------------------------------
# 5. Validate pilot completeness
# --------------------------------------------------

all_results = []

with PILOT_PATH.open(encoding="utf-8") as f:
    for line in f:
        if line.strip():
            all_results.append(json.loads(line))

pairs = [
    (r["model"], r["scenario_id"])
    for r in all_results
]

assert len(pairs) == len(set(pairs)), (
    "Duplicate model/scenario pairs."
)

expected = {
    (name, sid)
    for name in ADAPTERS
    for sid in PILOT_IDS
}

assert set(pairs) == expected, (
    f"Pilot incomplete: {expected - set(pairs)}"
)

print("\n" + "=" * 66)
print("THREE-MODEL PILOT INFERENCE: COMPLETED")
print("=" * 66)

print("Responses:", len(all_results))
print("Models:", list(ADAPTERS))
print("Scenarios:", PILOT_IDS)
print("Saved:", PILOT_PATH)
print("Config:", CONFIG_PATH)

print("\nGeneration summary:")

for row in all_results:
    print(
        row["model"],
        row["scenario_id"],
        "tokens =", row["generated_tokens"],
        "limit_hit =", row["hit_token_limit"]
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CURATELAW M5.5 — THREE-MODEL INFERENCE PILOT
GPU: Tesla T4


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Previously completed pilot responses: 0
CURATELAW M5.5 — TOKENIZER OUTPUT FIX
Tokenizer output handling: FIXED
Generation function updated.
No model reload or training performed.

Loading model: base


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Generating base / EVAL001 (full_agreement)
Generated tokens: 1024
Hit limit: True
Response preview:
**INDIAN EMPLOYMENT AGREEMENT**
**This Agreement** (the "Agreement") is made on this ___ day of __________, 20__, between:

**[Company Name]**, a company incorporated under the Indian Companies Act, 2013, with its registered office at [Company Address], Gujarat, India (the "Company"), and

**[Employee Name]**, residing at [Employee Address], Gujarat, India (the "Employee").

**WHEREAS**, the Company desires to employ the Employee as a Frontend Engineer under the terms and conditions set forth he

Generating base / EVAL017 (contract_customization)
Generated tokens: 384
Hit limit: True
Response preview:
The Employee will receive a fixed annual salary of INR 1120000, payable monthly.lllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllllll

Loading model: raw


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Generating raw / EVAL001 (full_agreement)
Generated tokens: 1016
Hit limit: False
Response preview:
**EMPLOYMENT AGREEMENT**

This Employment Agreement (“Agreement”) is made and entered into on this ___ day of __________, 2024 (“Effective Date”), by and between:

**[Company Name]**, a company incorporated under the Companies Act, 2013, having its registered office at [Company Address], Gujarat, India (“Company”), and

**[Employee Name]**, residing at [Employee Address], Gujarat, India (“Employee”).

**WHEREAS**, the Company desires to employ the Employee as a Frontend Engineer under the terms 

Generating raw / EVAL017 (contract_customization)
Generated tokens: 20
Hit limit: False
Response preview:
The Employee will receive a fixed annual salary of INR 1120000, payable monthly.

Loading model: curated


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Generating curated / EVAL001 (full_agreement)
Generated tokens: 1024
Hit limit: True
Response preview:
**EMPLOYMENT AGREEMENT**

This Employment Agreement (“Agreement”) is made and entered into on this ___ day of __________, 2024 (“Effective Date”), by and between:

**[Company Name]**, a company incorporated under the Companies Act, 2013, having its registered office at [Company Address], Gujarat, India (“Company”), and

**[Employee Name]**, residing at [Employee Address], Gujarat, India (“Employee”).

**WHEREAS**, the Company desires to employ the Employee as a Frontend Engineer under the terms 

Generating curated / EVAL017 (contract_customization)
Generated tokens: 20
Hit limit: False
Response preview:
The Employee will receive a fixed annual salary of INR 1120000, payable monthly.

THREE-MODEL PILOT INFERENCE: COMPLETED
Responses: 6
Models: ['base', 'raw', 'curated']
Scenarios: ['EVAL001', 'EVAL017']
Saved: /content/drive/MyDrive/LexiLabs/evaluation_artifacts/inference_pilot_v1.js

Generate all 120 evaluation responses

In [ ]:

import gc
import json
import hashlib
import os
import torch

from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import snapshot_download
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from transformers.utils import is_bitsandbytes_available
from peft import PeftModel

print("=" * 68)
print("CURATELAW M5.6 — FULL THREE-MODEL INFERENCE")
print("=" * 68)

drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/LexiLabs")
ART = ROOT / "finetuning_artifacts"
EVAL = ROOT / "evaluation_artifacts"

FULL_DIR = EVAL / "full_inference_v2"
FULL_DIR.mkdir(parents=True, exist_ok=True)

SHARED_PATH = ART / "shared_training_config.json"
BENCHMARK = EVAL / "heldout_eval_040.jsonl"

shared = json.loads(SHARED_PATH.read_text())

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

EXPECTED_HASH = (
    "944ba82ed4937a41f32c064aaaa89f1b34"
    "aeb2d44a89e0860aac51aa3d46bff9"
)

assert sha256(BENCHMARK) == EXPECTED_HASH

with BENCHMARK.open("r", encoding="utf-8") as f:
    scenarios = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(scenarios) == 40

scenario_ids = [str(x["scenario_id"]) for x in scenarios]
assert len(set(scenario_ids)) == 40

ADAPTERS = {
    "base": None,
    "raw": ART / "model_a_raw" / "checkpoints",
    "curated": ART / "model_b_curated" / "checkpoints",
}

for name, path in ADAPTERS.items():
    if path is not None:
        assert (path / "adapter_config.json").is_file()
        assert (
            (path / "adapter_model.safetensors").is_file()
            or (path / "adapter_model.bin").is_file()
        )

assert torch.cuda.is_available()
assert is_bitsandbytes_available()

# --------------------------------------------------
# 1. Immutable inference configuration
# --------------------------------------------------

CONFIG = {
    "experiment": "CurateLaw_M5",
    "version": "full_v2",
    "benchmark_sha256": EXPECTED_HASH,
    "base_model": shared["model_id"],
    "base_revision": shared["model_revision"],
    "models": ["base", "raw", "curated"],
    "scenario_count": 40,
    "prompt_policy": (
        "instruction, followed by two newlines and "
        "'Provided text:' plus input when nonempty"
    ),
    "generation": {
        "do_sample": False,
        "num_beams": 1,
        "max_new_tokens_full_agreement": 1536,
        "max_new_tokens_other": 384,
    },
    "evaluation_status": "exploratory_not_legally_verified",
}

CONFIG_PATH = FULL_DIR / "inference_config.json"

if CONFIG_PATH.exists():
    previous = json.loads(CONFIG_PATH.read_text())
    assert previous == CONFIG, (
        "Inference configuration changed between runs."
    )
else:
    CONFIG_PATH.write_text(
        json.dumps(CONFIG, indent=2),
        encoding="utf-8"
    )

print("Benchmark integrity: VERIFIED")
print("Inference configuration: FROZEN")

# --------------------------------------------------
# 2. Frozen prompt construction
# --------------------------------------------------

def build_prompt(scenario):
    instruction = str(
        scenario["instruction"]
    ).strip()

    supplied = scenario.get("input", "")

    if supplied is None:
        supplied = ""

    if not isinstance(supplied, str):
        supplied = json.dumps(
            supplied,
            ensure_ascii=False,
            sort_keys=True
        )

    supplied = supplied.strip()

    if supplied:
        return (
            instruction
            + "\n\nProvided text:\n"
            + supplied
        )

    return instruction

def token_limit(scenario):
    if scenario["task_type"] == "full_agreement":
        return 1536
    return 384

# --------------------------------------------------
# 3. Load exact frozen tokenizer and quantization
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_LEXI_TOKEN")
assert HF_TOKEN, "HF_LEXI_TOKEN is missing"

BASE_PATH = snapshot_download(
    repo_id=shared["model_id"],
    revision=shared["model_revision"],
    token=HF_TOKEN
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_PATH
)

assert tokenizer.chat_template is not None

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# --------------------------------------------------
# 4. Inference with corrected tokenizer handling
# --------------------------------------------------

def generate_response(model, scenario):
    prompt = build_prompt(scenario)

    messages = [
        {"role": "user", "content": prompt}
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    if isinstance(encoded, torch.Tensor):
        input_ids = encoded
        attention_mask = torch.ones_like(input_ids)
    else:
        input_ids = encoded["input_ids"]
        attention_mask = encoded.get("attention_mask")

        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)

    if input_ids.ndim == 1:
        input_ids = input_ids.unsqueeze(0)

    if attention_mask.ndim == 1:
        attention_mask = attention_mask.unsqueeze(0)

    input_ids = input_ids.to("cuda")
    attention_mask = attention_mask.to("cuda")

    limit = token_limit(scenario)

    with torch.inference_mode():
        output = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            do_sample=False,
            num_beams=1,
            max_new_tokens=limit,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = output[0, input_ids.shape[-1]:]

    text = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    result = {
        "prompt": prompt,
        "response": text,
        "prompt_tokens": int(input_ids.shape[-1]),
        "generated_tokens": int(generated.numel()),
        "max_new_tokens": limit,
        "hit_token_limit": (
            int(generated.numel()) >= limit
        ),
        "empty_response": not bool(text),
    }

    del input_ids, attention_mask, output, generated
    return result

# --------------------------------------------------
# 5. Read previous results safely
# --------------------------------------------------

def read_results(path, model_name):
    rows = {}

    if not path.exists():
        return rows

    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            if not line.strip():
                continue

            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError(
                    f"Invalid JSON at {path}:{line_number}. "
                    "Inspect before resuming."
                ) from exc

            assert row["model"] == model_name
            assert row["benchmark_sha256"] == EXPECTED_HASH
            assert row["inference_version"] == "full_v2"

            sid = row["scenario_id"]
            assert sid in scenario_ids

            if sid in rows:
                raise RuntimeError(
                    f"Duplicate result: {model_name}/{sid}"
                )

            rows[sid] = row

    return rows

# --------------------------------------------------
# 6. Run models one at a time
# --------------------------------------------------

for model_name, adapter_dir in ADAPTERS.items():

    output_path = FULL_DIR / f"{model_name}_predictions.jsonl"

    existing = read_results(output_path, model_name)

    pending = [
        scenario
        for scenario in scenarios
        if scenario["scenario_id"] not in existing
    ]

    print("\n" + "=" * 68)
    print("MODEL:", model_name.upper())
    print("Already completed:", len(existing))
    print("Remaining:", len(pending))
    print("=" * 68)

    if not pending:
        continue

    gc.collect()
    torch.cuda.empty_cache()

    allocated = torch.cuda.memory_allocated() / 1024**3
    if allocated >= 1:
        raise RuntimeError(
            f"GPU has {allocated:.2f} GB allocated. "
            "Release old model objects or restart Colab."
        )

    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_PATH,
        quantization_config=bnb_config,
        dtype=torch.float16,
        device_map={"": 0},
        attn_implementation="sdpa",
        low_cpu_mem_usage=True
    )

    if adapter_dir is None:
        inference_model = base_model
    else:
        inference_model = PeftModel.from_pretrained(
            base_model,
            str(adapter_dir),
            is_trainable=False,
            autocast_adapter_dtype=True
        )

    inference_model.eval()

    for scenario in pending:
        sid = scenario["scenario_id"]

        print(
            f"{model_name} / {sid} "
            f"({scenario['task_type']})"
        )

        generated = generate_response(
            inference_model,
            scenario
        )

        row = {
            "model": model_name,
            "scenario_id": sid,
            "task_type": scenario["task_type"],
            "benchmark_sha256": EXPECTED_HASH,
            "inference_version": "full_v2",
            **generated
        }

        # Save each completed response immediately.
        with output_path.open("a", encoding="utf-8") as f:
            f.write(
                json.dumps(row, ensure_ascii=False)
                + "\n"
            )
            f.flush()

        print(
            "  Tokens:", generated["generated_tokens"],
            "| Limit:", generated["hit_token_limit"],
            "| Empty:", generated["empty_response"]
        )

    del inference_model
    del base_model

    gc.collect()
    torch.cuda.empty_cache()

    print(model_name.upper(), "MODEL INFERENCE COMPLETE")

# --------------------------------------------------
# 7. Validate 120-response completeness
# --------------------------------------------------

print("\n" + "=" * 68)
print("FINAL INFERENCE VALIDATION")
print("=" * 68)

combined = []

for model_name in ADAPTERS:
    path = FULL_DIR / f"{model_name}_predictions.jsonl"

    rows = read_results(path, model_name)

    print(model_name, "responses:", len(rows))

    assert set(rows) == set(scenario_ids), (
        f"Missing scenarios for {model_name}: "
        f"{set(scenario_ids) - set(rows)}"
    )

    combined.extend(rows.values())

assert len(combined) == 120

summary = {}

for model_name in ADAPTERS:
    rows = [
        x for x in combined
        if x["model"] == model_name
    ]

    summary[model_name] = {
        "responses": len(rows),
        "empty_responses": sum(
            x["empty_response"] for x in rows
        ),
        "token_limit_hits": sum(
            x["hit_token_limit"] for x in rows
        ),
        "total_generated_tokens": sum(
            x["generated_tokens"] for x in rows
        )
    }

SUMMARY_PATH = FULL_DIR / "inference_summary.json"

SUMMARY_PATH.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

print("\n" + json.dumps(summary, indent=2))

print("\n" + "=" * 68)
print("M5.6 — ALL 120 RESPONSES GENERATED")
print("=" * 68)

print("Results:", FULL_DIR)
print("Summary:", SUMMARY_PATH)
print("No training performed.")


CURATELAW M5.6 — FULL THREE-MODEL INFERENCE
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Benchmark integrity: VERIFIED
Inference configuration: FROZEN


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]


MODEL: BASE
Already completed: 0
Remaining: 40


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

base / EVAL001 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL002 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL003 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL004 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL005 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL006 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL007 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL008 (full_agreement)
  Tokens: 1536 | Limit: True | Empty: False
base / EVAL009 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
base / EVAL010 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
base / EVAL011 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
base / EVAL012 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
base / EVAL013 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
base / EVAL0

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

raw / EVAL001 (full_agreement)
  Tokens: 1016 | Limit: False | Empty: False
raw / EVAL002 (full_agreement)
  Tokens: 936 | Limit: False | Empty: False
raw / EVAL003 (full_agreement)
  Tokens: 1256 | Limit: False | Empty: False
raw / EVAL004 (full_agreement)
  Tokens: 1151 | Limit: False | Empty: False
raw / EVAL005 (full_agreement)
  Tokens: 1390 | Limit: False | Empty: False
raw / EVAL006 (full_agreement)
  Tokens: 1342 | Limit: False | Empty: False
raw / EVAL007 (full_agreement)
  Tokens: 1272 | Limit: False | Empty: False
raw / EVAL008 (full_agreement)
  Tokens: 946 | Limit: False | Empty: False
raw / EVAL009 (clause_drafting)
  Tokens: 245 | Limit: False | Empty: False
raw / EVAL010 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
raw / EVAL011 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
raw / EVAL012 (clause_drafting)
  Tokens: 309 | Limit: False | Empty: False
raw / EVAL013 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
raw / EVAL014 (cl

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

curated / EVAL001 (full_agreement)
  Tokens: 1222 | Limit: False | Empty: False
curated / EVAL002 (full_agreement)
  Tokens: 1015 | Limit: False | Empty: False
curated / EVAL003 (full_agreement)
  Tokens: 1161 | Limit: False | Empty: False
curated / EVAL004 (full_agreement)
  Tokens: 1165 | Limit: False | Empty: False
curated / EVAL005 (full_agreement)
  Tokens: 910 | Limit: False | Empty: False
curated / EVAL006 (full_agreement)
  Tokens: 1173 | Limit: False | Empty: False
curated / EVAL007 (full_agreement)
  Tokens: 1452 | Limit: False | Empty: False
curated / EVAL008 (full_agreement)
  Tokens: 1198 | Limit: False | Empty: False
curated / EVAL009 (clause_drafting)
  Tokens: 255 | Limit: False | Empty: False
curated / EVAL010 (clause_drafting)
  Tokens: 384 | Limit: True | Empty: False
curated / EVAL011 (clause_drafting)
  Tokens: 160 | Limit: False | Empty: False
curated / EVAL012 (clause_drafting)
  Tokens: 314 | Limit: False | Empty: False
curated / EVAL013 (clause_drafting)
  Toke

AuditKIT automated diagnostics

In [ ]:
%pip install -U auditkit

Score the 120 saved predictions

In [ ]:

import json
import re
import csv
import hashlib
from pathlib import Path
from collections import defaultdict

import auditkit as ak

print("=" * 65)
print("CURATELAW M5.7 — AUDITKIT OUTPUT DIAGNOSTICS")
print("=" * 65)

ROOT = Path("/content/drive/MyDrive/LexiLabs")
FULL = ROOT / "evaluation_artifacts/full_inference_v2"
OUT = ROOT / "evaluation_artifacts/auditkit_results"
OUT.mkdir(parents=True, exist_ok=True)

BENCHMARK = (
    ROOT / "evaluation_artifacts/heldout_eval_040.jsonl"
)

EXPECTED_SHA = (
    "944ba82ed4937a41f32c064aaaa89f1b34"
    "aeb2d44a89e0860aac51aa3d46bff9"
)

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()

assert sha256(BENCHMARK) == EXPECTED_SHA

scenarios = [
    json.loads(line)
    for line in BENCHMARK.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

scenario_map = {
    row["scenario_id"]: row
    for row in scenarios
}

assert len(scenario_map) == 40

models = ["base", "raw", "curated"]
predictions = {}

for name in models:
    path = FULL / f"{name}_predictions.jsonl"

    rows = [
        json.loads(line)
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    assert len(rows) == 40
    assert len({r["scenario_id"] for r in rows}) == 40

    for row in rows:
        assert row["model"] == name
        assert row["benchmark_sha256"] == EXPECTED_SHA
        assert row["inference_version"] == "full_v2"
        assert row["scenario_id"] in scenario_map

    predictions[name] = {
        row["scenario_id"]: row
        for row in rows
    }

print("Prediction integrity: PASSED")
print("Matched scenarios:", len(scenario_map))
print("Models:", models)

# --------------------------------------------------
# Output-only diagnostic scorers
# --------------------------------------------------

@ak.scorer(
    direction=ak.Direction.MAXIMIZE,
    name="not_token_limited"
)
def not_token_limited(sample, output):
    return float(
        not sample.metadata["hit_token_limit"]
    )

@ak.scorer(
    direction=ak.Direction.MAXIMIZE,
    name="nonempty_response"
)
def nonempty_response(sample, output):
    return float(bool(str(output).strip()))

@ak.scorer(
    direction=ak.Direction.MAXIMIZE,
    name="no_character_run_40"
)
def no_character_run_40(sample, output):
    # Detect runs such as "llllllllllllllll..."
    # Diagnostic only, not a full repetition metric.
    return float(
        re.search(
            r"([A-Za-z])\1{39,}",
            str(output)
        ) is None
    )

scorers = [
    not_token_limited,
    nonempty_response,
    no_character_run_40,
]

results = {}
summary_rows = []
detail_rows = []

for model_name in models:
    samples = []

    for scenario in scenarios:
        sid = scenario["scenario_id"]
        pred = predictions[model_name][sid]

        samples.append(
            ak.Sample(
                id=sid,
                input=pred["prompt"],
                task=scenario["task_type"],
                actual_output=pred["response"],
                metadata={
                    "hit_token_limit": pred[
                        "hit_token_limit"
                    ],
                    "generated_tokens": pred[
                        "generated_tokens"
                    ],
                    "expected_facts": scenario[
                        "expected_facts"
                    ],
                },
            )
        )

    # AuditKIT scores existing outputs.
    # No model generation is performed.
    result = ak.evaluate(
        samples,
        model="precomputed",
        scorers=scorers,
    )

    if getattr(result, "errors", None):
        raise RuntimeError(
            f"AuditKIT errors for {model_name}: "
            f"{result.errors}"
        )

    if len(result.predictions) != 40:
        raise RuntimeError(
            f"AuditKIT scored fewer than 40 "
            f"items for {model_name}"
        )

    results[model_name] = result

    save_path = OUT / (
        f"{model_name}_output_diagnostics.json"
    )

    result.save(str(save_path), fmt="json")

    print(f"\n{model_name.upper()} — AUDITKIT")
    print(result.summary())

    model_rows = list(predictions[model_name].values())

    summary_rows.append({
        "model": model_name,
        "responses": 40,
        "token_limit_hits": sum(
            r["hit_token_limit"] for r in model_rows
        ),
        "nonempty_responses": sum(
            bool(r["response"].strip())
            for r in model_rows
        ),
        "long_character_run_40": sum(
            bool(re.search(
                r"([A-Za-z])\1{39,}",
                r["response"]
            ))
            for r in model_rows
        ),
        "generated_tokens": sum(
            r["generated_tokens"]
            for r in model_rows
        ),
    })

    for scenario in scenarios:
        sid = scenario["scenario_id"]
        pred = predictions[model_name][sid]

        detail_rows.append({
            "model": model_name,
            "scenario_id": sid,
            "task_type": scenario["task_type"],
            "token_limit_hit": int(
                pred["hit_token_limit"]
            ),
            "nonempty": int(
                bool(pred["response"].strip())
            ),
            "long_character_run_40": int(
                bool(re.search(
                    r"([A-Za-z])\1{39,}",
                    pred["response"]
                ))
            ),
            "generated_tokens": pred[
                "generated_tokens"
            ],
        })

# --------------------------------------------------
# Save summary and scenario-level diagnostics
# --------------------------------------------------

summary_path = OUT / "output_diagnostics_summary.csv"
detail_path = OUT / "output_diagnostics_per_scenario.csv"

with summary_path.open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f, fieldnames=list(summary_rows[0])
    )
    writer.writeheader()
    writer.writerows(summary_rows)

with detail_path.open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f, fieldnames=list(detail_rows[0])
    )
    writer.writeheader()
    writer.writerows(detail_rows)

print("\n" + "=" * 65)
print("AUDITKIT OUTPUT DIAGNOSTICS COMPLETED")
print("=" * 65)

for row in summary_rows:
    print(row)

print("\nSaved results:")
print(summary_path)
print(detail_path)
print(OUT)

print("\nNOTE:")
print("These are output diagnostics, not factual")
print("accuracy scores or legal correctness ratings.")


CURATELAW M5.7 — AUDITKIT OUTPUT DIAGNOSTICS
Prediction integrity: PASSED
Matched scenarios: 40
Models: ['base', 'raw', 'curated']

BASE — AUDITKIT
run d858348df1c93eb1 (fingerprint d858348df1c93eb1)
  not_token_limited: 0.0250 ± 0.0247  (n=40)
  nonempty_response: 1.0000 ± 0.0000  (n=40)
  no_character_run_40: 0.9750 ± 0.0247  (n=40)
  model: precomputed (not a local model -- no measurable size)

RAW — AUDITKIT
run 88c3be62b5b038f9 (fingerprint 88c3be62b5b038f9)
  not_token_limited: 0.8500 ± 0.0565  (n=40)
  nonempty_response: 1.0000 ± 0.0000  (n=40)
  no_character_run_40: 1.0000 ± 0.0000  (n=40)
  model: precomputed (not a local model -- no measurable size)

CURATED — AUDITKIT
run e3adaf07244cddc2 (fingerprint e3adaf07244cddc2)
  not_token_limited: 0.8750 ± 0.0523  (n=40)
  nonempty_response: 1.0000 ± 0.0000  (n=40)
  no_character_run_40: 1.0000 ± 0.0000  (n=40)
  model: precomputed (not a local model -- no measurable size)

AUDITKIT OUTPUT DIAGNOSTICS COMPLETED
{'model': 'base', 're

Expected-facts evaluation

In [18]:
import json
from pathlib import Path
from collections import defaultdict, Counter

ROOT = Path("/content/drive/MyDrive/LexiLabs")
BENCHMARK = ROOT / "evaluation_artifacts/heldout_eval_040.jsonl"

rows = [
    json.loads(line)
    for line in BENCHMARK.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(rows) == 40

fields = defaultdict(Counter)

for row in rows:
    for key, value in row["expected_facts"].items():
        fields[key][type(value).__name__] += 1

print("=" * 65)
print("M5.8 — EXPECTED FACT SCHEMA")
print("=" * 65)

for key in sorted(fields):
    print(f"{key}: {dict(fields[key])}")

print("\nFACTS BY SCENARIO")
for row in rows:
    print(
        row["scenario_id"],
        row["task_type"],
        json.dumps(row["expected_facts"], ensure_ascii=False)
    )

M5.8 — EXPECTED FACT SCHEMA
contractual_leave_days: {'int': 1}
duties: {'list': 1}
exception: {'str': 1}
exceptions: {'list': 1}
extension_months: {'int': 2}
hours_per_week: {'int': 10}
initial_months: {'int': 1}
leave_days: {'int': 2}
location: {'str': 2}
new_salary_inr: {'int': 1}
notice_days: {'int': 12}
office_days: {'int': 4}
parties: {'list': 1}
probation_months: {'int': 11}
remote_days: {'int': 4}
required: {'list': 1}
role: {'str': 10}
salary_inr: {'int': 10}
scope: {'str': 2}
state: {'str': 8}
weekly_hours: {'int': 1}
work_mode: {'str': 8}

FACTS BY SCENARIO
EVAL001 full_agreement {"role": "Frontend Engineer", "state": "Gujarat", "salary_inr": 1100000, "work_mode": "hybrid", "hours_per_week": 40, "probation_months": 4, "notice_days": 45}
EVAL002 full_agreement {"role": "Site Reliability Engineer", "state": "Karnataka", "salary_inr": 2100000, "work_mode": "remote", "hours_per_week": 40, "probation_months": 5, "notice_days": 60}
EVAL003 full_agreement {"role": "Product Designer"

AuditKIT expected-fact coverage and paired comparison

In [ ]:

import json
import re
import csv
import math
import random
import hashlib
import statistics
import unicodedata
from pathlib import Path
from collections import defaultdict

import auditkit as ak

print("=" * 68)
print("CURATELAW M5.8 — EXPECTED-FACT EVALUATION")
print("=" * 68)

ROOT = Path("/content/curatelaw_drive/MyDrive/LexiLabs")
EVAL_DIR = ROOT / "evaluation_artifacts"
FULL_DIR = EVAL_DIR / "full_inference_v2"
OUT_DIR = EVAL_DIR / "auditkit_results" / "expected_facts_v1"

OUT_DIR.mkdir(parents=True, exist_ok=True)

BENCHMARK = Path(
    "/content/drive/MyDrive/LexiLabs/evaluation_artifacts/"
    "heldout_eval_040.jsonl"
)

EXPECTED_SHA = (
    "944ba82ed4937a41f32c064aaaa89f1b34"
    "aeb2d44a89e0860aac51aa3d46bff9"
)

MODELS = ["base", "raw", "curated"]

def sha256(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

assert sha256(BENCHMARK) == EXPECTED_SHA

scenarios = load_jsonl(BENCHMARK)

assert len(scenarios) == 40

scenario_map = {
    str(row["scenario_id"]): row
    for row in scenarios
}

assert len(scenario_map) == 40

predictions = {}

for model_name in MODELS:
    path = FULL_DIR / f"{model_name}_predictions.jsonl"
    rows = load_jsonl(path)

    assert len(rows) == 40

    mapping = {}

    for row in rows:
        sid = str(row["scenario_id"])
        assert sid in scenario_map
        assert sid not in mapping
        assert row["model"] == model_name
        assert row["benchmark_sha256"] == EXPECTED_SHA
        assert row["inference_version"] == "full_v2"
        mapping[sid] = row

    assert set(mapping) == set(scenario_map)
    predictions[model_name] = mapping

print("Benchmark: VERIFIED")
print("Prediction count:", sum(len(x) for x in predictions.values()))

# --------------------------------------------------
# 1. Deterministic fact-presence rules
# --------------------------------------------------

def normalize_text(value):
    text = unicodedata.normalize(
        "NFKC", str(value)
    ).casefold()

    text = text.replace("–", "-").replace("—", "-")
    text = text.replace("‑", "-")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

# Numeric matching recognizes 1100000, 11,00,000,
# and 1,100,000 as the same value.
#
# It deliberately does not interpret prose such as
# "eleven lakh". Such cases require review.

NUM_PATTERN = re.compile(
    r"(?<![\w])"
    r"(?:\d{1,3}(?:,\d{2})+,\d{3}"
    r"|\d{1,3}(?:,\d{3})+"
    r"|\d+)"
    r"(?![\w])"
)

def numbers_in(text):
    found = set()

    for match in NUM_PATTERN.finditer(str(text)):
        raw = match.group(0).replace(",", "")
        try:
            found.add(int(raw))
        except ValueError:
            pass

    return found

ALIASES = {
    "on-site": ["on-site", "onsite", "on site"],
    "remote": ["remote", "work from home", "wfh"],
    "hybrid": ["hybrid"],

    "work-related creations": [
        "work-related creations",
        "work-related intellectual property",
        "work-related inventions",
    ],
    "work-related IP": [
        "work-related ip",
        "work-related intellectual property",
    ],

    "public information": [
        "public information",
        "publicly available information",
        "public domain",
    ],
    "lawful disclosure": [
        "lawful disclosure",
        "disclosure required by law",
        "legally required disclosure",
    ],
    "reporting channel": [
        "reporting channel",
        "reporting mechanism",
        "reporting procedure",
        "reporting process",
    ],
    "non-retaliation": [
        "non-retaliation",
        "non retaliation",
        "protection against retaliation",
        "no retaliation",
    ],
    "pipelines": ["pipelines", "pipeline"],
    "monitoring": ["monitoring", "monitor"],
    "documentation": ["documentation", "documenting"],
    "employer": ["employer", "company"],
    "employee": ["employee"],
}

def contains_phrase(text, phrase):
    # Word boundaries prevent partial matches, e.g.
    # "remote" inside an unrelated longer word.
    pattern = (
        r"(?<!\w)"
        + re.escape(normalize_text(phrase))
        + r"(?!\w)"
    )

    return bool(re.search(pattern, text))

def check_fact(response, key, value):
    normalized = normalize_text(response)

    if isinstance(value, bool):
        return {
            "matched": False,
            "method": "unsupported_boolean",
        }

    if isinstance(value, int):
        return {
            "matched": value in numbers_in(response),
            "method": "numeric_literal",
        }

    if isinstance(value, str):
        aliases = ALIASES.get(value, [value])

        return {
            "matched": any(
                contains_phrase(normalized, alias)
                for alias in aliases
            ),
            "method": (
                "literal_or_predefined_alias"
                if value in ALIASES
                else "literal_phrase"
            ),
        }

    return {
        "matched": False,
        "method": "unsupported_type",
    }

def evaluate_facts(response, expected_facts):
    details = []

    for key, value in expected_facts.items():
        values = value if isinstance(value, list) else [value]

        for item in values:
            result = check_fact(response, key, item)

            details.append({
                "field": key,
                "expected": item,
                "matched": result["matched"],
                "method": result["method"],
            })

    # Every required list entry is scored separately.
    total = len(details)
    matched = sum(x["matched"] for x in details)

    return {
        "matched": matched,
        "total": total,
        "coverage": matched / total if total else None,
        "details": details,
    }

# --------------------------------------------------
# 2. Flags for editing and contradiction tasks
# --------------------------------------------------

EDIT_TASKS = {
    "contract_customization",
    "contradiction_correction",
}

def old_number_flags(scenario, response):
    if scenario["task_type"] not in EDIT_TASKS:
        return []

    source_numbers = numbers_in(
        scenario.get("input", "")
    )

    expected_numbers = set()

    for value in scenario["expected_facts"].values():
        values = value if isinstance(value, list) else [value]

        for item in values:
            if isinstance(item, int) and not isinstance(item, bool):
                expected_numbers.add(item)

    # This is deliberately only a review flag:
    # an original number may legitimately be retained.
    old_candidates = source_numbers - expected_numbers
    present = numbers_in(response)

    return sorted(old_candidates & present)

# --------------------------------------------------
# 3. Build per-scenario results
# --------------------------------------------------

details = []
fact_rows = []

for model_name in MODELS:
    for scenario in scenarios:
        sid = str(scenario["scenario_id"])
        pred = predictions[model_name][sid]

        result = evaluate_facts(
            pred["response"],
            scenario["expected_facts"]
        )

        old_flags = old_number_flags(
            scenario,
            pred["response"]
        )

        item = {
            "model": model_name,
            "scenario_id": sid,
            "task_type": scenario["task_type"],
            "matched_facts": result["matched"],
            "total_facts": result["total"],
            "fact_coverage": result["coverage"],
            "all_expected_facts_detected": (
                result["matched"] == result["total"]
            ),
            "old_numeric_values_flagged": old_flags,
            "token_limit_hit": bool(
                pred["hit_token_limit"]
            ),
            "generated_tokens": int(
                pred["generated_tokens"]
            ),
        }

        details.append(item)

        for fact in result["details"]:
            fact_rows.append({
                "model": model_name,
                "scenario_id": sid,
                "task_type": scenario["task_type"],
                **fact,
            })

assert len(details) == 120

# --------------------------------------------------
# 4. AuditKIT scorer using precomputed predictions
# --------------------------------------------------

@ak.scorer(
    direction=ak.Direction.MAXIMIZE,
    name="expected_fact_coverage"
)
def expected_fact_coverage(sample, output):
    return evaluate_facts(
        str(output),
        sample.metadata["expected_facts"]
    )["coverage"]

audit_results = {}

for model_name in MODELS:
    samples = []

    for scenario in scenarios:
        sid = scenario["scenario_id"]
        pred = predictions[model_name][sid]

        samples.append(
            ak.Sample(
                id=sid,
                input=pred["prompt"],
                task=scenario["task_type"],
                actual_output=pred["response"],
                metadata={
                    "expected_facts": scenario[
                        "expected_facts"
                    ],
                },
            )
        )

    audit_result = ak.evaluate(
        samples,
        model="precomputed",
        scorers=[expected_fact_coverage],
    )

    if getattr(audit_result, "errors", None):
        raise RuntimeError(
            f"AuditKIT scoring failed for {model_name}: "
            f"{audit_result.errors}"
        )

    assert len(audit_result.predictions) == 40

    audit_result.save(
        str(OUT_DIR / f"{model_name}_auditkit.json"),
        fmt="json"
    )

    audit_results[model_name] = audit_result

    print(f"\n{model_name.upper()} — AUDITKIT")
    print(audit_result.summary())

# --------------------------------------------------
# 5. Matched model comparison
# --------------------------------------------------

by_model = {
    model_name: {
        row["scenario_id"]: row
        for row in details
        if row["model"] == model_name
    }
    for model_name in MODELS
}

def model_average(model_name, task=None):
    rows = list(by_model[model_name].values())

    if task is not None:
        rows = [
            row for row in rows
            if row["task_type"] == task
        ]

    return statistics.mean(
        row["fact_coverage"] for row in rows
    )

def paired_difference(model_left, model_right, task=None):
    ids = [
        scenario["scenario_id"]
        for scenario in scenarios
        if task is None or scenario["task_type"] == task
    ]

    return [
        by_model[model_left][sid]["fact_coverage"]
        - by_model[model_right][sid]["fact_coverage"]
        for sid in ids
    ]

raw_curated_diffs = paired_difference("curated", "raw")

wins = sum(d > 1e-12 for d in raw_curated_diffs)
ties = sum(abs(d) <= 1e-12 for d in raw_curated_diffs)
losses = sum(d < -1e-12 for d in raw_curated_diffs)

# Paired bootstrap uncertainty for the exploratory
# 40-scenario score difference. Not legal validation.
rng = random.Random(42)
bootstrap = []

for _ in range(10000):
    sampled = [
        raw_curated_diffs[rng.randrange(40)]
        for _ in range(40)
    ]
    bootstrap.append(statistics.mean(sampled))

bootstrap.sort()

ci_low = bootstrap[249]
ci_high = bootstrap[9749]

task_types = sorted({
    row["task_type"]
    for row in scenarios
})

summary = {
    "benchmark_sha256": EXPECTED_SHA,
    "evaluation_version": "expected_facts_v1",
    "metric": "heuristic_expected_fact_presence",
    "legal_validation": False,
    "model_average_coverage": {
        name: model_average(name)
        for name in MODELS
    },
    "model_average_by_task": {
        task: {
            name: model_average(name, task)
            for name in MODELS
        }
        for task in task_types
    },
    "curated_minus_raw": {
        "mean_paired_difference": statistics.mean(
            raw_curated_diffs
        ),
        "wins": wins,
        "ties": ties,
        "losses": losses,
        "bootstrap_95pct_interval": [
            ci_low, ci_high
        ],
    },
    "cautions": [
        "No gold reference answers or completed legal verification.",
        "Numeric detection recognizes digit forms, not number words.",
        "Text detection uses predefined aliases and literal phrases.",
        "Fact presence does not establish correct legal context.",
        "Old numeric values are review flags, not automatic failures.",
        "Bootstrap estimates are exploratory with only 40 scenarios.",
    ],
}

# --------------------------------------------------
# 6. Save CSV and JSON outputs
# --------------------------------------------------

def write_csv(path, rows, fields):
    with path.open(
        "w", newline="", encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fields
        )
        writer.writeheader()
        writer.writerows(rows)

detail_csv_rows = []

for row in details:
    new_row = dict(row)
    new_row["old_numeric_values_flagged"] = json.dumps(
        new_row["old_numeric_values_flagged"]
    )
    detail_csv_rows.append(new_row)

write_csv(
    OUT_DIR / "scenario_scores.csv",
    detail_csv_rows,
    list(detail_csv_rows[0].keys())
)

fact_csv_rows = []

for row in fact_rows:
    new_row = dict(row)
    new_row["expected"] = json.dumps(
        new_row["expected"],
        ensure_ascii=False
    )
    fact_csv_rows.append(new_row)

write_csv(
    OUT_DIR / "fact_level_checks.csv",
    fact_csv_rows,
    list(fact_csv_rows[0].keys())
)

(OUT_DIR / "summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# --------------------------------------------------
# 7. Display results
# --------------------------------------------------

print("\n" + "=" * 68)
print("EXPECTED-FACT COVERAGE SUMMARY")
print("=" * 68)

for name in MODELS:
    print(
        f"{name.upper():8s}",
        f"{100 * model_average(name):.2f}%"
    )

print("\nBY TASK")
print(
    f"{'Task':28s} {'Base':>8s} "
    f"{'Raw':>8s} {'Curated':>8s}"
)

for task in task_types:
    print(
        f"{task:28s} "
        f"{100*model_average('base',task):8.2f} "
        f"{100*model_average('raw',task):8.2f} "
        f"{100*model_average('curated',task):8.2f}"
    )

print("\nRAW VS CURATED — MATCHED COMPARISON")
print("Curated wins:", wins)
print("Ties:", ties)
print("Raw wins:", losses)

print(
    "Mean difference (curated - raw):",
    round(statistics.mean(raw_curated_diffs), 4)
)

print(
    "Exploratory paired 95% bootstrap interval:",
    [round(ci_low, 4), round(ci_high, 4)]
)

print("\nSCENARIOS NEEDING REVIEW")

for row in details:
    if (
        row["old_numeric_values_flagged"]
        or row["fact_coverage"] < 1.0
    ):
        print(
            row["model"],
            row["scenario_id"],
            "coverage=",
            round(row["fact_coverage"], 3),
            "old_numeric_flags=",
            row["old_numeric_values_flagged"]
        )

print("\n" + "=" * 68)
print("M5.8 — AUTOMATED FACT CHECKS COMPLETE")
print("=" * 68)

print("Saved directory:", OUT_DIR)
print("Scenario scores:", OUT_DIR / "scenario_scores.csv")
print("Fact checks:", OUT_DIR / "fact_level_checks.csv")
print("Summary:", OUT_DIR / "summary.json")

print("\nThese scores measure detectable fact presence.")
print("They are NOT legally verified accuracy scores.")


CURATELAW M5.8 — EXPECTED-FACT EVALUATION
Benchmark: VERIFIED
Prediction count: 120

BASE — AUDITKIT
run cae23d0dd253fbf6 (fingerprint cae23d0dd253fbf6)
  expected_fact_coverage: 0.7399 ± 0.0619  (n=40)
  model: precomputed (not a local model -- no measurable size)

RAW — AUDITKIT
run ff5198772a87061a (fingerprint ff5198772a87061a)
  expected_fact_coverage: 0.7179 ± 0.0615  (n=40)
  model: precomputed (not a local model -- no measurable size)

CURATED — AUDITKIT
run e2d7c748072b7278 (fingerprint e2d7c748072b7278)
  expected_fact_coverage: 0.6893 ± 0.0660  (n=40)
  model: precomputed (not a local model -- no measurable size)

EXPECTED-FACT COVERAGE SUMMARY
BASE     73.99%
RAW      71.79%
CURATED  68.93%

BY TASK
Task                             Base      Raw  Curated
clause_drafting                 81.25    62.50    50.00
contract_customization          75.00    75.00    75.00
contradiction_correction        75.00    87.50    75.00
full_agreement                  80.36    71.43    82.14

Inspect the six decisive disagreements

In [ ]:

import json
import csv
from pathlib import Path

print("=" * 65)
print("CURATELAW M5.9 — RAW VS CURATED DISAGREEMENTS")
print("=" * 65)

ROOT = Path(
    "/content/curatelaw_drive/MyDrive/LexiLabs"
)

EVAL = ROOT / "evaluation_artifacts"
PRED = EVAL / "full_inference_v2"
SCORES = EVAL / "auditkit_results/expected_facts_v1"

REVIEW_DIR = EVAL / "disagreement_review_v1"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)

IDS = [
    "EVAL001",
    "EVAL004",
    "EVAL006",
    "EVAL008",
    "EVAL013",
    "EVAL026",
]

def read_jsonl(path):
    assert path.is_file(), path
    return [
        json.loads(line)
        for line in path.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

benchmark = {
    r["scenario_id"]: r
    for r in read_jsonl(EVAL / "heldout_eval_040.jsonl")
}

predictions = {}

for model in ["raw", "curated"]:
    path = PRED / f"{model}_predictions.jsonl"

    predictions[model] = {
        r["scenario_id"]: r
        for r in read_jsonl(path)
    }

with (SCORES / "scenario_scores.csv").open(
    encoding="utf-8", newline=""
) as f:
    score_rows = list(csv.DictReader(f))

scores = {
    (r["model"], r["scenario_id"]): r
    for r in score_rows
}

review_rows = []

for sid in IDS:
    scenario = benchmark[sid]

    print("\n" + "=" * 65)
    print("SCENARIO:", sid)
    print("TASK:", scenario["task_type"])
    print("=" * 65)

    print("\nINSTRUCTION:")
    print(scenario["instruction"])

    print("\nORIGINAL INPUT:")
    print(scenario.get("input", ""))

    print("\nEXPECTED FACTS:")
    print(json.dumps(
        scenario["expected_facts"],
        indent=2,
        ensure_ascii=False
    ))

    for model in ["raw", "curated"]:
        prediction = predictions[model][sid]
        score = scores[(model, sid)]

        print("\n" + "-" * 65)
        print(model.upper())
        print("-" * 65)

        print("Heuristic coverage:", score["fact_coverage"])
        print("Token limit hit:", prediction["hit_token_limit"])

        print("\nFULL MODEL RESPONSE:")
        print(prediction["response"])

        review_rows.append({
            "scenario_id": sid,
            "task_type": scenario["task_type"],
            "model": model,
            "instruction": scenario["instruction"],
            "input": scenario.get("input", ""),
            "expected_facts": json.dumps(
                scenario["expected_facts"],
                ensure_ascii=False
            ),
            "response": prediction["response"],
            "heuristic_coverage": score["fact_coverage"],
            "hit_token_limit": prediction["hit_token_limit"],
            "review_status": "PENDING",
            "review_notes": "",
        })

review_path = REVIEW_DIR / "six_disagreements.csv"

# Protect any existing reviewer annotations.
if review_path.exists():
    raise RuntimeError(
        "Review CSV already exists. "
        "Preserving previous reviewer work."
    )

with review_path.open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f, fieldnames=list(review_rows[0])
    )
    writer.writeheader()
    writer.writerows(review_rows)

print("\n" + "=" * 65)
print("DISAGREEMENT AUDIT PREPARED")
print("=" * 65)

print("Scenarios:", len(IDS))
print("Responses inspected:", len(review_rows))
print("Review CSV:", review_path)
print("Existing evaluation scores unchanged.")


CURATELAW M5.9 — RAW VS CURATED DISAGREEMENTS

SCENARIO: EVAL001
TASK: full_agreement

INSTRUCTION:
Prepare an illustrative Indian employment agreement for a Frontend Engineer employed in Gujarat. The annual fixed salary is INR 1100000; the arrangement is hybrid; the scheduled week is 40 hours; probation is 4 months; and mutual contractual notice is 45 days, subject to applicable law. Use party placeholders. Cover appointment, duties, location, salary, leave, applicable statutory benefits, confidentiality, work-related IP, workplace conduct, termination, disputes and signatures. Do not invent statutory entitlements or impose a post-employment non-compete.

ORIGINAL INPUT:


EXPECTED FACTS:
{
  "role": "Frontend Engineer",
  "state": "Gujarat",
  "salary_inr": 1100000,
  "work_mode": "hybrid",
  "hours_per_week": 40,
  "probation_months": 4,
  "notice_days": 45
}

-----------------------------------------------------------------
RAW
------------------------------------------------------

Correct the numeric scorer consistently

In [6]:
%pip install -q auditkit

In [7]:

import json
import csv
import re
import random
import hashlib
import statistics
import unicodedata
from pathlib import Path

import auditkit as ak

print("=" * 68)
print("CURATELAW M5.10 — CORRECTED FACT COVERAGE")
print("=" * 68)

# ============================================================
# 1. PATHS AND FROZEN BENCHMARK
# ============================================================

ROOT = Path("/content/drive/MyDrive/LexiLabs")
EVAL = ROOT / "evaluation_artifacts"
FULL = EVAL / "full_inference_v2"
OUT = EVAL / "auditkit_results" / "expected_facts_v2"

BENCHMARK = EVAL / "heldout_eval_040.jsonl"

EXPECTED_SHA = (
    "944ba82ed4937a41f32c064aaaa89f1b34"
    "aeb2d44a89e0860aac51aa3d46bff9"
)

MODELS = ["base", "raw", "curated"]

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def load_jsonl(path):
    assert path.is_file(), f"Missing file: {path}"
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

assert BENCHMARK.is_file()
assert file_hash(BENCHMARK) == EXPECTED_SHA

scenarios = load_jsonl(BENCHMARK)
assert len(scenarios) == 40

scenario_ids = {
    str(s["scenario_id"]) for s in scenarios
}

assert len(scenario_ids) == 40

predictions = {}

for model in MODELS:
    path = FULL / f"{model}_predictions.jsonl"
    rows = load_jsonl(path)

    assert len(rows) == 40

    mapping = {}

    for row in rows:
        sid = str(row["scenario_id"])

        assert sid in scenario_ids
        assert sid not in mapping
        assert row["model"] == model
        assert row["benchmark_sha256"] == EXPECTED_SHA
        assert row["inference_version"] == "full_v2"

        mapping[sid] = row

    assert set(mapping) == scenario_ids
    predictions[model] = mapping

print("Frozen benchmark: VERIFIED")
print("All 120 predictions: VERIFIED")

# ============================================================
# 2. ORIGINAL V1 MATCHING RULES
# ============================================================

def normalize_text(value):
    text = unicodedata.normalize(
        "NFKC", str(value)
    ).casefold()

    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

NUM_PATTERN = re.compile(
    r"(?<![\w])"
    r"(?:\d{1,3}(?:,\d{2})+,\d{3}"
    r"|\d{1,3}(?:,\d{3})+"
    r"|\d+)"
    r"(?![\w])"
)

def numbers_in(text):
    found = set()

    for match in NUM_PATTERN.finditer(str(text)):
        found.add(int(match.group(0).replace(",", "")))

    return found

ALIASES = {
    "on-site": ["on-site", "onsite", "on site"],
    "remote": ["remote", "work from home", "wfh"],
    "hybrid": ["hybrid"],

    "work-related creations": [
        "work-related creations",
        "work-related intellectual property",
        "work-related inventions",
    ],
    "work-related IP": [
        "work-related ip",
        "work-related intellectual property",
    ],

    "public information": [
        "public information",
        "publicly available information",
        "public domain",
    ],
    "lawful disclosure": [
        "lawful disclosure",
        "disclosure required by law",
        "legally required disclosure",
    ],
    "reporting channel": [
        "reporting channel",
        "reporting mechanism",
        "reporting procedure",
        "reporting process",
    ],
    "non-retaliation": [
        "non-retaliation",
        "non retaliation",
        "protection against retaliation",
        "no retaliation",
    ],
    "pipelines": ["pipelines", "pipeline"],
    "monitoring": ["monitoring", "monitor"],
    "documentation": ["documentation", "documenting"],
    "employer": ["employer", "company"],
    "employee": ["employee"],
}

def contains_phrase(text, phrase):
    pattern = (
        r"(?<!\w)"
        + re.escape(normalize_text(phrase))
        + r"(?!\w)"
    )
    return bool(re.search(pattern, text))

def v1_check(response, key, value):
    if isinstance(value, bool):
        return False, "unsupported_boolean"

    if isinstance(value, int):
        return value in numbers_in(response), "numeric_literal"

    if isinstance(value, str):
        aliases = ALIASES.get(value, [value])
        matched = any(
            contains_phrase(normalize_text(response), alias)
            for alias in aliases
        )
        return matched, "literal_or_alias"

    return False, "unsupported_type"

# ============================================================
# 3. V2 IMPROVEMENT — SPELLED-OUT NUMBERS
# ============================================================

ONES = {
    0: "zero", 1: "one", 2: "two",
    3: "three", 4: "four", 5: "five",
    6: "six", 7: "seven", 8: "eight",
    9: "nine", 10: "ten", 11: "eleven",
    12: "twelve", 13: "thirteen",
    14: "fourteen", 15: "fifteen",
    16: "sixteen", 17: "seventeen",
    18: "eighteen", 19: "nineteen",
}

TENS = {
    20: "twenty", 30: "thirty",
    40: "forty", 50: "fifty",
    60: "sixty", 70: "seventy",
    80: "eighty", 90: "ninety",
}

UNIT_PATTERNS = {
    "probation_months": r"months?",
    "initial_months": r"months?",
    "extension_months": r"months?",
    "notice_days": r"days?",
    "leave_days": r"days?",
    "contractual_leave_days": r"days?",
    "office_days": r"days?",
    "remote_days": r"days?",
    "hours_per_week": r"hours?",
    "weekly_hours": r"hours?",
}

def number_words(number):
    if number in ONES:
        return ONES[number]

    if number in TENS:
        return TENS[number]

    if 20 < number < 100:
        tens = (number // 10) * 10
        ones = number % 10
        return TENS[tens] + "-" + ONES[ones]

    return None

def spelled_match(response, key, value):
    if key not in UNIT_PATTERNS:
        return False

    if not isinstance(value, int) or isinstance(value, bool):
        return False

    words = number_words(value)

    if words is None:
        return False

    normalized = normalize_text(response)

    number_pattern = re.escape(words).replace(
        r"\-", r"[-\s]+"
    )

    unit = UNIT_PATTERNS[key]

    # Allows e.g.:
    # "three months"
    # "forty-five days"
    # "forty five working days"
    # "three calendar months"
    pattern = (
        r"(?<!\w)"
        + number_pattern
        + r"(?!\w)"
        + r"(?:[\s,()\-]+\w+){0,3}"
        + r"[\s,()\-]+"
        + unit
        + r"\b"
    )

    return bool(re.search(pattern, normalized))

def corrected_check(response, key, value):
    original_match, original_method = v1_check(
        response, key, value
    )

    if original_match:
        return True, original_method

    if spelled_match(response, key, value):
        return True, "spelled_number_with_unit"

    return False, original_method

def evaluate_response(response, expected_facts, version="v2"):
    checks = []

    for key, value in expected_facts.items():
        values = value if isinstance(value, list) else [value]

        for item in values:
            if version == "v1":
                matched, method = v1_check(
                    response, key, item
                )
            else:
                matched, method = corrected_check(
                    response, key, item
                )

            checks.append({
                "field": key,
                "expected": item,
                "matched": bool(matched),
                "method": method,
            })

    total = len(checks)
    assert total > 0

    matched = sum(c["matched"] for c in checks)

    return {
        "coverage": matched / total,
        "matched": matched,
        "total": total,
        "checks": checks,
    }

# ============================================================
# 4. REGRESSION TESTS
# ============================================================

assert corrected_check(
    "Probation is three months from joining.",
    "probation_months", 3
)[0]

assert corrected_check(
    "Notice is forty-five days.",
    "notice_days", 45
)[0]

assert corrected_check(
    "Notice is forty five days.",
    "notice_days", 45
)[0]

assert corrected_check(
    "The scheduled week is forty hours.",
    "hours_per_week", 40
)[0]

assert not corrected_check(
    "Notice is [Number of Days] days.",
    "notice_days", 45
)[0]

print("Scorer regression tests: PASSED")

# ============================================================
# 5. AUDITKIT EVALUATION
# ============================================================

@ak.scorer(
    direction=ak.Direction.MAXIMIZE,
    name="corrected_expected_fact_coverage"
)
def corrected_fact_scorer(sample, output):
    return evaluate_response(
        str(output),
        sample.metadata["expected_facts"],
        version="v2"
    )["coverage"]

score_rows = []
fact_rows = []
audit_results = {}

OUT.mkdir(parents=True, exist_ok=True)

for model in MODELS:
    samples = []

    for scenario in scenarios:
        sid = str(scenario["scenario_id"])
        prediction = predictions[model][sid]
        response = prediction["response"]

        v1 = evaluate_response(
            response,
            scenario["expected_facts"],
            version="v1"
        )

        v2 = evaluate_response(
            response,
            scenario["expected_facts"],
            version="v2"
        )

        # Added matching capabilities must not lower
        # any scenario's fact-presence score.
        assert v2["coverage"] >= v1["coverage"] - 1e-12

        samples.append(ak.Sample(
            id=sid,
            input=prediction["prompt"],
            task=scenario["task_type"],
            actual_output=response,
            metadata={
                "expected_facts": scenario["expected_facts"]
            }
        ))

        score_rows.append({
            "model": model,
            "scenario_id": sid,
            "task_type": scenario["task_type"],
            "v1_coverage": v1["coverage"],
            "v2_coverage": v2["coverage"],
            "delta": v2["coverage"] - v1["coverage"],
            "matched": v2["matched"],
            "total": v2["total"],
            "token_limit_hit": bool(
                prediction["hit_token_limit"]
            ),
        })

        for check in v2["checks"]:
            fact_rows.append({
                "model": model,
                "scenario_id": sid,
                "task_type": scenario["task_type"],
                **check,
            })

    result = ak.evaluate(
        samples,
        model="precomputed",
        scorers=[corrected_fact_scorer]
    )

    if getattr(result, "errors", None):
        raise RuntimeError(
            f"AuditKIT errors for {model}: {result.errors}"
        )

    assert len(result.predictions) == 40

    result.save(
        str(OUT / f"{model}_auditkit.json"),
        fmt="json"
    )

    audit_results[model] = result

    print(f"\n{model.upper()} — AUDITKIT V2")
    print(result.summary())

# ============================================================
# 6. PAIRED COMPARISON
# ============================================================

by_model = {
    model: {
        row["scenario_id"]: row
        for row in score_rows
        if row["model"] == model
    }
    for model in MODELS
}

ids = [str(s["scenario_id"]) for s in scenarios]

def mean_coverage(model, task=None, version="v2"):
    selected = [
        s["scenario_id"]
        for s in scenarios
        if task is None or s["task_type"] == task
    ]

    key = f"{version}_coverage"

    return statistics.mean(
        by_model[model][sid][key]
        for sid in selected
    )

differences = [
    by_model["curated"][sid]["v2_coverage"]
    - by_model["raw"][sid]["v2_coverage"]
    for sid in ids
]

wins = sum(x > 1e-12 for x in differences)
ties = sum(abs(x) <= 1e-12 for x in differences)
losses = sum(x < -1e-12 for x in differences)

rng = random.Random(42)

bootstrap = sorted(
    statistics.mean(
        differences[rng.randrange(len(differences))]
        for _ in differences
    )
    for _ in range(10000)
)

ci = [bootstrap[249], bootstrap[9749]]

task_types = sorted({
    s["task_type"] for s in scenarios
})

# ============================================================
# 7. SAVE RESULTS
# ============================================================

def save_csv(path, records):
    with path.open(
        "w", newline="", encoding="utf-8"
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=list(records[0].keys())
        )
        writer.writeheader()
        writer.writerows(records)

save_csv(
    OUT / "scenario_scores.csv",
    score_rows
)

fact_csv = []

for row in fact_rows:
    converted = dict(row)
    converted["expected"] = json.dumps(
        converted["expected"],
        ensure_ascii=False
    )
    fact_csv.append(converted)

save_csv(
    OUT / "fact_level_checks.csv",
    fact_csv
)

summary = {
    "version": "expected_facts_v2",
    "benchmark_sha256": EXPECTED_SHA,
    "scenario_count": 40,
    "prediction_count": 120,
    "metric": "heuristic_expected_fact_presence",
    "scores": {
        model: {
            "v1_recomputed": mean_coverage(
                model, version="v1"
            ),
            "v2": mean_coverage(
                model, version="v2"
            ),
        }
        for model in MODELS
    },
    "task_scores_v2": {
        task: {
            model: mean_coverage(
                model, task=task
            )
            for model in MODELS
        }
        for task in task_types
    },
    "curated_minus_raw": {
        "mean": statistics.mean(differences),
        "wins": wins,
        "ties": ties,
        "losses": losses,
        "bootstrap_95pct": ci,
    },
    "limitations": [
        "Exploratory post-hoc scorer revision.",
        "Fact presence does not establish contextual correctness.",
        "Numeric literal matching is not field-context-aware.",
        "No independent legally verified gold answers.",
        "No legal compliance or contradiction score.",
    ],
}

(OUT / "summary.json").write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

# ============================================================
# 8. PRINT RESULTS
# ============================================================

print("\n" + "=" * 68)
print("M5.10 — FINAL CORRECTED FACT-COVERAGE SUMMARY")
print("=" * 68)

print(
    f"{'Model':10s} {'V1':>10s} "
    f"{'V2':>10s} {'Change':>10s}"
)

for model in MODELS:
    before = mean_coverage(model, version="v1")
    after = mean_coverage(model, version="v2")

    print(
        f"{model.upper():10s} "
        f"{before*100:9.2f}% "
        f"{after*100:9.2f}% "
        f"{(after-before)*100:+9.2f}pp"
    )

print("\nBY TASK — V2")

print(
    f"{'Task':28s} "
    f"{'Base':>8s} {'Raw':>8s} {'Curated':>8s}"
)

for task in task_types:
    print(
        f"{task:28s} "
        f"{mean_coverage('base',task)*100:8.2f} "
        f"{mean_coverage('raw',task)*100:8.2f} "
        f"{mean_coverage('curated',task)*100:8.2f}"
    )

print("\nRAW VS CURATED — PAIRED V2")

print("Curated wins:", wins)
print("Ties:", ties)
print("Raw wins:", losses)

print(
    "Mean difference (percentage points):",
    round(statistics.mean(differences)*100, 2)
)

print(
    "Exploratory paired 95% bootstrap interval:",
    [round(x*100, 2) for x in ci]
)

print("\nSCENARIOS CHANGED FROM V1")

for row in score_rows:
    if row["delta"] > 1e-12:
        print(
            row["model"],
            row["scenario_id"],
            f"{row['v1_coverage']:.3f}",
            "->",
            f"{row['v2_coverage']:.3f}"
        )

print("\nEVAL013 AND EVAL026")

for sid in ["EVAL013", "EVAL026"]:
    for model in ["raw", "curated"]:
        row = by_model[model][sid]

        print(
            sid,
            model,
            f"V1={row['v1_coverage']:.3f}",
            f"V2={row['v2_coverage']:.3f}"
        )

print("\n" + "=" * 68)
print("M5.10 COMPLETE")
print("=" * 68)
print("Saved:", OUT)
print("M5.8 v1 results: PRESERVED")
print("Training and inference: NOT REPEATED")
print("Legal correctness: NOT ESTABLISHED")


CURATELAW M5.10 — CORRECTED FACT COVERAGE
Frozen benchmark: VERIFIED
All 120 predictions: VERIFIED
Scorer regression tests: PASSED

BASE — AUDITKIT V2
run e5d2b8c305aa4677 (fingerprint e5d2b8c305aa4677)
  corrected_expected_fact_coverage: 0.8399 ± 0.0488  (n=40)
  model: precomputed (not a local model -- no measurable size)

RAW — AUDITKIT V2
run c2cc7c5e9ea0a482 (fingerprint c2cc7c5e9ea0a482)
  corrected_expected_fact_coverage: 0.8054 ± 0.0529  (n=40)
  model: precomputed (not a local model -- no measurable size)

CURATED — AUDITKIT V2
run 4551f94c967c5e69 (fingerprint 4551f94c967c5e69)
  corrected_expected_fact_coverage: 0.8018 ± 0.0559  (n=40)
  model: precomputed (not a local model -- no measurable size)

M5.10 — FINAL CORRECTED FACT-COVERAGE SUMMARY
Model              V1         V2     Change
BASE           73.99%     83.99%    +10.00pp
RAW            71.79%     80.54%     +8.75pp
CURATED        68.93%     80.18%    +11.25pp

BY TASK — V2
Task                             Base     

Contract consistency and instruction compliance

In [8]:

import json
import csv
import re
import hashlib
from pathlib import Path
from collections import Counter

print("=" * 68)
print("CURATELAW M5.11 — CONTRACT QUALITY RISK SCREEN")
print("=" * 68)

ROOT = Path("/content/drive/MyDrive/LexiLabs")
EVAL = ROOT / "evaluation_artifacts"
PRED_DIR = EVAL / "full_inference_v2"
OUT = EVAL / "auditkit_results" / "contract_quality_screen_v1"

EXPECTED_SHA = (
    "944ba82ed4937a41f32c064aaaa89f1b34"
    "aeb2d44a89e0860aac51aa3d46bff9"
)

def read_jsonl(path):
    assert path.is_file(), f"Missing: {path}"
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

benchmark_path = EVAL / "heldout_eval_040.jsonl"

assert hashlib.sha256(
    benchmark_path.read_bytes()
).hexdigest() == EXPECTED_SHA

benchmark = {
    r["scenario_id"]: r
    for r in read_jsonl(benchmark_path)
}

assert len(benchmark) == 40

predictions = {}

for model in ["base", "raw", "curated"]:
    rows = read_jsonl(
        PRED_DIR / f"{model}_predictions.jsonl"
    )

    assert len(rows) == 40

    predictions[model] = {
        r["scenario_id"]: r
        for r in rows
    }

    assert set(predictions[model]) == set(benchmark)

    for pred in rows:
        assert pred["model"] == model
        assert pred["benchmark_sha256"] == EXPECTED_SHA
        assert pred["inference_version"] == "full_v2"

print("Benchmark and predictions: VERIFIED")

# -----------------------------------------------------------
# Diagnostic patterns: flags, not automatic correctness scores
# -----------------------------------------------------------

NUMERIC_PLACEHOLDER = re.compile(
    r"\[(?:number(?: of (?:days|months|hours))?"
    r"|amount|salary|notice period|duration)\]",
    flags=re.IGNORECASE,
)

SALARY_WORDS = re.compile(
    r"\b(?:crores?|lakhs?|millions?)\b",
    flags=re.IGNORECASE,
)

LEGAL_REFERENCE = re.compile(
    r"\b(?:[A-Z][A-Za-z&,\s-]{2,85})"
    r"\s(?:Act|Code),?\s*(?:19|20)\d{2}\b"
)

def repeated_paragraphs(text):
    paragraphs = [
        re.sub(r"\s+", " ", p).strip().casefold()
        for p in re.split(r"\n\s*\n", text)
    ]

    paragraphs = [
        p for p in paragraphs
        if len(p) >= 80
    ]

    counts = Counter(paragraphs)

    return sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

def inspect_output(scenario, prediction):
    response = prediction["response"]
    instruction = scenario["instruction"]

    placeholder_matches = sorted(set(
        m.group(0)
        for m in NUMERIC_PLACEHOLDER.finditer(response)
    ))

    salary_word_matches = sorted(set(
        m.group(0).lower()
        for m in SALARY_WORDS.finditer(response)
    ))

    legal_refs = sorted(set(
        m.group(0).strip()
        for m in LEGAL_REFERENCE.finditer(response)
    ))

    explanation_present = bool(re.search(
        r"(?im)^\s*(?:\*\*)?explanation"
        r"(?:\*\*)?\s*:",
        response
    ))

    corrected_wording_only_requested = (
        scenario["task_type"] == "contradiction_correction"
        and bool(re.search(
            r"\b(?:return|provide)\b.{0,60}"
            r"\b(?:corrected|revised)\b.{0,35}"
            r"\bwording\b",
            instruction,
            flags=re.IGNORECASE | re.DOTALL
        ))
    )

    redundant_explanation_flag = (
        corrected_wording_only_requested
        and explanation_present
    )

    repeat_count = repeated_paragraphs(response)

    flags = []

    if placeholder_matches:
        flags.append("unresolved_numeric_placeholder")

    if salary_word_matches and (
        "salary_inr" in scenario["expected_facts"]
        or "new_salary_inr" in scenario["expected_facts"]
    ):
        flags.append("salary_words_need_consistency_review")

    if redundant_explanation_flag:
        flags.append("extra_explanation_on_wording_only_task")

    if repeat_count:
        flags.append("repeated_paragraph")

    if prediction["hit_token_limit"]:
        flags.append("token_limit_hit")

    return {
        "flags": flags,
        "placeholders": placeholder_matches,
        "salary_words": salary_word_matches,
        "legal_references_for_review": legal_refs,
        "repeated_paragraph_count": repeat_count,
        "extra_explanation_flag": redundant_explanation_flag,
    }

# -----------------------------------------------------------
# Evaluate all 120 stored responses
# -----------------------------------------------------------

results = []

for sid in sorted(benchmark):
    scenario = benchmark[sid]

    for model in ["base", "raw", "curated"]:
        pred = predictions[model][sid]
        diagnostics = inspect_output(scenario, pred)

        results.append({
            "scenario_id": sid,
            "task_type": scenario["task_type"],
            "model": model,
            "flags": diagnostics["flags"],
            "flag_count": len(diagnostics["flags"]),
            "placeholders": diagnostics["placeholders"],
            "salary_words": diagnostics["salary_words"],
            "legal_references_for_review":
                diagnostics["legal_references_for_review"],
            "repeated_paragraph_count":
                diagnostics["repeated_paragraph_count"],
            "extra_explanation_flag":
                diagnostics["extra_explanation_flag"],
            "token_limit_hit": bool(pred["hit_token_limit"]),
        })

assert len(results) == 120

# -----------------------------------------------------------
# Save non-destructively
# -----------------------------------------------------------

OUT.mkdir(parents=True, exist_ok=True)

csv_path = OUT / "quality_risk_flags.csv"
json_path = OUT / "quality_risk_summary.json"

if csv_path.exists() or json_path.exists():
    raise FileExistsError(
        "M5.11 results already exist. "
        "Preserving previous output; inspect before rerunning."
    )

csv_rows = []

for row in results:
    converted = dict(row)

    for field in [
        "flags",
        "placeholders",
        "salary_words",
        "legal_references_for_review",
    ]:
        converted[field] = json.dumps(
            converted[field],
            ensure_ascii=False
        )

    csv_rows.append(converted)

with csv_path.open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.DictWriter(
        f, fieldnames=list(csv_rows[0])
    )
    writer.writeheader()
    writer.writerows(csv_rows)

summary = {}

for model in ["base", "raw", "curated"]:
    subset = [
        r for r in results
        if r["model"] == model
    ]

    counts = Counter(
        flag
        for r in subset
        for flag in r["flags"]
    )

    summary[model] = {
        "responses": len(subset),
        "responses_with_any_flag": sum(
            bool(r["flags"]) for r in subset
        ),
        "flag_counts": dict(counts),
        "legal_reference_mentions_for_review": sum(
            len(r["legal_references_for_review"])
            for r in subset
        ),
    }

json_path.write_text(
    json.dumps({
        "version": "contract_quality_screen_v1",
        "benchmark_sha256": EXPECTED_SHA,
        "disclaimer": (
            "Risk flags only; not verified factual "
            "errors, legal judgments, or quality scores."
        ),
        "models": summary
    }, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

# -----------------------------------------------------------
# Display diagnostic counts and key examples
# -----------------------------------------------------------

print("\n" + "=" * 68)
print("M5.11 — RISK FLAG SUMMARY")
print("=" * 68)

for model in ["base", "raw", "curated"]:
    data = summary[model]

    print(f"\n{model.upper()}")
    print(
        "Responses with at least one flag:",
        data["responses_with_any_flag"], "/ 40"
    )

    for flag, count in sorted(
        data["flag_counts"].items()
    ):
        print(f"  {flag}: {count}")

print("\nSELECTED SCENARIO FLAGS")

for sid in [
    "EVAL004",
    "EVAL006",
    "EVAL013",
    "EVAL026",
]:
    print(f"\n{sid}")

    for model in ["raw", "curated"]:
        row = next(
            r for r in results
            if r["scenario_id"] == sid
            and r["model"] == model
        )

        print(
            f"  {model.upper():8s}",
            row["flags"] or ["none"]
        )

print("\n" + "=" * 68)
print("M5.11A SCREENING COMPLETE")
print("=" * 68)

print("CSV:", csv_path)
print("Summary:", json_path)
print("All original predictions: UNCHANGED")
print("All previous AuditKIT scores: UNCHANGED")
print("Legal verification: NOT PERFORMED")


CURATELAW M5.11 — CONTRACT QUALITY RISK SCREEN
Benchmark and predictions: VERIFIED

M5.11 — RISK FLAG SUMMARY

BASE
Responses with at least one flag: 39 / 40
  extra_explanation_on_wording_only_task: 1
  repeated_paragraph: 6
  salary_words_need_consistency_review: 6
  token_limit_hit: 39
  unresolved_numeric_placeholder: 4

RAW
Responses with at least one flag: 15 / 40
  extra_explanation_on_wording_only_task: 3
  salary_words_need_consistency_review: 4
  token_limit_hit: 6
  unresolved_numeric_placeholder: 4

CURATED
Responses with at least one flag: 16 / 40
  extra_explanation_on_wording_only_task: 3
  repeated_paragraph: 2
  salary_words_need_consistency_review: 5
  token_limit_hit: 5
  unresolved_numeric_placeholder: 3

SELECTED SCENARIO FLAGS

EVAL004
  RAW      ['unresolved_numeric_placeholder', 'salary_words_need_consistency_review']
  CURATED  ['salary_words_need_consistency_review']

EVAL006
  RAW      ['salary_words_need_consistency_review']
  CURATED  ['salary_words_need_co

Blinded quality review

In [9]:

import csv
import json
import random
import hashlib
from pathlib import Path
from collections import Counter

print("=" * 68)
print("CURATELAW M5.11B — BLINDED QUALITY REVIEW")
print("=" * 68)

ROOT = Path("/content/drive/MyDrive/LexiLabs")
EVAL = ROOT / "evaluation_artifacts"
PRED_DIR = EVAL / "full_inference_v2"
RISK_DIR = EVAL / "auditkit_results/contract_quality_screen_v1"
OUT = EVAL / "blinded_quality_review_v1"

EXPECTED_SHA = (
    "944ba82ed4937a41f32c064aaaa89f1b34"
    "aeb2d44a89e0860aac51aa3d46bff9"
)

def load_jsonl(path):
    assert path.is_file(), f"Missing file: {path}"
    with path.open("r", encoding="utf-8") as f:
        return [
            json.loads(line)
            for line in f
            if line.strip()
        ]

benchmark_path = EVAL / "heldout_eval_040.jsonl"

assert hashlib.sha256(
    benchmark_path.read_bytes()
).hexdigest() == EXPECTED_SHA

benchmark = {
    r["scenario_id"]: r
    for r in load_jsonl(benchmark_path)
}

assert len(benchmark) == 40

predictions = {}

for model in ["base", "raw", "curated"]:
    rows = load_jsonl(
        PRED_DIR / f"{model}_predictions.jsonl"
    )

    assert len(rows) == 40

    predictions[model] = {
        r["scenario_id"]: r
        for r in rows
    }

    assert set(predictions[model]) == set(benchmark)
    assert all(
        r["model"] == model
        and r["benchmark_sha256"] == EXPECTED_SHA
        and r["inference_version"] == "full_v2"
        for r in rows
    )

# Priority is for review order, NOT for scoring weights.
decisive_cases = {
    "EVAL001", "EVAL004", "EVAL006",
    "EVAL008", "EVAL013", "EVAL026"
}

flagged_scenarios = set()

with (
    RISK_DIR / "quality_risk_flags.csv"
).open("r", encoding="utf-8", newline="") as f:
    risk_rows = list(csv.DictReader(f))

assert len(risk_rows) == 120

for row in risk_rows:
    if int(row["flag_count"]) > 0:
        flagged_scenarios.add(row["scenario_id"])

def priority(sid):
    if sid in decisive_cases:
        return 1
    if sid in flagged_scenarios:
        return 2
    return 3

ordered_ids = sorted(
    benchmark,
    key=lambda sid: (priority(sid), sid)
)

rng = random.Random(20261114)

review_rows = []
answer_key = []

for sid in ordered_ids:
    scenario = benchmark[sid]

    # Shuffle model identity separately for every scenario.
    shuffled_models = ["base", "raw", "curated"]
    rng.shuffle(shuffled_models)

    for label, model in zip(
        ["A", "B", "C"], shuffled_models
    ):
        pred = predictions[model][sid]

        review_id = f"{sid}_{label}"

        review_rows.append({
            "review_id": review_id,
            "scenario_id": sid,
            "review_priority": priority(sid),
            "candidate": label,
            "task_type": scenario["task_type"],
            "instruction": scenario["instruction"],
            "original_input": scenario.get("input", ""),
            "expected_facts": json.dumps(
                scenario["expected_facts"],
                ensure_ascii=False
            ),
            "response": pred["response"],

            # Fill these manually using 0, 1, or 2.
            "fact_correctness_0_2": "",
            "internal_consistency_0_2": "",
            "instruction_compliance_0_2": "",
            "no_unsupported_additions_0_2": "",

            "material_errors": "",
            "reviewer_notes": "",
            "reviewer_id": "",
        })

        answer_key.append({
            "review_id": review_id,
            "scenario_id": sid,
            "candidate": label,
            "model": model,
        })

assert len(review_rows) == 120
assert len(answer_key) == 120

OUT.mkdir(parents=True, exist_ok=True)

review_file = OUT / "blind_review_form.csv"
key_file = OUT / "MODEL_IDENTITY_KEY.json"
rubric_file = OUT / "REVIEW_RUBRIC.txt"

# Prevent accidental loss of previously entered reviews.
for path in [review_file, key_file, rubric_file]:
    if path.exists():
        raise FileExistsError(
            f"Review artifact already exists: {path}. "
            "Existing review work was preserved."
        )

with review_file.open(
    "w", encoding="utf-8", newline=""
) as f:
    writer = csv.DictWriter(
        f, fieldnames=list(review_rows[0].keys())
    )
    writer.writeheader()
    writer.writerows(review_rows)

key_file.write_text(
    json.dumps({
        "benchmark_sha256": EXPECTED_SHA,
        "random_seed": 20261114,
        "mapping": answer_key,
    }, indent=2),
    encoding="utf-8"
)

rubric = """CURATELAW BLINDED DRAFT QUALITY RUBRIC

REVIEW PROCESS
- Review all candidates without opening MODEL_IDENTITY_KEY.json.
- Use the same criteria for every candidate.
- Assess the requested contractual facts and drafting behavior.
- Record specific supporting text in reviewer_notes.
- Do not infer model quality from response length alone.
- Record uncertain or unverifiable claims in material_errors.
- Do not treat this review as completed legal verification.

SCORES (each criterion: 0, 1, or 2)

1. fact_correctness_0_2
0 = Material requested facts incorrect or absent
1 = Some requested facts correct, some incomplete/ambiguous
2 = All requested facts correctly represented

2. internal_consistency_0_2
0 = Material internal contradiction
1 = Minor or ambiguous inconsistency
2 = No detected internal contradiction

3. instruction_compliance_0_2
0 = Major failure to follow requested task/format
1 = Partially follows instructions
2 = Fully follows the requested task and format

4. no_unsupported_additions_0_2
0 = Material unsupported additions or claims
1 = Minor unnecessary or potentially unsupported content
2 = No material unsupported additions detected

NOTES
- Keep all scores blank until reviewed.
- A factual inconsistency and an unsupported claim may be
  independently recorded if both criteria apply.
- Legal validity must not be assumed from this rubric.
- Do not open the identity key before ratings are finalized.
- The same 40 scenarios must be used for every model.
"""

rubric_file.write_text(
    rubric, encoding="utf-8"
)

print("Benchmark integrity: VERIFIED")
print("Scenarios:", len(benchmark))
print("Blinded candidate responses:", len(review_rows))

print("\nReview priority distribution:")
for p, count in sorted(Counter(
    priority(sid) for sid in ordered_ids
).items()):
    print(f" Priority {p}: {count} scenarios")

print("\nFiles saved:")
print("Review form:", review_file)
print("Rubric:", rubric_file)
print("Identity key:", key_file)

print("\nIMPORTANT:")
print("Do not open the model identity key while scoring.")
print("Review candidates A, B, and C independently.")
print("No existing training or evaluation files changed.")

print("\nM5.11B — REVIEW PACKET READY")


CURATELAW M5.11B — BLINDED QUALITY REVIEW
Benchmark integrity: VERIFIED
Scenarios: 40
Blinded candidate responses: 120

Review priority distribution:
 Priority 1: 6 scenarios
 Priority 2: 33 scenarios
 Priority 3: 1 scenarios

Files saved:
Review form: /content/drive/MyDrive/LexiLabs/evaluation_artifacts/blinded_quality_review_v1/blind_review_form.csv
Rubric: /content/drive/MyDrive/LexiLabs/evaluation_artifacts/blinded_quality_review_v1/REVIEW_RUBRIC.txt
Identity key: /content/drive/MyDrive/LexiLabs/evaluation_artifacts/blinded_quality_review_v1/MODEL_IDENTITY_KEY.json

IMPORTANT:
Do not open the model identity key while scoring.
Review candidates A, B, and C independently.
No existing training or evaluation files changed.

M5.11B — REVIEW PACKET READY
